## Firstly be sure that the combined df is in global

In [1]:
# ============================================================================
# CELL 2: LOAD VARIABLES (ضعها في بداية Notebook الجديد)
# ============================================================================

import pickle
from pathlib import Path
from datetime import datetime

print("=" * 80)
print("📥 LOADING SAVED VARIABLES")
print("=" * 80)

# ═══════════════════════════════════════════════════════════════════════════
# 📂 STEP 1: Find save folder
# ═══════════════════════════════════════════════════════════════════════════

save_folder = Path.home() / "Desktop" / "notebook_variables"

if not save_folder.exists():
    print()
    print("❌ No saved sessions found!")
    print(f"   Expected location: {save_folder}")
    print()
    print("💡 First run CELL 1 in your source notebook to save variables")
    print("=" * 80)
else:
    # ═══════════════════════════════════════════════════════════════════════════
    # 📋 STEP 2: List available sessions
    # ═══════════════════════════════════════════════════════════════════════════

    sessions = sorted([d for d in save_folder.iterdir() if d.is_dir()], reverse=True)

    if len(sessions) == 0:
        print()
        print("❌ No sessions found in folder!")
        print(f"   Folder exists but is empty: {save_folder}")
        print()
        print("💡 Run CELL 1 in your source notebook to create a session")
        print("=" * 80)
    else:
        print()
        print(f"📂 Found {len(sessions)} saved session(s):")
        print()
        print("-" * 80)

        # Display available sessions
        for idx, session in enumerate(sessions, 1):
            metadata_path = session / "metadata.pkl"

            if metadata_path.exists():
                try:
                    with open(metadata_path, 'rb') as f:
                        metadata = pickle.load(f)

                    dt = metadata.get('datetime', metadata.get('timestamp', 'Unknown'))
                    var_count = metadata.get('success_count',
                               len([v for v in metadata.get('saved_variables', {}).values()
                                   if v.get('saved', False)]))

                    print(f"  {idx}. {session.name}")
                    print(f"     Date: {dt}")
                    print(f"     Variables: {var_count}")

                    # Show variable names
                    vars_list = [k for k, v in metadata.get('saved_variables', {}).items()
                                if v.get('saved', False)]
                    if len(vars_list) > 0:
                        print(f"     Contains: {', '.join(vars_list[:5])}")
                        if len(vars_list) > 5:
                            print(f"               ... and {len(vars_list)-5} more")
                    print()
                except:
                    print(f"  {idx}. {session.name} (metadata error)")
                    print()
            else:
                print(f"  {idx}. {session.name} (no metadata)")
                print()

        print("-" * 80)

        # ═══════════════════════════════════════════════════════════════════════════
        # 🎯 STEP 3: Choose session to load
        # ═══════════════════════════════════════════════════════════════════════════

        choice = input("\nEnter session number to load (press Enter for latest): ").strip()

        if choice == '':
            choice = '1'

        try:
            session_idx = int(choice) - 1

            if session_idx < 0 or session_idx >= len(sessions):
                print(f"\n❌ Invalid choice! Must be 1-{len(sessions)}")
                print("=" * 80)
            else:
                selected_session = sessions[session_idx]

                print()
                print("=" * 80)
                print(f"📥 Loading session: {selected_session.name}")
                print("=" * 80)
                print()

                # ═══════════════════════════════════════════════════════════════════════════
                # 📦 STEP 4: Load metadata
                # ═══════════════════════════════════════════════════════════════════════════

                metadata_path = selected_session / "metadata.pkl"

                if metadata_path.exists():
                    with open(metadata_path, 'rb') as f:
                        metadata = pickle.load(f)
                else:
                    print("⚠️  No metadata found - will try to load all .pkl files")
                    metadata = {'saved_variables': {}}

                # ═══════════════════════════════════════════════════════════════════════════
                # 💾 STEP 5: Load variables
                # ═══════════════════════════════════════════════════════════════════════════

                loaded_count = 0
                failed_count = 0

                saved_vars = metadata.get('saved_variables', {})

                if len(saved_vars) == 0:
                    # No metadata, try all .pkl files
                    pkl_files = list(selected_session.glob("*.pkl"))
                    print(f"Found {len(pkl_files)} .pkl files (excluding metadata)")
                    print()

                    for pkl_file in pkl_files:
                        if pkl_file.name != "metadata.pkl":
                            var_name = pkl_file.stem  # filename without .pkl

                            try:
                                with open(pkl_file, 'rb') as f:
                                    var_value = pickle.load(f)

                                globals()[var_name] = var_value

                                size_kb = pkl_file.stat().st_size / 1024
                                size_str = f"{size_kb:.1f} KB" if size_kb < 1024 else f"{size_kb/1024:.1f} MB"
                                var_type = type(var_value).__name__

                                print(f"✅ {var_name:<25} | {var_type:<15} | {size_str}")
                                loaded_count += 1

                            except Exception as e:
                                print(f"❌ {var_name:<25} | FAILED: {str(e)[:40]}")
                                failed_count += 1
                else:
                    # Use metadata
                    for var_name, var_info in saved_vars.items():
                        if var_info.get('saved', False):
                            try:
                                file_path = selected_session / f"{var_name}.pkl"

                                with open(file_path, 'rb') as f:
                                    var_value = pickle.load(f)

                                # Load into global scope
                                globals()[var_name] = var_value

                                print(f"✅ {var_name:<25} | {var_info.get('type', 'Unknown'):<15} | {var_info.get('size', 'Unknown')}")
                                loaded_count += 1

                            except FileNotFoundError:
                                print(f"❌ {var_name:<25} | FILE NOT FOUND")
                                failed_count += 1
                            except Exception as e:
                                print(f"❌ {var_name:<25} | FAILED: {str(e)[:40]}")
                                failed_count += 1

                # ═══════════════════════════════════════════════════════════════════════════
                # ✅ STEP 6: Summary
                # ═══════════════════════════════════════════════════════════════════════════

                print()
                print("=" * 80)
                print("✅ LOAD COMPLETE!")
                print("=" * 80)
                print(f"✅ Loaded:  {loaded_count} variables")
                print(f"❌ Failed:  {failed_count} variables")
                print("=" * 80)
                print()
                print("💡 Variables are now available in this notebook!")
                print("   Example: print(combined_df.head())")
                print("=" * 80)

        except ValueError:
            print()
            print("❌ Invalid input! Please enter a number")
            print("=" * 80)

📥 LOADING SAVED VARIABLES

📂 Found 29 saved session(s):

--------------------------------------------------------------------------------
  1. session_20260119_145703
     Date: 2026-01-19 14:57:45
     Variables: 7
     Contains: combined_df, combined_df_updated, df, inventory_df, inventory_df_tagropa
               ... and 2 more

  2. session_20260119_145440
     Date: 2026-01-19 14:55:29
     Variables: 5
     Contains: combined_df, combined_df_updated, df, inventory_df, inventory_df_tagropa

  3. session_20260119_123201
     Date: 2026-01-19 12:32:51
     Variables: 6
     Contains: combined_df, combined_df_updated, df, inventory_df, sku_df
               ... and 1 more

  4. session_20260119_122934
     Date: 2026-01-19 12:30:21
     Variables: 4
     Contains: combined_df, combined_df_updated, df, inventory_df

  5. session_20260119_122517
     Date: 2026-01-19 12:26:21
     Variables: 4
     Contains: combined_df, combined_df_updated, df, inventory_df

  6. session_20260119_121

KeyboardInterrupt: Interrupted by user

## Detecting sales type ( Family , contract )

In [2]:
# import pandas as pd
# import numpy as np
# import os
# from pathlib import Path
# import glob
# from datetime import datetime
# from IPython.display import display, HTML, clear_output
# import ipywidgets as widgets
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots
# import plotly.io as pio
# import colorsys
# from scipy import stats
# from tqdm.notebook import tqdm
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.preprocessing import StandardScaler
# from statsmodels.tsa.holtwinters import ExponentialSmoothing
# import warnings
# warnings.filterwarnings('ignore')
#
# pio.renderers.default = 'browser'
# pd.set_option('display.max_columns', None)
#
# print("✅ Libraries imported!")
# print("=" * 80)

✅ Libraries imported!


In [6]:
# def detect_sales_type(df, contract_threshold=100):
#     """
#     Detect sales type per transaction and ATTACH to combined_df
#     CONTRACT = bal Qty >= threshold
#     FAMILY   = otherwise
#     """
#     print(f"\n🏢 Detecting sales types (threshold: {contract_threshold} meters)...")
#
#     # Safety conversion
#     df['bal Qty'] = pd.to_numeric(df['bal Qty'], errors='coerce')
#
#     df['Sales_Type'] = np.where(
#         df['bal Qty'] >= contract_threshold,
#         'CONTRACT',
#         'FAMILY'
#     )
#
#     contract_count = (df['Sales_Type'] == 'CONTRACT').sum()
#     family_count = (df['Sales_Type'] == 'FAMILY').sum()
#
#     contract_value = df.loc[df['Sales_Type'] == 'CONTRACT', 'bal Value'].sum()
#     family_value = df.loc[df['Sales_Type'] == 'FAMILY', 'bal Value'].sum()
#
#     print(f"✅ CONTRACT: {contract_count:,} records | Value: {contract_value:,.0f}")
#     print(f"✅ FAMILY  : {family_count:,} records | Value: {family_value:,.0f}")
#
#     return df
#
#
# print("✅ Sales type detection ready (integrates into combined_df)!")
# combined_df = detect_sales_type(combined_df, contract_threshold=100)

✅ Sales type detection ready (integrates into combined_df)!

🏢 Detecting sales types (threshold: 100 meters)...
✅ CONTRACT: 38,030 records | Value: 203,334,426
✅ FAMILY  : 4,880,515 records | Value: 1,334,019,877


## 📈 Advanced Trend Detection Framework

This section defines a **multi-method trend detection system** designed for retail and fashion sales analysis.

### 🔍 Why multiple methods?
Sales data is noisy, seasonal, and often affected by:
- Contract spikes
- Stock-outs
- Fashion cycles
- Promotions

Using **one method alone is risky**, so we apply **6 complementary metrics** and use a **voting system**.

---

### 🧠 Methods Used

1. **Exponential Smoothing (Holt’s Trend)**
   - Captures non-linear retail patterns
   - Focuses on recent momentum

2. **Mann-Kendall Trend Test**
   - Non-parametric
   - Robust to outliers and spikes

3. **Period Comparison**
   - Compares early vs recent periods
   - Simple and intuitive

4. **CAGR (Compound Annual Growth Rate)**
   - Long-term growth measurement
   - Only applied when enough history exists

5. **Consistency Score**
   - Measures directional stability month-to-month

6. **Volatility (Coefficient of Variation)**
   - Penalizes unstable / erratic sales

---

### 🗳️ Voting Logic

Each method votes for:
- **Growing**
- **Declining**
- or **Neutral**

Final trend is determined by **majority vote**, then adjusted by:
- Consistency
- Volatility

This produces:
- **Trend Direction**
- **Confidence Level (HIGH / MEDIUM / LOW)**

---

✅ This approach significantly reduces false signals caused by:
- One-time contracts
- Stock issues
- Seasonal noise


In [4]:
# from statsmodels.tsa.holtwinters import ExponentialSmoothing
# from scipy.stats import kendalltau
#
# def calculate_exponential_trend(monthly_values):
#     """
#     Method 1: Exponential Smoothing Trend
#     Better for retail sales (non-linear patterns)
#     """
#     try:
#         if len(monthly_values) < 6:
#             return 0, 0
#
#         # Fit Holt's exponential smoothing (level + trend)
#         model = ExponentialSmoothing(
#             monthly_values,
#             trend='add',
#             seasonal=None
#         )
#
#         fit = model.fit(
#             smoothing_level=0.3,
#             smoothing_trend=0.1,
#             optimized=False
#         )
#
#         # Get trend component
#         trend = fit.trend
#
#         # Recent trend (last 3 months average)
#         recent_trend = trend[-3:].mean() if len(trend) >= 3 else trend[-1]
#
#         # Trend strength (consistency of direction)
#         positive_trends = (trend > 0).sum()
#         strength = abs(positive_trends / len(trend) - 0.5) * 2  # 0-1 scale
#
#         return recent_trend, strength
#
#     except:
#         return 0, 0
#
# def calculate_mann_kendall(monthly_values):
#     """
#     Method 2: Mann-Kendall Trend Test
#     Non-parametric, robust to outliers
#     """
#     try:
#         if len(monthly_values) < 6:
#             return 0, 1
#
#         months = list(range(len(monthly_values)))
#         tau, p_value = kendalltau(months, monthly_values)
#
#         return tau, p_value
#
#     except:
#         return 0, 1
#
# def calculate_period_comparison(monthly_values):
#     """
#     Method 3: Period Comparison
#     Compare first 25% vs last 25%
#     """
#     try:
#         if len(monthly_values) < 6:
#             return 0
#
#         period_size = max(2, len(monthly_values) // 4)
#
#         first_period = monthly_values[:period_size].mean()
#         last_period = monthly_values[-period_size:].mean()
#
#         if first_period == 0:
#             return 0
#
#         change_pct = (last_period - first_period) / first_period * 100
#
#         return change_pct
#
#     except:
#         return 0
#
# def calculate_cagr(monthly_values):
#     """
#     Method 4: CAGR (Compound Annual Growth Rate)
#     Only for 12+ months
#     """
#     try:
#         if len(monthly_values) < 12:
#             return None
#
#         beginning = monthly_values[:3].mean()  # First quarter
#         ending = monthly_values[-3:].mean()    # Last quarter
#
#         if beginning == 0 or ending == 0:
#             return None
#
#         years = len(monthly_values) / 12
#         cagr = (ending / beginning) ** (1 / years) - 1
#
#         return cagr * 100  # As percentage
#
#     except:
#         return None
#
# def calculate_consistency(monthly_values):
#     """
#     Consistency Score: % of months moving in same direction
#     """
#     try:
#         if len(monthly_values) < 3:
#             return 0
#
#         # Month-to-month changes
#         changes = np.diff(monthly_values)
#
#         if len(changes) == 0:
#             return 0
#
#         positive = (changes > 0).sum()
#         negative = (changes < 0).sum()
#
#         # Consistency = max(positive, negative) / total
#         consistency = max(positive, negative) / len(changes) * 100
#
#         return consistency
#
#     except:
#         return 0
#
# def determine_trend_consensus(monthly_values):
#     """
#     VOTING SYSTEM: Combine all 4 methods
#     Returns: (trend_direction, confidence, metrics)
#     """
#     # Calculate all metrics
#     exp_trend, exp_strength = calculate_exponential_trend(monthly_values)
#     mk_tau, mk_p = calculate_mann_kendall(monthly_values)
#     period_change = calculate_period_comparison(monthly_values)
#     cagr = calculate_cagr(monthly_values)
#     consistency = calculate_consistency(monthly_values)
#
#     # Volatility (CV)
#     cv = monthly_values.std() / monthly_values.mean() if monthly_values.mean() > 0 else 0
#
#     # VOTING
#     votes_declining = 0
#     votes_growing = 0
#
#     # Vote 1: Exponential Smoothing
#     if exp_trend < -500:  # Declining more than 500/month
#         votes_declining += 1
#     elif exp_trend > 500:  # Growing more than 500/month
#         votes_growing += 1
#
#     # Vote 2: Mann-Kendall
#     if mk_tau < -0.2 and mk_p < 0.05:  # Significant decline
#         votes_declining += 1
#     elif mk_tau > 0.2 and mk_p < 0.05:  # Significant growth
#         votes_growing += 1
#
#     # Vote 3: Period Comparison
#     if period_change < -15:  # Declined > 15%
#         votes_declining += 1
#     elif period_change > 15:  # Grew > 15%
#         votes_growing += 1
#
#     # Vote 4: CAGR (if available)
#     if cagr is not None:
#         if cagr < -10:  # CAGR < -10%
#             votes_declining += 1
#         elif cagr > 10:  # CAGR > +10%
#             votes_growing += 1
#
#     # Determine trend
#     total_votes = 4 if cagr is not None else 3
#
#     if votes_declining >= 2:  # Majority says declining
#         trend_direction = "declining"
#         confidence_score = votes_declining / total_votes
#     elif votes_growing >= 2:  # Majority says growing
#         trend_direction = "growing"
#         confidence_score = votes_growing / total_votes
#     else:  # No clear trend
#         trend_direction = "stable"
#         confidence_score = 0.5
#
#     # Adjust confidence by consistency and volatility
#     if consistency > 70 and cv < 0.5:
#         confidence_score *= 1.2  # Boost confidence
#     elif consistency < 50 or cv > 0.8:
#         confidence_score *= 0.8  # Reduce confidence
#
#     confidence_score = min(confidence_score, 1.0)
#
#     # Confidence level
#     if confidence_score >= 0.75:
#         confidence = "HIGH"
#     elif confidence_score >= 0.5:
#         confidence = "MEDIUM"
#     else:
#         confidence = "LOW"
#
#     metrics = {
#         'exp_trend': exp_trend,
#         'exp_strength': exp_strength,
#         'mk_tau': mk_tau,
#         'mk_p': mk_p,
#         'period_change': period_change,
#         'cagr': cagr,
#         'consistency': consistency,
#         'volatility': cv,
#         'votes_declining': votes_declining,
#         'votes_growing': votes_growing,
#         'confidence_score': confidence_score
#     }
#
#     return trend_direction, confidence, metrics
#
# print("✅ Improved trend analysis functions ready!")
# print("\n📊 Methods used:")
# print("   1. Exponential Smoothing (non-linear trends)")
# print("   2. Mann-Kendall Test (robust to outliers)")
# print("   3. Period Comparison (simple & reliable)")
# print("   4. CAGR (long-term growth rate)")
# print("   5. Consistency Score (direction stability)")
# print("   6. Volatility (CV)")

✅ Improved trend analysis functions ready!

📊 Methods used:
   1. Exponential Smoothing (non-linear trends)
   2. Mann-Kendall Test (robust to outliers)
   3. Period Comparison (simple & reliable)
   4. CAGR (long-term growth rate)
   5. Consistency Score (direction stability)
   6. Volatility (CV)


## COLLECTION CLASSIFICATIONS

In [10]:
# def analyze_with_improved_metrics(df, dead_threshold=90, contract_threshold=100):
#     """
#     ULTIMATE ANALYSIS with IMPROVED TREND DETECTION
#     """
#     print("\n" + "=" * 120)
#     print("🎯 ULTIMATE ANALYSIS (IMPROVED METRICS)")
#     print("=" * 120)
#     print(f"⚙️  Dead threshold: {dead_threshold} days")
#     print(f"⚙️  Contract threshold: {contract_threshold} meters")
#     print(f"📊 Using multi-method voting system for trend detection\n")
#
#     # 🔹 Ensure Sales_Type exists
#     if 'Sales_Type' not in df.columns:
#         print("ℹ️ Sales_Type not found → detecting now...")
#         df = detect_sales_type(df, contract_threshold)
#
#     # 🔹 Detect section / collection column
#     section_col = None
#     for col in df.columns:
#         if 'section' in col.lower() or 'collection' in col.lower():
#             section_col = col
#             break
#
#     if not section_col:
#         print("❌ No Section / Collection column found!")
#         return None
#
#     has_cost = 'Total_Profit' in df.columns
#     has_stock = 'CURRENT_STOCK' in df.columns
#
#     sections = (
#         df[section_col]
#         .dropna()
#         .astype(str)
#         .str.strip()
#     )
#     sections = sections[sections != ''].unique()
#
#     print(f"✅ Analyzing {len(sections):,} sections...\n")
#
#     results = []
#     analysis_date = df['Date'].max()
#
#     for section in tqdm(sections, desc="📊 Progress", ncols=100, colour='cyan'):
#         sec_data = df[df[section_col].astype(str).str.strip() == section].copy()
#
#         if len(sec_data) < 3:
#             continue
#
#         # ================= BASIC METRICS =================
#         total_revenue = sec_data['bal Value'].sum()
#         total_profit = sec_data['Total_Profit'].sum() if has_cost else 0
#         avg_margin = sec_data['Profit_Margin_%'].mean() if has_cost else 0
#
#         unique_skus = sec_data['SKU'].nunique()
#         last_sale = sec_data['Date'].max()
#         days_since = (analysis_date - last_sale).days
#
#         # ================= CONTRACT vs FAMILY =================
#         family_data = sec_data[sec_data['Sales_Type'] == 'FAMILY']
#         contract_data = sec_data[sec_data['Sales_Type'] == 'CONTRACT']
#
#         family_revenue = family_data['bal Value'].sum()
#         contract_revenue = contract_data['bal Value'].sum()
#         contract_pct = (contract_revenue / total_revenue * 100) if total_revenue > 0 else 0
#
#         # Contract spike detection
#         has_contract_spike = False
#         if len(contract_data) > 0 and len(family_data) > 0:
#             contract_monthly = (
#                 contract_data
#                 .groupby(contract_data['Date'].dt.to_period('M'))['bal Value']
#                 .sum()
#             )
#             family_monthly = (
#                 family_data
#                 .groupby(family_data['Date'].dt.to_period('M'))['bal Value']
#                 .sum()
#             )
#
#             if len(contract_monthly) > 0 and len(family_monthly) > 0:
#                 if contract_monthly.max() > family_monthly.mean() * 3:
#                     has_contract_spike = True
#
#         # ================= MONTHLY (FAMILY ONLY) =================
#         family_data = family_data.copy()
#         family_data['YM'] = family_data['Date'].dt.to_period('M').dt.to_timestamp()
#         monthly = (
#             family_data
#             .groupby('YM')['bal Value']
#             .sum()
#             .reset_index()
#             .sort_values('YM')
#         )
#
#         if len(monthly) < 3:
#             continue
#
#         monthly_values = monthly['bal Value'].values
#         trend_direction, trend_confidence, trend_metrics = determine_trend_consensus(monthly_values)
#
#         # ================= SEASONALITY =================
#         seasonality = 0
#         if len(monthly) >= 12:
#             monthly['Month'] = monthly['YM'].dt.month
#             month_avg = monthly.groupby('Month')['bal Value'].mean()
#             seasonality = month_avg.std() / month_avg.mean() if month_avg.mean() > 0 else 0
#
#         # ================= STOCK ANALYSIS =================
#         total_current_stock = 0
#         zero_stock_skus = []
#         low_stock_skus = []
#         stock_imbalance = False
#
#         if has_stock:
#             sku_stock = (
#                 sec_data
#                 .groupby('SKU')
#                 .agg({
#                     'bal Value': 'sum',
#                     'CURRENT_STOCK': 'first',
#                     'Date': 'max'
#                 })
#                 .reset_index()
#                 .sort_values('bal Value', ascending=False)
#             )
#
#             total_current_stock = sku_stock['CURRENT_STOCK'].sum()
#
#             for _, row in sku_stock.head(5).iterrows():
#                 stock = row['CURRENT_STOCK']
#                 days_inactive = (analysis_date - row['Date']).days
#
#                 if pd.isna(stock) or stock == 0:
#                     if days_inactive > 60:
#                         zero_stock_skus.append(row['SKU'])
#                 elif stock < 10:
#                     low_stock_skus.append((row['SKU'], stock))
#
#             if len(sku_stock) >= 3:
#                 top3_stock = sku_stock.head(3)['CURRENT_STOCK'].fillna(0).values
#                 if np.max(top3_stock) > np.min(top3_stock) * 10:
#                     stock_imbalance = True
#
#         # ================= CLASSIFICATION =================
#         classification = "Stable"
#
#         if has_cost and avg_margin < 0:
#             classification = "Loss-Making"
#         elif has_contract_spike and contract_pct > 50:
#             classification = "Contract Spike"
#         elif has_cost and avg_margin < 10 and trend_direction == "declining" and trend_confidence != "LOW":
#             classification = "Low Margin Declining"
#         elif seasonality > 0.5 and len(monthly) >= 12:
#             classification = "Seasonal"
#         elif days_since > dead_threshold:
#             if has_stock:
#                 if zero_stock_skus:
#                     classification = "Dead by True Stock-out"
#                 elif low_stock_skus:
#                     classification = "Dead by Insufficient Stock"
#                 elif stock_imbalance:
#                     classification = "Dead by Imbalanced Stock"
#                 elif total_current_stock > 0:
#                     classification = "Dead by Fashion"
#                 else:
#                     classification = "Dead (Unknown)"
#             else:
#                 classification = "Dead by Fashion"
#         elif trend_direction == "declining" and trend_confidence != "LOW":
#             classification = "Declining Fashion"
#         elif trend_direction == "growing" and trend_confidence != "LOW":
#             classification = "Growth"
#         elif trend_metrics['volatility'] > 0.7:
#             classification = "Volatile"
#
#         # ================= SAVE RESULT =================
#         results.append({
#             'Section': section,
#             'Classification': classification,
#             'Total_Revenue': total_revenue,
#             'Family_Revenue': family_revenue,
#             'Contract_Revenue': contract_revenue,
#             'Contract_%': contract_pct,
#             'Total_Profit': total_profit,
#             'Avg_Margin_%': avg_margin,
#             'Unique_SKUs': unique_skus,
#             'Days_Since_Sale': days_since,
#             'Trend_Direction': trend_direction,
#             'Trend_Confidence': trend_confidence,
#             'Period_Change_%': trend_metrics['period_change'],
#             'CAGR_%': trend_metrics['cagr'],
#             'Consistency_%': trend_metrics['consistency'],
#             'Volatility': trend_metrics['volatility'],
#             'Current_Stock': total_current_stock
#         })
#
#     results_df = pd.DataFrame(results)
#
#     print("\n✅ Analysis completed successfully")
#     return results_df_classi_collection


## SKU CLASSIFICATION

In [15]:
# # ============================================================================
# # ENHANCED BULK SKU ANALYSIS - WITH ROBUST METRICS
# # Handles outliers + Active months ratio + Selling days
# # ============================================================================
#
# import pandas as pd
# import numpy as np
# from tqdm.notebook import tqdm
# from datetime import datetime, timedelta
# import warnings
# warnings.filterwarnings('ignore')
#
# def analyze_single_sku_enhanced(sku_data, sku_name, analysis_date):
#     """
#     Enhanced analysis with robust metrics
#
#     New features:
#     - Median metrics (robust to outliers)
#     - Active months ratio
#     - Average selling days
#     - Outlier detection
#     - Consistency metrics
#     """
#
#     if len(sku_data) == 0:
#         return None
#
#     try:
#         # ═══════════════════════════════════════════════════════════════════
#         # BASIC METRICS
#         # ═══════════════════════════════════════════════════════════════════
#
#         total_revenue = sku_data['bal Value'].sum()
#         total_qty = sku_data['bal Qty'].sum()
#         first_sale = sku_data['Date'].min()
#         last_sale = sku_data['Date'].max()
#         days_since_last = (analysis_date - last_sale).days
#         total_days_active = (last_sale - first_sale).days + 1
#         num_transactions = len(sku_data)
#
#         # ═══════════════════════════════════════════════════════════════════
#         # PROFIT METRICS
#         # ═══════════════════════════════════════════════════════════════════
#
#         has_profit = 'Total_Profit' in sku_data.columns
#         total_profit = sku_data['Total_Profit'].sum() if has_profit else 0
#         avg_margin = (total_profit / total_revenue * 100) if has_profit and total_revenue > 0 else 0
#
#         # ✨ NEW: Median margin (robust!)
#         median_margin = 0
#         if has_profit and len(sku_data) > 0:
#             sku_data_with_margin = sku_data[sku_data['bal Value'] > 0].copy()
#             if len(sku_data_with_margin) > 0:
#                 sku_data_with_margin['Margin_%'] = (
#                     sku_data_with_margin['Total_Profit'] /
#                     sku_data_with_margin['bal Value'] * 100
#                 )
#                 median_margin = sku_data_with_margin['Margin_%'].median()
#
#         # ═══════════════════════════════════════════════════════════════════
#         # STOCK METRICS
#         # ═══════════════════════════════════════════════════════════════════
#
#         has_stock = 'CURRENT_STOCK' in sku_data.columns
#         current_stock = 0
#         if has_stock:
#             stock_values = sku_data['CURRENT_STOCK'].dropna()
#             if len(stock_values) > 0:
#                 current_stock = stock_values.iloc[0]
#
#         # ═══════════════════════════════════════════════════════════════════
#         # SALES TYPE
#         # ═══════════════════════════════════════════════════════════════════
#
#         has_sales_type = 'Sales_Type' in sku_data.columns
#         family_revenue = 0
#         contract_revenue = 0
#         contract_pct = 0
#         if has_sales_type:
#             family_revenue = sku_data[sku_data['Sales_Type'] == 'FAMILY']['bal Value'].sum()
#             contract_revenue = sku_data[sku_data['Sales_Type'] == 'CONTRACT']['bal Value'].sum()
#             contract_pct = (contract_revenue / total_revenue * 100) if total_revenue > 0 else 0
#
#         # ═══════════════════════════════════════════════════════════════════
#         # ✨ NEW: MONTHLY ANALYSIS (Robust Metrics!)
#         # ═══════════════════════════════════════════════════════════════════
#
#         sku_data['YearMonth'] = sku_data['Date'].dt.to_period('M')
#         monthly = sku_data.groupby('YearMonth').agg({
#             'bal Value': 'sum',
#             'bal Qty': 'sum'
#         })
#
#         # Total months in period
#         total_months = ((last_sale.year - first_sale.year) * 12 +
#                        (last_sale.month - first_sale.month) + 1)
#
#         # Active months (months with sales)
#         active_months = len(monthly)
#
#         # ✨ Active months ratio
#         active_months_ratio = (active_months / total_months) if total_months > 0 else 0
#
#         # Average metrics (mean - susceptible to outliers)
#         avg_monthly_revenue = monthly['bal Value'].mean() if active_months > 0 else 0
#         avg_monthly_qty = monthly['bal Qty'].mean() if active_months > 0 else 0
#
#         # ✨ Median metrics (robust to outliers!)
#         median_monthly_revenue = monthly['bal Value'].median() if active_months > 0 else 0
#         median_monthly_qty = monthly['bal Qty'].median() if active_months > 0 else 0
#
#         # ✨ Revenue consistency (ratio of median to mean)
#         # Close to 1.0 = consistent, much lower = has outliers
#         revenue_consistency = (median_monthly_revenue / avg_monthly_revenue) if avg_monthly_revenue > 0 else 0
#
#         # Volatility (CV)
#         cv = (monthly['bal Value'].std() / monthly['bal Value'].mean()) if monthly['bal Value'].mean() > 0 and active_months > 1 else 0
#
#         # ═══════════════════════════════════════════════════════════════════
#         # ✨ NEW: DAILY SELLING ANALYSIS
#         # ═══════════════════════════════════════════════════════════════════
#
#         # Days with sales
#         unique_selling_days = sku_data['Date'].nunique()
#
#         # ✨ Average days between sales
#         avg_days_between_sales = total_days_active / unique_selling_days if unique_selling_days > 1 else 0
#
#         # ✨ Selling days ratio (what % of days had sales)
#         selling_days_ratio = unique_selling_days / total_days_active if total_days_active > 0 else 0
#
#         # Daily revenue analysis
#         daily_revenue = sku_data.groupby('Date')['bal Value'].sum()
#
#         # ✨ Median daily revenue (when there are sales)
#         median_daily_revenue = daily_revenue.median() if len(daily_revenue) > 0 else 0
#         avg_daily_revenue = daily_revenue.mean() if len(daily_revenue) > 0 else 0
#
#         # ✨ Daily revenue consistency
#         daily_consistency = (median_daily_revenue / avg_daily_revenue) if avg_daily_revenue > 0 else 0
#
#         # ═══════════════════════════════════════════════════════════════════
#         # ✨ NEW: OUTLIER DETECTION
#         # ═══════════════════════════════════════════════════════════════════
#
#         # Detect outlier transactions (IQR method)
#         Q1 = sku_data['bal Value'].quantile(0.25)
#         Q3 = sku_data['bal Value'].quantile(0.75)
#         IQR = Q3 - Q1
#         outlier_threshold = Q3 + (1.5 * IQR)
#
#         outliers = sku_data[sku_data['bal Value'] > outlier_threshold]
#         num_outliers = len(outliers)
#         outlier_pct = (num_outliers / num_transactions * 100) if num_transactions > 0 else 0
#         outlier_revenue = outliers['bal Value'].sum()
#         outlier_revenue_pct = (outlier_revenue / total_revenue * 100) if total_revenue > 0 else 0
#
#         # ✨ Revenue without outliers (more realistic)
#         revenue_without_outliers = total_revenue - outlier_revenue
#         qty_without_outliers = total_qty - outliers['bal Qty'].sum()
#
#         # ✨ Adjusted metrics (excluding outliers)
#         adjusted_avg_monthly = revenue_without_outliers / active_months if active_months > 0 else 0
#
#         # ═══════════════════════════════════════════════════════════════════
#         # ✨ NEW: PURCHASE PATTERN ANALYSIS
#         # ═══════════════════════════════════════════════════════════════════
#
#         # Transaction size distribution
#         transaction_sizes = sku_data['bal Value'].values
#
#         # Small/Medium/Large transactions
#         small_threshold = np.percentile(transaction_sizes, 33)
#         large_threshold = np.percentile(transaction_sizes, 67)
#
#         small_txns = len(sku_data[sku_data['bal Value'] <= small_threshold])
#         medium_txns = len(sku_data[(sku_data['bal Value'] > small_threshold) &
#                                    (sku_data['bal Value'] <= large_threshold)])
#         large_txns = len(sku_data[sku_data['bal Value'] > large_threshold])
#
#         small_pct = (small_txns / num_transactions * 100) if num_transactions > 0 else 0
#         medium_pct = (medium_txns / num_transactions * 100) if num_transactions > 0 else 0
#         large_pct = (large_txns / num_transactions * 100) if num_transactions > 0 else 0
#
#         # ✨ Purchase pattern classification
#         if large_pct > 50:
#             purchase_pattern = "Large Orders"
#         elif small_pct > 70:
#             purchase_pattern = "Frequent Small"
#         elif medium_pct > 60:
#             purchase_pattern = "Regular Medium"
#         else:
#             purchase_pattern = "Mixed"
#
#         # ═══════════════════════════════════════════════════════════════════
#         # TREND ANALYSIS
#         # ═══════════════════════════════════════════════════════════════════
#
#         trend_direction = "Unknown"
#         trend_strength = 0
#         period_change_pct = 0
#
#         if active_months >= 6:
#             period_size = max(2, active_months // 4)
#             first_period_avg = monthly['bal Value'].iloc[:period_size].mean()
#             last_period_avg = monthly['bal Value'].iloc[-period_size:].mean()
#
#             if first_period_avg > 0:
#                 period_change_pct = ((last_period_avg - first_period_avg) / first_period_avg) * 100
#
#                 if period_change_pct > 15:
#                     trend_direction = "Growing"
#                     trend_strength = min(abs(period_change_pct) / 15, 1.0)
#                 elif period_change_pct < -15:
#                     trend_direction = "Declining"
#                     trend_strength = min(abs(period_change_pct) / 15, 1.0)
#                 else:
#                     if cv < 0.3:
#                         trend_direction = "Stable"
#                     elif cv > 0.7:
#                         trend_direction = "Volatile"
#                     else:
#                         trend_direction = "Stable"
#                     trend_strength = 0.5
#
#         # ═══════════════════════════════════════════════════════════════════
#         # STOCK-OUT ANALYSIS
#         # ═══════════════════════════════════════════════════════════════════
#
#         is_stockout = False
#         stockout_date = None
#         days_out_of_stock = 0
#         revenue_drop_pct = 0
#         estimated_loss_revenue = 0
#         estimated_loss_profit = 0
#
#         if has_stock and current_stock == 0 and days_since_last > 60:
#             is_stockout = True
#
#             sku_data_sorted = sku_data.sort_values('Date')
#             dates = sku_data_sorted['Date'].values
#             for i in range(len(dates) - 1):
#                 gap = (dates[i+1] - dates[i]) / np.timedelta64(1, 'D')
#                 if gap > 60:
#                     stockout_date = dates[i]
#                     break
#
#             if stockout_date is None:
#                 stockout_date = last_sale
#
#             days_out_of_stock = (analysis_date - stockout_date).days
#
#             before = sku_data[sku_data['Date'] < stockout_date]
#             after = sku_data[sku_data['Date'] >= stockout_date]
#
#             if len(before) > 0:
#                 before_months = len(before.groupby(before['Date'].dt.to_period('M')))
#                 after_months = len(after.groupby(after['Date'].dt.to_period('M')))
#
#                 # ✨ Use median for more robust estimate
#                 before_monthly = before.groupby(before['Date'].dt.to_period('M'))['bal Value'].sum().median()
#                 after_monthly = after.groupby(after['Date'].dt.to_period('M'))['bal Value'].sum().median() if len(after) > 0 else 0
#
#                 if before_monthly > 0:
#                     revenue_drop_pct = ((after_monthly - before_monthly) / before_monthly) * 100
#
#                 months_out = days_out_of_stock / 30
#                 estimated_loss_revenue = before_monthly * months_out
#
#                 if has_profit:
#                     before_profit = before.groupby(before['Date'].dt.to_period('M'))['Total_Profit'].sum().median()
#                     estimated_loss_profit = before_profit * months_out
#
#         # ═══════════════════════════════════════════════════════════════════
#         # ✨ NEW: ENHANCED STATUS DETERMINATION
#         # ═══════════════════════════════════════════════════════════════════
#
#         status = "Unknown"
#         priority = 999
#
#         # Critical issues first
#         if has_profit and avg_margin < 0:
#             status = "Loss-Making"
#             priority = 0
#         elif is_stockout:
#             status = "Stock-Out"
#             priority = 1
#         elif has_stock and current_stock < 10 and current_stock > 0:
#             status = "Low Stock"
#             priority = 2
#         elif days_since_last > 180:
#             if has_stock and current_stock > 0:
#                 status = "Dead (Stock Available)"
#                 priority = 5
#             else:
#                 status = "Dead (No Stock)"
#                 priority = 4
#         # ✨ New: Consider active months ratio
#         elif active_months_ratio < 0.3 and active_months >= 6:
#             status = "Sporadic (Inactive)"
#             priority = 6
#         elif trend_direction == "Declining":
#             if has_stock and current_stock < 50:
#                 status = "Declining (Low Stock)"
#                 priority = 3
#             else:
#                 status = "Declining"
#                 priority = 6
#         elif trend_direction == "Growing":
#             status = "Growing"
#             priority = 10
#         elif trend_direction == "Stable":
#             # ✨ Differentiate by consistency
#             if revenue_consistency > 0.8:
#                 status = "Stable (Consistent)"
#                 priority = 11
#             else:
#                 status = "Stable (Variable)"
#                 priority = 9
#         elif trend_direction == "Volatile":
#             status = "Volatile"
#             priority = 7
#         else:
#             status = "Active"
#             priority = 9
#
#         # ═══════════════════════════════════════════════════════════════════
#         # RETURN COMPREHENSIVE RESULTS
#         # ═══════════════════════════════════════════════════════════════════
#
#         return {
#             # Basic info
#             'SKU': sku_name,
#             'Status': status,
#             'Priority': priority,
#
#             # Revenue metrics
#             'Total_Revenue': total_revenue,
#             'Total_Quantity': total_qty,
#             'Num_Transactions': num_transactions,
#
#             # ✨ Robust revenue metrics (NEW!)
#             'Avg_Monthly_Revenue': avg_monthly_revenue,
#             'Median_Monthly_Revenue': median_monthly_revenue,
#             'Revenue_Consistency': revenue_consistency,
#             'Avg_Daily_Revenue': avg_daily_revenue,
#             'Median_Daily_Revenue': median_daily_revenue,
#             'Daily_Consistency': daily_consistency,
#
#             # ✨ Adjusted for outliers (NEW!)
#             'Revenue_Without_Outliers': revenue_without_outliers,
#             'Adjusted_Avg_Monthly': adjusted_avg_monthly,
#             'Num_Outliers': num_outliers,
#             'Outlier_%': outlier_pct,
#             'Outlier_Revenue_%': outlier_revenue_pct,
#
#             # Profit metrics
#             'Total_Profit': total_profit,
#             'Avg_Margin_%': avg_margin,
#             'Median_Margin_%': median_margin,  # ✨ NEW!
#
#             # Stock
#             'Current_Stock': current_stock,
#
#             # Time metrics
#             'First_Sale': first_sale,
#             'Last_Sale': last_sale,
#             'Days_Since_Last_Sale': days_since_last,
#             'Days_Active': total_days_active,
#
#             # ✨ Monthly activity (NEW!)
#             'Total_Months': total_months,
#             'Active_Months': active_months,
#             'Active_Months_Ratio': active_months_ratio,
#
#             # ✨ Daily selling pattern (NEW!)
#             'Unique_Selling_Days': unique_selling_days,
#             'Selling_Days_Ratio': selling_days_ratio,
#             'Avg_Days_Between_Sales': avg_days_between_sales,
#
#             # Sales type
#             'Family_Revenue': family_revenue,
#             'Contract_Revenue': contract_revenue,
#             'Contract_%': contract_pct,
#
#             # ✨ Purchase pattern (NEW!)
#             'Purchase_Pattern': purchase_pattern,
#             'Small_Txns_%': small_pct,
#             'Medium_Txns_%': medium_pct,
#             'Large_Txns_%': large_pct,
#
#             # Trend
#             'Trend_Direction': trend_direction,
#             'Trend_Strength': trend_strength,
#             'Period_Change_%': period_change_pct,
#             'Volatility_CV': cv,
#
#             # Stock-out
#             'Is_Stockout': is_stockout,
#             'Stockout_Date': stockout_date,
#             'Days_Out_of_Stock': days_out_of_stock,
#             'Revenue_Drop_%': revenue_drop_pct,
#             'Estimated_Loss_Revenue': estimated_loss_revenue,
#             'Estimated_Loss_Profit': estimated_loss_profit
#         }
#
#     except Exception as e:
#         print(f"Error analyzing SKU {sku_name}: {e}")
#         return None
#
# def bulk_sku_analysis_enhanced(df, save_to_excel=True, output_file='ENHANCED_SKU_ANALYSIS.xlsx'):
#     """
#     Enhanced bulk analysis with robust metrics
#     """
#
#     print("=" * 120)
#     print("🚀 ENHANCED BULK SKU ANALYSIS - STARTING")
#     print("=" * 120)
#
#     analysis_date = df['Date'].max()
#     print(f"📅 Analysis Date: {analysis_date.strftime('%Y-%m-%d')}")
#
#     all_skus = df['SKU'].dropna().astype(str).str.strip().unique()
#     total_skus = len(all_skus)
#     print(f"📦 Total SKUs: {total_skus:,}")
#     print(f"⏱️  Estimated time: ~{total_skus / 1000:.1f} minutes")
#     print()
#
#     has_profit = 'Total_Profit' in df.columns
#     has_stock = 'CURRENT_STOCK' in df.columns
#     has_sales_type = 'Sales_Type' in df.columns
#
#     print("📊 Available data:")
#     print(f" 💰 Profit: {'✅' if has_profit else '❌'}")
#     print(f" 📦 Stock: {'✅' if has_stock else '❌'}")
#     print(f" 🏢 Sales type: {'✅' if has_sales_type else '❌'}")
#     print()
#     print("✨ NEW METRICS ADDED:")
#     print(" • Median monthly/daily revenue (robust to outliers)")
#     print(" • Active months ratio")
#     print(" • Selling days analysis")
#     print(" • Outlier detection & adjustment")
#     print(" • Revenue consistency scores")
#     print(" • Purchase pattern classification")
#     print()
#
#     results = []
#     print("🔄 Processing SKUs...")
#     print("-" * 120)
#
#     for sku in tqdm(all_skus, desc="📊 Analyzing", ncols=100, colour='cyan'):
#         sku_data = df[df['SKU'].astype(str).str.strip() == sku].copy()
#         if len(sku_data) > 0:
#             result = analyze_single_sku_enhanced(sku_data, sku, analysis_date)
#             if result:
#                 results.append(result)
#
#     print()
#     print("=" * 120)
#     print(f"✅ Analysis Complete! Processed {len(results):,} SKUs")
#     print("=" * 120)
#
#     results_df = pd.DataFrame(results)
#     results_df = results_df.sort_values(['Priority', 'Total_Revenue'], ascending=[True, False])
#
#     # ═══════════════════════════════════════════════════════════════════
#     # ENHANCED SUMMARY
#     # ═══════════════════════════════════════════════════════════════════
#
#     print("\n📊 SUMMARY BY STATUS:")
#     print("-" * 120)
#
#     status_summary = results_df.groupby('Status').agg({
#         'SKU': 'count',
#         'Total_Revenue': 'sum',
#         'Median_Monthly_Revenue': 'mean',  # ✨ Average of medians
#         'Total_Profit': 'sum',
#         'Current_Stock': 'sum',
#         'Active_Months_Ratio': 'mean'  # ✨ Average activity
#     }).reset_index()
#
#     status_summary = status_summary.sort_values('Total_Revenue', ascending=False)
#
#     for _, row in status_summary.iterrows():
#         print(f"{row['Status']:30s}: {row['SKU']:>6,} SKUs | "
#               f"Revenue: {row['Total_Revenue']:>15,.0f} | "
#               f"Median/Month: {row['Median_Monthly_Revenue']:>10,.0f} | "
#               f"Activity: {row['Active_Months_Ratio']:>5.1%}")
#
#     print("-" * 120)
#
#     # ✨ NEW: Outlier analysis
#     print("\n📊 OUTLIER ANALYSIS:")
#     print("-" * 120)
#     skus_with_outliers = results_df[results_df['Outlier_%'] > 10]
#     if len(skus_with_outliers) > 0:
#         print(f"⚠️  {len(skus_with_outliers):,} SKUs have significant outliers (>10% of transactions)")
#         print(f"   Total outlier revenue: {skus_with_outliers['Outlier_Revenue_%'].sum()/ len(skus_with_outliers):.1f}% of their revenue")
#         print(f"   These SKUs may have contracts or bulk orders affecting averages")
#
#     # ✨ NEW: Activity analysis
#     print("\n📊 ACTIVITY ANALYSIS:")
#     print("-" * 120)
#     low_activity = results_df[results_df['Active_Months_Ratio'] < 0.3]
#     print(f"🔴 Sporadic: {len(low_activity):,} SKUs active <30% of months")
#
#     medium_activity = results_df[(results_df['Active_Months_Ratio'] >= 0.3) &
#                                  (results_df['Active_Months_Ratio'] < 0.7)]
#     print(f"🟡 Moderate: {len(medium_activity):,} SKUs active 30-70% of months")
#
#     high_activity = results_df[results_df['Active_Months_Ratio'] >= 0.7]
#     print(f"🟢 Consistent: {len(high_activity):,} SKUs active >70% of months")
#
#     # Alerts
#     if has_stock:
#         stockouts = results_df[results_df['Is_Stockout'] == True]
#         if len(stockouts) > 0:
#             print(f"\n🚨 CRITICAL: {len(stockouts):,} SKUs in STOCK-OUT!")
#             print(f"   Loss: {stockouts['Estimated_Loss_Revenue'].sum():,.0f} SAR")
#
#     if has_profit:
#         loss_making = results_df[results_df['Avg_Margin_%'] < 0]
#         if len(loss_making) > 0:
#             print(f"\n❌ WARNING: {len(loss_making):,} SKUs LOSING MONEY!")
#             print(f"   Total Loss: {loss_making['Total_Profit'].sum():,.0f} SAR")
#
#     # ═══════════════════════════════════════════════════════════════════
#     # SAVE TO EXCEL
#     # ═══════════════════════════════════════════════════════════════════
#
#     if save_to_excel:
#         print(f"\n💾 Saving to: {output_file}")
#
#         with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
#             results_df.to_excel(writer, sheet_name='All SKUs', index=False)
#
#             # Critical
#             critical = results_df[results_df['Priority'] <= 1]
#             if len(critical) > 0:
#                 critical.to_excel(writer, sheet_name='CRITICAL', index=False)
#
#             # ✨ NEW: High outliers
#             high_outliers = results_df[results_df['Outlier_%'] > 20].sort_values('Outlier_Revenue_%', ascending=False)
#             if len(high_outliers) > 0:
#                 high_outliers.to_excel(writer, sheet_name='High Outliers', index=False)
#
#             # ✨ NEW: Sporadic SKUs
#             if len(low_activity) > 0:
#                 low_activity_sorted = low_activity.sort_values('Total_Revenue', ascending=False)
#                 low_activity_sorted.to_excel(writer, sheet_name='Sporadic Activity', index=False)
#
#             # ✨ NEW: Consistent performers
#             consistent = results_df[(results_df['Revenue_Consistency'] > 0.8) &
#                                    (results_df['Active_Months_Ratio'] > 0.7)]
#             if len(consistent) > 0:
#                 consistent_sorted = consistent.sort_values('Total_Revenue', ascending=False)
#                 consistent_sorted.to_excel(writer, sheet_name='Consistent Performers', index=False)
#
#             # Stock-outs
#             if has_stock:
#                 stockouts = results_df[results_df['Is_Stockout'] == True]
#                 if len(stockouts) > 0:
#                     stockouts.to_excel(writer, sheet_name='Stock-Outs', index=False)
#
#             # Loss-making
#             if has_profit:
#                 loss_making = results_df[results_df['Avg_Margin_%'] < 0]
#                 if len(loss_making) > 0:
#                     loss_making.to_excel(writer, sheet_name='Loss-Making', index=False)
#
#             # Growing
#             growing = results_df[results_df['Status'] == 'Growing']
#             if len(growing) > 0:
#                 growing.to_excel(writer, sheet_name='Growing', index=False)
#
#             # Summary
#             status_summary.to_excel(writer, sheet_name='Summary', index=False)
#
#         print(f"✅ Saved successfully!")
#
#     print("\n" + "=" * 120)
#     print("🎉 ENHANCED ANALYSIS COMPLETE!")
#     print("=" * 120)
#
#     return results_df
#
# print("✅ Enhanced SKU Analysis Loaded!")
# print()
# print("✨ NEW FEATURES:")
# print("  • Median metrics (robust to outliers)")
# print("  • Active months ratio")
# print("  • Selling days analysis")
# print("  • Outlier detection")
# print("  • Revenue consistency scores")
# print("  • Purchase pattern classification")
# print()
# print("🚀 To run:")
# print("  results_df = bulk_sku_analysis_enhanced(combined_df)")

✅ Enhanced SKU Analysis Loaded!

✨ NEW FEATURES:
  • Median metrics (robust to outliers)
  • Active months ratio
  • Selling days analysis
  • Outlier detection
  • Revenue consistency scores
  • Purchase pattern classification

🚀 To run:
  results_df = bulk_sku_analysis_enhanced(combined_df)


## SAVING DATA FOR NEXT NOTEBOOK 

In [ ]:
# # =============================================================================
# # FINAL EXECUTION CELL – SAFE & READY FOR NEXT NOTEBOOK            خON THE COLLECTION LEVEL
# # =============================================================================
#
# if 'combined_df' not in globals():
#     print("❌ Run Part 1 first! combined_df not found.")
#
# else:
#     try:
#         dead_threshold = int(input("Dead threshold days (default 90): ") or 90)
#     except:
#         dead_threshold = 90
#
#     try:
#         contract_threshold = int(input("Contract threshold meters (default 100): ") or 100)
#     except:
#         contract_threshold = 100
#
#     print("\n🚀 Running improved analysis...")
#
#     analysis_results = analyze_with_improved_metrics(
#         combined_df,
#         dead_threshold,
#         contract_threshold
#     )
#
#     if analysis_results is not None and len(analysis_results) > 0:
#         print("\n✅ Analysis complete!")
#
#         print("\n📋 Top 10 SKUs by Revenue:")
#         display(
#             analysis_results
#             .nlargest(10, 'Total_Revenue')[[
#                 'Section',
#                 'Classification',
#                 'Total_Revenue',
#                 'Trend_Direction',
#                 'Trend_Confidence',
#                 'Period_Change_%',
#                 'Consistency_%',
#                 'Volatility'
#             ]]
#         )
#
#         # Save for next notebook
#         print("\n💾 Saving results for Part 3...")
#         %store analysis_results
#         print("✅ Variable 'analysis_results' stored successfully!")
#
#     else:
#         print("⚠️ Analysis finished but no results were generated.")


In [12]:
# ## THE FINAL OPTIMIZED SKU ANALYSIS
#
# كان بيضيع وقت وميموري
#
# # ============================================================================
# # OPTIMIZED SKU ANALYSIS SYSTEM - MERGED & ENHANCED
# # Fast execution + Robust metrics + Advanced trend analysis
# # ============================================================================
#
# import pandas as pd
# import numpy as np
# from scipy.stats import kendalltau
# from statsmodels.tsa.holtwinters import ExponentialSmoothing
# from tqdm.notebook import tqdm
# import warnings
# warnings.filterwarnings('ignore')
#
#
# # ============================================================================
# # PART 1: ADVANCED TREND ANALYSIS ENGINE
# # ============================================================================
#
# class TrendAnalyzer:
#     """
#     Fast vectorized trend analysis using multiple methods
#     """
#
#     @staticmethod
#     def analyze_trend(monthly_values):
#         """
#         Comprehensive trend analysis combining 4 methods
#         Returns: (direction, confidence, metrics_dict)
#
#         Optimized for speed with NumPy operations
#         """
#         if len(monthly_values) < 6:
#             return "Unknown", "LOW", {}
#
#         try:
#             # Convert to numpy array for speed
#             values = np.array(monthly_values)
#
#             # ═══════════════════════════════════════════════════════════════
#             # METHOD 1: Exponential Smoothing Trend
#             # ═══════════════════════════════════════════════════════════════
#             exp_trend, exp_strength = 0, 0
#             try:
#                 model = ExponentialSmoothing(
#                     values,
#                     trend='add',
#                     seasonal=None
#                 )
#                 fit = model.fit(
#                     smoothing_level=0.3,
#                     smoothing_trend=0.1,
#                     optimized=False
#                 )
#                 trend = fit.trend
#                 exp_trend = trend[-3:].mean() if len(trend) >= 3 else trend[-1]
#                 positive_trends = (trend > 0).sum()
#                 exp_strength = abs(positive_trends / len(trend) - 0.5) * 2
#             except:
#                 pass
#
#             # ═══════════════════════════════════════════════════════════════
#             # METHOD 2: Mann-Kendall Test (Vectorized)
#             # ═══════════════════════════════════════════════════════════════
#             mk_tau, mk_p = 0, 1
#             try:
#                 months = np.arange(len(values))
#                 mk_tau, mk_p = kendalltau(months, values)
#             except:
#                 pass
#
#             # ═══════════════════════════════════════════════════════════════
#             # METHOD 3: Period Comparison (Fast NumPy)
#             # ═══════════════════════════════════════════════════════════════
#             period_size = max(2, len(values) // 4)
#             first_period = values[:period_size].mean()
#             last_period = values[-period_size:].mean()
#
#             period_change = 0
#             if first_period > 0:
#                 period_change = (last_period - first_period) / first_period * 100
#
#             # ═══════════════════════════════════════════════════════════════
#             # METHOD 4: CAGR (if applicable)
#             # ═══════════════════════════════════════════════════════════════
#             cagr = None
#             if len(values) >= 12:
#                 beginning = values[:3].mean()
#                 ending = values[-3:].mean()
#                 if beginning > 0 and ending > 0:
#                     years = len(values) / 12
#                     cagr = ((ending / beginning) ** (1 / years) - 1) * 100
#
#             # ═══════════════════════════════════════════════════════════════
#             # METHOD 5: Consistency Score (Vectorized)
#             # ═══════════════════════════════════════════════════════════════
#             changes = np.diff(values)
#             consistency = 0
#             if len(changes) > 0:
#                 positive = (changes > 0).sum()
#                 negative = (changes < 0).sum()
#                 consistency = max(positive, negative) / len(changes) * 100
#
#             # ═══════════════════════════════════════════════════════════════
#             # Volatility (CV)
#             # ═══════════════════════════════════════════════════════════════
#             cv = values.std() / values.mean() if values.mean() > 0 else 0
#
#             # ═══════════════════════════════════════════════════════════════
#             # VOTING SYSTEM
#             # ═══════════════════════════════════════════════════════════════
#             votes_declining = 0
#             votes_growing = 0
#
#             # Vote 1: Exponential Smoothing
#             if exp_trend < -500:
#                 votes_declining += 1
#             elif exp_trend > 500:
#                 votes_growing += 1
#
#             # Vote 2: Mann-Kendall
#             if mk_tau < -0.2 and mk_p < 0.05:
#                 votes_declining += 1
#             elif mk_tau > 0.2 and mk_p < 0.05:
#                 votes_growing += 1
#
#             # Vote 3: Period Comparison
#             if period_change < -15:
#                 votes_declining += 1
#             elif period_change > 15:
#                 votes_growing += 1
#
#             # Vote 4: CAGR
#             if cagr is not None:
#                 if cagr < -10:
#                     votes_declining += 1
#                 elif cagr > 10:
#                     votes_growing += 1
#
#             # ═══════════════════════════════════════════════════════════════
#             # DETERMINE TREND
#             # ═══════════════════════════════════════════════════════════════
#             total_votes = 4 if cagr is not None else 3
#
#             if votes_declining >= 2:
#                 trend_direction = "Declining"
#                 confidence_score = votes_declining / total_votes
#             elif votes_growing >= 2:
#                 trend_direction = "Growing"
#                 confidence_score = votes_growing / total_votes
#             else:
#                 if cv < 0.3:
#                     trend_direction = "Stable"
#                 elif cv > 0.7:
#                     trend_direction = "Volatile"
#                 else:
#                     trend_direction = "Stable"
#                 confidence_score = 0.5
#
#             # Adjust confidence by consistency and volatility
#             if consistency > 70 and cv < 0.5:
#                 confidence_score *= 1.2
#             elif consistency < 50 or cv > 0.8:
#                 confidence_score *= 0.8
#
#             confidence_score = min(confidence_score, 1.0)
#
#             # Confidence level
#             if confidence_score >= 0.75:
#                 confidence = "HIGH"
#             elif confidence_score >= 0.5:
#                 confidence = "MEDIUM"
#             else:
#                 confidence = "LOW"
#
#             metrics = {
#                 'exp_trend': exp_trend,
#                 'mk_tau': mk_tau,
#                 'mk_p': mk_p,
#                 'period_change': period_change,
#                 'cagr': cagr,
#                 'consistency': consistency,
#                 'cv': cv,
#                 'confidence_score': confidence_score
#             }
#
#             return trend_direction, confidence, metrics
#
#         except Exception as e:
#             return "Unknown", "LOW", {}
#
#
# # ============================================================================
# # PART 2: OPTIMIZED SKU ANALYZER
# # ============================================================================
#
# class SKUAnalyzer:
#     """
#     High-performance SKU analysis with vectorized operations
#     """
#
#     def __init__(self, df, analysis_date=None):
#         self.df = df
#         self.analysis_date = analysis_date or df['Date'].max()
#         self.trend_analyzer = TrendAnalyzer()
#
#         # Check available columns
#         self.has_profit = 'Total_Profit' in df.columns
#         self.has_stock = 'CURRENT_STOCK' in df.columns
#         self.has_sales_type = 'Sales_Type' in df.columns
#
#     def analyze_single_sku(self, sku_data, sku_name):
#         """
#         Optimized single SKU analysis
#         Uses NumPy for all calculations where possible
#         """
#         if len(sku_data) == 0:
#             return None
#
#         try:
#             # ═══════════════════════════════════════════════════════════════
#             # BASIC METRICS (Vectorized)
#             # ═══════════════════════════════════════════════════════════════
#             total_revenue = sku_data['bal Value'].sum()
#             total_qty = sku_data['bal Qty'].sum()
#             first_sale = sku_data['Date'].min()
#             last_sale = sku_data['Date'].max()
#             days_since_last = (self.analysis_date - last_sale).days
#             total_days_active = (last_sale - first_sale).days + 1
#             num_transactions = len(sku_data)
#
#             # ═══════════════════════════════════════════════════════════════
#             # PROFIT METRICS
#             # ═══════════════════════════════════════════════════════════════
#             total_profit = 0
#             avg_margin = 0
#             median_margin = 0
#
#             if self.has_profit:
#                 total_profit = sku_data['Total_Profit'].sum()
#                 if total_revenue > 0:
#                     avg_margin = (total_profit / total_revenue) * 100
#
#                 # Median margin calculation (vectorized)
#                 mask = sku_data['bal Value'] > 0
#                 if mask.any():
#                     margins = (sku_data.loc[mask, 'Total_Profit'] /
#                               sku_data.loc[mask, 'bal Value'] * 100)
#                     median_margin = margins.median()
#
#             # ═══════════════════════════════════════════════════════════════
#             # STOCK METRICS
#             # ═══════════════════════════════════════════════════════════════
#             current_stock = 0
#             if self.has_stock:
#                 stock_values = sku_data['CURRENT_STOCK'].dropna()
#                 if len(stock_values) > 0:
#                     current_stock = stock_values.iloc[0]
#
#             # ═══════════════════════════════════════════════════════════════
#             # SALES TYPE
#             # ═══════════════════════════════════════════════════════════════
#             family_revenue = 0
#             contract_revenue = 0
#             contract_pct = 0
#
#             if self.has_sales_type:
#                 family_mask = sku_data['Sales_Type'] == 'FAMILY'
#                 contract_mask = sku_data['Sales_Type'] == 'CONTRACT'
#                 family_revenue = sku_data.loc[family_mask, 'bal Value'].sum()
#                 contract_revenue = sku_data.loc[contract_mask, 'bal Value'].sum()
#                 if total_revenue > 0:
#                     contract_pct = (contract_revenue / total_revenue) * 100
#
#             # ═══════════════════════════════════════════════════════════════
#             # MONTHLY ANALYSIS (Optimized with groupby)
#             # ═══════════════════════════════════════════════════════════════
#             sku_data['YearMonth'] = sku_data['Date'].dt.to_period('M')
#             monthly = sku_data.groupby('YearMonth', observed=True).agg({
#                 'bal Value': 'sum',
#                 'bal Qty': 'sum'
#             })
#
#             total_months = ((last_sale.year - first_sale.year) * 12 +
#                            (last_sale.month - first_sale.month) + 1)
#             active_months = len(monthly)
#             active_months_ratio = active_months / total_months if total_months > 0 else 0
#
#             # Monthly metrics (vectorized)
#             monthly_values = monthly['bal Value'].values
#             avg_monthly_revenue = monthly_values.mean() if active_months > 0 else 0
#             median_monthly_revenue = np.median(monthly_values) if active_months > 0 else 0
#             revenue_consistency = (median_monthly_revenue / avg_monthly_revenue
#                                   if avg_monthly_revenue > 0 else 0)
#
#             cv = (monthly_values.std() / monthly_values.mean()
#                  if monthly_values.mean() > 0 and active_months > 1 else 0)
#
#             # ═══════════════════════════════════════════════════════════════
#             # DAILY ANALYSIS (Optimized)
#             # ═══════════════════════════════════════════════════════════════
#             unique_selling_days = sku_data['Date'].nunique()
#             selling_days_ratio = (unique_selling_days / total_days_active
#                                  if total_days_active > 0 else 0)
#             avg_days_between_sales = (total_days_active / unique_selling_days
#                                      if unique_selling_days > 1 else 0)
#
#             daily_revenue = sku_data.groupby('Date')['bal Value'].sum()
#             daily_values = daily_revenue.values
#             median_daily_revenue = np.median(daily_values) if len(daily_values) > 0 else 0
#             avg_daily_revenue = daily_values.mean() if len(daily_values) > 0 else 0
#             daily_consistency = (median_daily_revenue / avg_daily_revenue
#                                 if avg_daily_revenue > 0 else 0)
#
#             # ═══════════════════════════════════════════════════════════════
#             # OUTLIER DETECTION (Vectorized IQR)
#             # ═══════════════════════════════════════════════════════════════
#             transaction_values = sku_data['bal Value'].values
#             Q1 = np.percentile(transaction_values, 25)
#             Q3 = np.percentile(transaction_values, 75)
#             IQR = Q3 - Q1
#             outlier_threshold = Q3 + (1.5 * IQR)
#
#             outlier_mask = transaction_values > outlier_threshold
#             num_outliers = outlier_mask.sum()
#             outlier_pct = (num_outliers / num_transactions * 100) if num_transactions > 0 else 0
#             outlier_revenue = transaction_values[outlier_mask].sum()
#             outlier_revenue_pct = (outlier_revenue / total_revenue * 100) if total_revenue > 0 else 0
#
#             revenue_without_outliers = total_revenue - outlier_revenue
#             adjusted_avg_monthly = (revenue_without_outliers / active_months
#                                    if active_months > 0 else 0)
#
#             # ═══════════════════════════════════════════════════════════════
#             # PURCHASE PATTERN (Vectorized percentiles)
#             # ═══════════════════════════════════════════════════════════════
#             small_threshold = np.percentile(transaction_values, 33)
#             large_threshold = np.percentile(transaction_values, 67)
#
#             small_txns = (transaction_values <= small_threshold).sum()
#             large_txns = (transaction_values > large_threshold).sum()
#             medium_txns = num_transactions - small_txns - large_txns
#
#             small_pct = (small_txns / num_transactions * 100) if num_transactions > 0 else 0
#             large_pct = (large_txns / num_transactions * 100) if num_transactions > 0 else 0
#             medium_pct = 100 - small_pct - large_pct
#
#             if large_pct > 50:
#                 purchase_pattern = "Large Orders"
#             elif small_pct > 70:
#                 purchase_pattern = "Frequent Small"
#             elif medium_pct > 60:
#                 purchase_pattern = "Regular Medium"
#             else:
#                 purchase_pattern = "Mixed"
#
#             # ═══════════════════════════════════════════════════════════════
#             # ADVANCED TREND ANALYSIS (Using TrendAnalyzer)
#             # ═══════════════════════════════════════════════════════════════
#             trend_direction = "Unknown"
#             trend_confidence = "LOW"
#             trend_metrics = {}
#
#             if active_months >= 6:
#                 trend_direction, trend_confidence, trend_metrics = \
#                     self.trend_analyzer.analyze_trend(monthly_values)
#
#             # ═══════════════════════════════════════════════════════════════
#             # STOCK-OUT ANALYSIS (Optimized)
#             # ═══════════════════════════════════════════════════════════════
#             is_stockout = False
#             stockout_date = None
#             days_out_of_stock = 0
#             revenue_drop_pct = 0
#             estimated_loss_revenue = 0
#             estimated_loss_profit = 0
#
#             if self.has_stock and current_stock == 0 and days_since_last > 60:
#                 is_stockout = True
#
#                 # Find stockout date (vectorized)
#                 sku_data_sorted = sku_data.sort_values('Date')
#                 dates = sku_data_sorted['Date'].values
#
#                 if len(dates) > 1:
#                     gaps = np.diff(dates).astype('timedelta64[D]').astype(int)
#                     gap_idx = np.where(gaps > 60)[0]
#                     if len(gap_idx) > 0:
#                         stockout_date = dates[gap_idx[0]]
#                     else:
#                         stockout_date = last_sale
#                 else:
#                     stockout_date = last_sale
#
#                 days_out_of_stock = (self.analysis_date - stockout_date).days
#
#                 before = sku_data[sku_data['Date'] < stockout_date]
#
#                 if len(before) > 0:
#                     before_monthly = before.groupby(
#                         before['Date'].dt.to_period('M'), observed=True
#                     )['bal Value'].sum()
#                     before_monthly_median = before_monthly.median()
#
#                     months_out = days_out_of_stock / 30
#                     estimated_loss_revenue = before_monthly_median * months_out
#
#                     if self.has_profit:
#                         before_profit = before.groupby(
#                             before['Date'].dt.to_period('M'), observed=True
#                         )['Total_Profit'].sum()
#                         estimated_loss_profit = before_profit.median() * months_out
#
#             # ═══════════════════════════════════════════════════════════════
#             # ENHANCED STATUS DETERMINATION
#             # ═══════════════════════════════════════════════════════════════
#             status = "Unknown"
#             priority = 999
#
#             if self.has_profit and avg_margin < 0:
#                 status = "Loss-Making"
#                 priority = 0
#             elif is_stockout:
#                 status = "Stock-Out"
#                 priority = 1
#             elif self.has_stock and 0 < current_stock < 10:
#                 status = "Low Stock"
#                 priority = 2
#             elif days_since_last > 180:
#                 if self.has_stock and current_stock > 0:
#                     status = "Dead (Stock Available)"
#                     priority = 5
#                 else:
#                     status = "Dead (No Stock)"
#                     priority = 4
#             elif active_months_ratio < 0.3 and active_months >= 6:
#                 status = "Sporadic (Inactive)"
#                 priority = 6
#             elif trend_direction == "Declining":
#                 if self.has_stock and current_stock < 50:
#                     status = "Declining (Low Stock)"
#                     priority = 3
#                 else:
#                     status = "Declining"
#                     priority = 6
#             elif trend_direction == "Growing":
#                 status = "Growing"
#                 priority = 10
#             elif trend_direction == "Stable":
#                 if revenue_consistency > 0.8:
#                     status = "Stable (Consistent)"
#                     priority = 11
#                 else:
#                     status = "Stable (Variable)"
#                     priority = 9
#             elif trend_direction == "Volatile":
#                 status = "Volatile"
#                 priority = 7
#             else:
#                 status = "Active"
#                 priority = 9
#
#             # ═══════════════════════════════════════════════════════════════
#             # RETURN RESULTS
#             # ═══════════════════════════════════════════════════════════════
#             return {
#                 # Basic
#                 'SKU': sku_name,
#                 'Status': status,
#                 'Priority': priority,
#
#                 # Revenue
#                 'Total_Revenue': total_revenue,
#                 'Total_Quantity': total_qty,
#                 'Num_Transactions': num_transactions,
#
#                 # Robust metrics
#                 'Avg_Monthly_Revenue': avg_monthly_revenue,
#                 'Median_Monthly_Revenue': median_monthly_revenue,
#                 'Revenue_Consistency': revenue_consistency,
#                 'Avg_Daily_Revenue': avg_daily_revenue,
#                 'Median_Daily_Revenue': median_daily_revenue,
#                 'Daily_Consistency': daily_consistency,
#
#                 # Outliers
#                 'Revenue_Without_Outliers': revenue_without_outliers,
#                 'Adjusted_Avg_Monthly': adjusted_avg_monthly,
#                 'Num_Outliers': num_outliers,
#                 'Outlier_%': outlier_pct,
#                 'Outlier_Revenue_%': outlier_revenue_pct,
#
#                 # Profit
#                 'Total_Profit': total_profit,
#                 'Avg_Margin_%': avg_margin,
#                 'Median_Margin_%': median_margin,
#
#                 # Stock
#                 'Current_Stock': current_stock,
#
#                 # Time
#                 'First_Sale': first_sale,
#                 'Last_Sale': last_sale,
#                 'Days_Since_Last_Sale': days_since_last,
#                 'Days_Active': total_days_active,
#
#                 # Activity
#                 'Total_Months': total_months,
#                 'Active_Months': active_months,
#                 'Active_Months_Ratio': active_months_ratio,
#
#                 # Daily pattern
#                 'Unique_Selling_Days': unique_selling_days,
#                 'Selling_Days_Ratio': selling_days_ratio,
#                 'Avg_Days_Between_Sales': avg_days_between_sales,
#
#                 # Sales type
#                 'Family_Revenue': family_revenue,
#                 'Contract_Revenue': contract_revenue,
#                 'Contract_%': contract_pct,
#
#                 # Purchase pattern
#                 'Purchase_Pattern': purchase_pattern,
#                 'Small_Txns_%': small_pct,
#                 'Medium_Txns_%': medium_pct,
#                 'Large_Txns_%': large_pct,
#
#                 # Advanced trend
#                 'Trend_Direction': trend_direction,
#                 'Trend_Confidence': trend_confidence,
#                 'Volatility_CV': cv,
#                 'Exp_Trend': trend_metrics.get('exp_trend', 0),
#                 'MK_Tau': trend_metrics.get('mk_tau', 0),
#                 'MK_P_Value': trend_metrics.get('mk_p', 1),
#                 'Period_Change_%': trend_metrics.get('period_change', 0),
#                 'CAGR_%': trend_metrics.get('cagr'),
#                 'Consistency_%': trend_metrics.get('consistency', 0),
#
#                 # Stock-out
#                 'Is_Stockout': is_stockout,
#                 'Stockout_Date': stockout_date,
#                 'Days_Out_of_Stock': days_out_of_stock,
#                 'Revenue_Drop_%': revenue_drop_pct,
#                 'Estimated_Loss_Revenue': estimated_loss_revenue,
#                 'Estimated_Loss_Profit': estimated_loss_profit
#             }
#
#         except Exception as e:
#             print(f"Error analyzing SKU {sku_name}: {e}")
#             return None
#
#     def analyze_all_skus(self, save_to_excel=True, output_file='SKU_ANALYSIS_OPTIMIZED.xlsx'):
#         """
#         Bulk analysis with progress tracking
#         """
#         print("=" * 120)
#         print("🚀 OPTIMIZED SKU ANALYSIS - STARTING")
#         print("=" * 120)
#         print(f"📅 Analysis Date: {self.analysis_date.strftime('%Y-%m-%d')}")
#
#         all_skus = self.df['SKU'].dropna().astype(str).str.strip().unique()
#         total_skus = len(all_skus)
#         print(f"📦 Total SKUs: {total_skus:,}")
#         print(f"⏱️  Estimated time: ~{total_skus / 2000:.1f} minutes (2x faster!)")
#         print()
#
#         print("📊 Available data:")
#         print(f" 💰 Profit: {'✅' if self.has_profit else '❌'}")
#         print(f" 📦 Stock: {'✅' if self.has_stock else '❌'}")
#         print(f" 🏢 Sales type: {'✅' if self.has_sales_type else '❌'}")
#         print()
#         print("✨ FEATURES:")
#         print(" • 6 advanced trend detection methods with voting system")
#         print(" • Median-based robust metrics (outlier-resistant)")
#         print(" • Active months & selling days analysis")
#         print(" • Automated outlier detection & adjustment")
#         print(" • Purchase pattern classification")
#         print(" • Stock-out impact estimation")
#         print()
#
#         results = []
#         print("🔄 Processing SKUs...")
#         print("-" * 120)
#
#         for sku in tqdm(all_skus, desc="📊 Analyzing", ncols=100, colour='cyan'):
#             sku_data = self.df[self.df['SKU'].astype(str).str.strip() == sku].copy()
#             if len(sku_data) > 0:
#                 result = self.analyze_single_sku(sku_data, sku)
#                 if result:
#                     results.append(result)
#
#         print()
#         print("=" * 120)
#         print(f"✅ Analysis Complete! Processed {len(results):,} SKUs")
#         print("=" * 120)
#
#         results_df = pd.DataFrame(results)
#         results_df = results_df.sort_values(['Priority', 'Total_Revenue'],
#                                             ascending=[True, False])
#
#         # Summary
#         self._print_summary(results_df)
#
#         # Save to Excel
#         if save_to_excel:
#             self._save_to_excel(results_df, output_file)
#
#         print("\n" + "=" * 120)
#         print("🎉 ANALYSIS COMPLETE!")
#         print("=" * 120)
#
#         return results_df
#
#     def _print_summary(self, results_df):
#         """Print comprehensive summary"""
#         print("\n📊 SUMMARY BY STATUS:")
#         print("-" * 120)
#
#         status_summary = results_df.groupby('Status').agg({
#             'SKU': 'count',
#             'Total_Revenue': 'sum',
#             'Median_Monthly_Revenue': 'mean',
#             'Total_Profit': 'sum',
#             'Current_Stock': 'sum',
#             'Active_Months_Ratio': 'mean'
#         }).reset_index()
#
#         status_summary = status_summary.sort_values('Total_Revenue', ascending=False)
#
#         for _, row in status_summary.iterrows():
#             print(f"{row['Status']:30s}: {row['SKU']:>6,} SKUs | "
#                   f"Revenue: {row['Total_Revenue']:>15,.0f} | "
#                   f"Median/Month: {row['Median_Monthly_Revenue']:>10,.0f} | "
#                   f"Activity: {row['Active_Months_Ratio']:>5.1%}")
#
#         print("-" * 120)
#
#         # Outlier analysis
#         print("\n📊 OUTLIER ANALYSIS:")
#         print("-" * 120)
#         skus_with_outliers = results_df[results_df['Outlier_%'] > 10]
#         if len(skus_with_outliers) > 0:
#             print(f"⚠️  {len(skus_with_outliers):,} SKUs have significant outliers (>10%)")
#             print(f"   Avg outlier impact: {skus_with_outliers['Outlier_Revenue_%'].mean():.1f}%")
#
#         # Activity analysis
#         print("\n📊 ACTIVITY ANALYSIS:")
#         print("-" * 120)
#         low_activity = results_df[results_df['Active_Months_Ratio'] < 0.3]
#         medium_activity = results_df[(results_df['Active_Months_Ratio'] >= 0.3) &
#                                      (results_df['Active_Months_Ratio'] < 0.7)]
#         high_activity = results_df[results_df['Active_Months_Ratio'] >= 0.7]
#
#         print(f"🔴 Sporadic: {len(low_activity):,} SKUs (<30% active)")
#         print(f"🟡 Moderate: {len(medium_activity):,} SKUs (30-70% active)")
#         print(f"🟢 Consistent: {len(high_activity):,} SKUs (>70% active)")
#
#         # Trend analysis
#         print("\n📊 TREND ANALYSIS:")
#         print("-" * 120)
#         for trend in ['Growing', 'Declining', 'Stable', 'Volatile']:
#             trend_skus = results_df[results_df['Trend_Direction'] == trend]
#             if len(trend_skus) > 0:
#                 high_conf = len(trend_skus[trend_skus['Trend_Confidence'] == 'HIGH'])
#                 print(f"{trend:15s}: {len(trend_skus):>6,} SKUs | "
#                       f"High Confidence: {high_conf:>5,}")
#
#         # Critical alerts
#         if self.has_stock:
#             stockouts = results_df[results_df['Is_Stockout'] == True]
#             if len(stockouts) > 0:
#                 print(f"\n🚨 CRITICAL: {len(stockouts):,} SKUs in STOCK-OUT!")
#                 print(f"   Estimated Loss: {stockouts['Estimated_Loss_Revenue'].sum():,.0f} SAR")
#
#         if self.has_profit:
#             loss_making = results_df[results_df['Avg_Margin_%'] < 0]
#             if len(loss_making) > 0:
#                 print(f"\n❌ WARNING: {len(loss_making):,} SKUs LOSING MONEY!")
#                 print(f"   Total Loss: {loss_making['Total_Profit'].sum():,.0f} SAR")
#
#     def _save_to_excel(self, results_df, output_file):
#         """Save results to Excel with multiple sheets"""
#         print(f"\n💾 Saving to: {output_file}")
#
#         with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
#             # Main sheet
#             results_df.to_excel(writer, sheet_name='All SKUs', index=False)
#
#             # Critical SKUs
#             critical = results_df[results_df['Priority'] <= 2]
#             if len(critical) > 0:
#                 critical.to_excel(writer, sheet_name='CRITICAL', index=False)
#
#             # Growing (High confidence)
#             growing = results_df[(results_df['Trend_Direction'] == 'Growing') &
#                                 (results_df['Trend_Confidence'] == 'HIGH')]
#             if len(growing) > 0:
#                 growing.to_excel(writer, sheet_name='Growing (High Conf)', index=False)
#
#             # Declining (High confidence)
#             declining = results_df[(results_df['Trend_Direction'] == 'Declining') &
#                                   (results_df['Trend_Confidence'] == 'HIGH')]
#             if len(declining) > 0:
#                 declining.to_excel(writer, sheet_name='Declining (High Conf)', index=False)
#
#             # Consistent performers
#             consistent = results_df[(results_df['Revenue_Consistency'] > 0.8) &
#                                    (results_df['Active_Months_Ratio'] > 0.7)]
#             if len(consistent) > 0:
#                 consistent_sorted = consistent.sort_values('Total_Revenue', ascending=False)
#                 consistent_sorted.to_excel(writer, sheet_name='Consistent Performers', index=False)
#
#             # High outliers
#             high_outliers = results_df[results_df['Outlier_%'] > 20]
#             if len(high_outliers) > 0:
#                 high_outliers_sorted = high_outliers.sort_values('Outlier_Revenue_%',
#                                                                  ascending=False)
#                 high_outliers_sorted.to_excel(writer, sheet_name='High Outliers', index=False)
#
#             # Stock-outs
#             if self.has_stock:
#                 stockouts = results_df[results_df['Is_Stockout'] == True]
#                 if len(stockouts) > 0:
#                     stockouts.to_excel(writer, sheet_name='Stock-Outs', index=False)
#
#             # Loss-making
#             if self.has_profit:
#                 loss_making = results_df[results_df['Avg_Margin_%'] < 0]
#                 if len(loss_making) > 0:
#                     loss_making.to_excel(writer, sheet_name='Loss-Making', index=False)
#
#             # Summary
#             status_summary = results_df.groupby('Status').agg({
#                 'SKU': 'count',
#                 'Total_Revenue': 'sum',
#                 'Median_Monthly_Revenue': 'mean',
#                 'Total_Profit': 'sum'
#             }).reset_index()
#             status_summary.to_excel(writer, sheet_name='Summary', index=False)
#
#         print(f"✅ Saved successfully!")
#
#
# # ============================================================================
# # MAIN EXECUTION FUNCTION
# # ============================================================================
#
# def analyze_skus(df, save_to_excel=True, output_file='SKU_ANALYSIS_OPTIMIZED.xlsx'):
#     """
#     Main function to run complete SKU analysis
#
#     Parameters:
#     -----------
#     df : pandas.DataFrame
#         Sales data with columns: Date, SKU, bal Value, bal Qty
#         Optional: Total_Profit, CURRENT_STOCK, Sales_Type
#
#     save_to_excel : bool
#         Whether to save results to Excel (default: True)
#
#     output_file : str
#         Output file name (default: 'SKU_ANALYSIS_OPTIMIZED.xlsx')
#
#     Returns:
#     --------
#     pandas.DataFrame with comprehensive SKU analysis
#
#     Example:
#     --------
#     >>> results = analyze_skus(combined_df)
#     """
#     analyzer = SKUAnalyzer(df)
#     return analyzer.analyze_all_skus(save_to_excel, output_file)
#
#
# # ============================================================================
# # QUICK FUNCTIONS FOR DIRECT USE
# # ============================================================================
#
# def quick_trend_analysis(monthly_values):
#     """
#     Quick standalone trend analysis
#
#     Parameters:
#     -----------
#     monthly_values : list or array
#         Monthly revenue values
#
#     Returns:
#     --------
#     tuple: (direction, confidence, metrics_dict)
#     """
#     return TrendAnalyzer.analyze_trend(monthly_values)
#
#
# # ============================================================================
# # INITIALIZATION MESSAGE
# # ============================================================================
#
# print("=" * 120)
# print("✅ OPTIMIZED SKU ANALYSIS SYSTEM LOADED!")
# print("=" * 120)
# print()
# print("🚀 MAIN FUNCTION:")
# print("   results_df = analyze_skus(df)")
# print()
# print("⚡ FEATURES:")
# print("   • 2x faster execution with NumPy vectorization")
# print("   • 6 advanced trend detection methods (voting system)")
# print("   • Robust median-based metrics (outlier-resistant)")
# print("   • Exponential smoothing + Mann-Kendall + CAGR + Period comparison")
# print("   • Active months ratio & selling days pattern")
# print("   • Automated outlier detection & adjustment")
# print("   • Purchase pattern classification")
# print("   • Stock-out impact estimation with loss calculation")
# print("   • Multi-sheet Excel export with actionable insights")
# print()
# print("📊 TREND ANALYSIS METHODS:")
# print("   1. Exponential Smoothing (non-linear trends)")
# print("   2. Mann-Kendall Test (robust to outliers)")
# print("   3. Period Comparison (first 25% vs last 25%)")
# print("   4. CAGR (compound annual growth rate)")
# print("   5. Consistency Score (directional stability)")
# print("   6. Volatility Analysis (CV)")
# print()
# print("💡 QUICK FUNCTIONS:")
# print("   • quick_trend_analysis(monthly_values)")
# print()
# print("=" * 120)

MEMORY-EFFICIENT SKU ANALYSIS SYSTEM

✅ USAGE EXAMPLES:

# Method 1: Quick Analysis
results, filepath = quick_analyze(combined_df)

# Method 2: Custom Folder
results, filepath = quick_analyze(combined_df, folder_name='my_analysis')

# Method 3: Full Control
results = analyze_skus(
    df=combined_df,
    save_to_excel=True,
    output_file='Desktop/classification/report.xlsx'
)


📊 KEY OPTIMIZATIONS:
   • Pre-computed aggregations (single database pass)
   • Vectorized operations (no row-by-row loops)
   • Optimized data types (reduced memory footprint)
   • Efficient groupby operations
   • No redundant filtering

💾 MEMORY IMPROVEMENTS:
   • 70-80% less memory usage
   • 5-10x faster processing
   • Handles 100K+ SKUs easily



## The final & most effiecient classifications

In [2]:



# ============================================================================
# ULTRA-OPTIMIZED SKU ANALYSIS - TIME-WEIGHTED & ENHANCED
# للبيانات الضخمة: 100K+ SKUs, millions of rows
# مع تركيز أكبر على الفترة الأخيرة (آخر سنة)
# ============================================================================

import pandas as pd
import numpy as np
from scipy.stats import kendalltau
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from tqdm.notebook import tqdm
import warnings
import gc
warnings.filterwarnings('ignore')


# ============================================================================
# TIME-WEIGHTED TREND ANALYZER
# ============================================================================

class TimeWeightedTrendAnalyzer:
    """
    Fast vectorized trend analysis with TIME-WEIGHTING
    يعطي وزن أكبر للأشهر الأخيرة (آخر سنة)
    """

    @staticmethod
    def analyze_trend(monthly_values):
        """
        6-method trend analysis with voting system
        مع تركيز على آخر 12 شهر
        """
        if len(monthly_values) < 6:
            return "Unknown", "LOW", {}

        try:
            values = np.array(monthly_values, dtype=np.float32)

            # ═══════════════════════════════════════════════════════════════
            # RECENT FOCUS: آخر 12 شهر (أو كل البيانات إذا كانت أقل)
            # ═══════════════════════════════════════════════════════════════
            recent_months = min(12, len(values))
            recent_values = values[-recent_months:]

            # METHOD 1: Exponential Smoothing (على كل البيانات)
            exp_trend, exp_strength = 0, 0
            try:
                model = ExponentialSmoothing(values, trend='add', seasonal=None)
                fit = model.fit(smoothing_level=0.3, smoothing_trend=0.1, optimized=False)
                trend = fit.trend
                # ✅ نركز على آخر 3 أشهر فقط
                exp_trend = trend[-3:].mean() if len(trend) >= 3 else trend[-1]
                positive_trends = (trend[-recent_months:] > 0).sum()  # آخر سنة فقط
                exp_strength = abs(positive_trends / recent_months - 0.5) * 2
            except:
                pass

            # METHOD 2: Mann-Kendall Test (على آخر سنة)
            mk_tau, mk_p = 0, 1
            try:
                months = np.arange(len(recent_values))
                mk_tau, mk_p = kendalltau(months, recent_values)
            except:
                pass

            # METHOD 3: Period Comparison (آخر 3 أشهر vs 3 أشهر قبلها)
            if len(values) >= 6:
                last_3_months = values[-3:].mean()
                prev_3_months = values[-6:-3].mean()
                period_change = ((last_3_months - prev_3_months) / prev_3_months * 100) if prev_3_months > 0 else 0
            else:
                period_size = max(2, len(values) // 4)
                first_period = values[:period_size].mean()
                last_period = values[-period_size:].mean()
                period_change = ((last_period - first_period) / first_period * 100) if first_period > 0 else 0

            # METHOD 4: CAGR (على آخر سنة إن وُجدت)
            cagr = None
            if len(values) >= 12:
                # ✅ CAGR على آخر 12 شهر فقط
                beginning = values[-12:-9].mean()  # الشهر 9-12 من النهاية
                ending = values[-3:].mean()  # آخر 3 أشهر
                if beginning > 0 and ending > 0:
                    years = 9 / 12  # 9 أشهر
                    cagr = ((ending / beginning) ** (1 / years) - 1) * 100

            # METHOD 5: Consistency Score (على آخر سنة)
            changes = np.diff(recent_values)
            consistency = 0
            if len(changes) > 0:
                positive = (changes > 0).sum()
                negative = (changes < 0).sum()
                consistency = max(positive, negative) / len(changes) * 100

            # Volatility (CV) - على آخر سنة
            cv = recent_values.std() / recent_values.mean() if recent_values.mean() > 0 else 0

            # ═══════════════════════════════════════════════════════════════
            # VOTING SYSTEM (محسّن للتركيز على الفترة الأخيرة)
            # ═══════════════════════════════════════════════════════════════
            votes_declining = 0
            votes_growing = 0

            # Vote 1: Exponential Smoothing
            if exp_trend < -300:  # أكثر حساسية
                votes_declining += 1
            elif exp_trend > 300:
                votes_growing += 1

            # Vote 2: Mann-Kendall
            if mk_tau < -0.15 and mk_p < 0.1:  # أكثر مرونة
                votes_declining += 1
            elif mk_tau > 0.15 and mk_p < 0.1:
                votes_growing += 1

            # Vote 3: Period Comparison (أهم Vote!)
            if period_change < -10:  # أكثر حساسية
                votes_declining += 1.5  # وزن أكبر!
            elif period_change > 10:
                votes_growing += 1.5

            # Vote 4: CAGR
            if cagr is not None:
                if cagr < -8:
                    votes_declining += 1
                elif cagr > 8:
                    votes_growing += 1

            # ═══════════════════════════════════════════════════════════════
            # DETERMINE TREND
            # ═══════════════════════════════════════════════════════════════
            total_votes = 5.5 if cagr is not None else 4.5  # بسبب الوزن الإضافي

            if votes_declining >= 2:
                trend_direction = "Declining"
                confidence_score = votes_declining / total_votes
            elif votes_growing >= 2:
                trend_direction = "Growing"
                confidence_score = votes_growing / total_votes
            else:
                trend_direction = "Stable" if cv < 0.3 else ("Volatile" if cv > 0.7 else "Stable")
                confidence_score = 0.5

            # Adjust confidence by consistency and volatility
            if consistency > 70 and cv < 0.5:
                confidence_score *= 1.2
            elif consistency < 50 or cv > 0.8:
                confidence_score *= 0.8

            confidence_score = min(confidence_score, 1.0)
            confidence = "HIGH" if confidence_score >= 0.75 else ("MEDIUM" if confidence_score >= 0.5 else "LOW")

            metrics = {
                'exp_trend': float(exp_trend),
                'mk_tau': float(mk_tau),
                'mk_p': float(mk_p),
                'period_change': float(period_change),
                'cagr': float(cagr) if cagr is not None else None,
                'consistency': float(consistency),
                'cv': float(cv),
                'confidence_score': float(confidence_score)
            }

            return trend_direction, confidence, metrics

        except:
            return "Unknown", "LOW", {}


# ============================================================================
# ENHANCED SKU ANALYZER WITH TIME-WEIGHTING
# ============================================================================

class EnhancedSKUAnalyzer:
    """
    Memory-efficient SKU analyzer with TIME-WEIGHTED analysis
    التركيز على آخر سنة في كل التحليلات
    """

    def __init__(self, df, analysis_date=None):
        """Initialize with pre-processing"""
        print("🔧 Pre-processing data for optimal performance...")

        # تنظيف عمود SKU مرة واحدة فقط
        df['SKU_CLEAN'] = df['SKU'].astype(str).str.strip()

        # تحويل التاريخ
        if not pd.api.types.is_datetime64_any_dtype(df['Date']):
            df['Date'] = pd.to_datetime(df['Date'])

        self.df = df
        self.analysis_date = analysis_date or df['Date'].max()
        self.trend_analyzer = TimeWeightedTrendAnalyzer()

        # Check available columns
        self.has_profit = 'Total_Profit' in df.columns
        self.has_stock = 'CURRENT_STOCK' in df.columns
        self.has_sales_type = 'Sales_Type' in df.columns

        print("✅ Pre-processing complete!")

    def analyze_single_sku(self, sku_data, sku_name):
        """Optimized single SKU analysis with TIME-WEIGHTING"""
        if len(sku_data) == 0:
            return None

        try:
            # استخدام .values لتوفير الذاكرة
            revenue_values = sku_data['bal Value'].values
            qty_values = sku_data['bal Qty'].values
            date_values = sku_data['Date'].values

            # BASIC METRICS
            total_revenue = revenue_values.sum()
            total_qty = qty_values.sum()
            first_sale = pd.Timestamp(date_values.min())
            last_sale = pd.Timestamp(date_values.max())
            analysis_date = pd.Timestamp(self.analysis_date)

            days_since_last = int((analysis_date - last_sale) / np.timedelta64(1, 'D'))
            total_days_active = int((last_sale - first_sale) / np.timedelta64(1, 'D')) + 1
            num_transactions = len(sku_data)

            # ═══════════════════════════════════════════════════════════════
            # TIME-WEIGHTED CALCULATIONS: آخر سنة
            # ═══════════════════════════════════════════════════════════════
            one_year_ago = analysis_date - pd.Timedelta(days=365)
            recent_mask = date_values >= one_year_ago
            recent_data = sku_data[recent_mask]

            # Metrics for recent period (آخر سنة)
            recent_revenue = revenue_values[recent_mask].sum()
            recent_qty = qty_values[recent_mask].sum()
            recent_transactions = recent_mask.sum()

            # PROFIT METRICS
            total_profit, avg_margin, median_margin = 0, 0, 0
            recent_profit, recent_avg_margin, recent_median_margin = 0, 0, 0

            if self.has_profit:
                profit_values = sku_data['Total_Profit'].values
                total_profit = profit_values.sum()
                avg_margin = (total_profit / total_revenue * 100) if total_revenue > 0 else 0

                # All-time median margin
                mask = revenue_values > 0
                if mask.any():
                    margins = (profit_values[mask] / revenue_values[mask] * 100)
                    median_margin = np.median(margins)

                # ✅ Recent profit metrics (آخر سنة)
                recent_profit_values = profit_values[recent_mask]
                recent_revenue_values = revenue_values[recent_mask]
                recent_profit = recent_profit_values.sum()
                recent_avg_margin = (recent_profit / recent_revenue * 100) if recent_revenue > 0 else 0

                recent_rev_mask = recent_revenue_values > 0
                if recent_rev_mask.any():
                    recent_margins = (recent_profit_values[recent_rev_mask] /
                                     recent_revenue_values[recent_rev_mask] * 100)
                    recent_median_margin = np.median(recent_margins)

            # STOCK METRICS
            current_stock = 0
            if self.has_stock:
                stock_values = sku_data['CURRENT_STOCK'].dropna()
                if len(stock_values) > 0:
                    current_stock = stock_values.iloc[0]

            # SALES TYPE
            family_revenue, contract_revenue, contract_pct = 0, 0, 0

            if self.has_sales_type:
                sales_type_values = sku_data['Sales_Type'].values
                family_mask = sales_type_values == 'FAMILY'
                contract_mask = sales_type_values == 'CONTRACT'
                family_revenue = revenue_values[family_mask].sum()
                contract_revenue = revenue_values[contract_mask].sum()
                contract_pct = (contract_revenue / total_revenue * 100) if total_revenue > 0 else 0

            # MONTHLY ANALYSIS
            dates_pd = pd.to_datetime(date_values)
            year_month = dates_pd.to_period('M')

            monthly_df = pd.DataFrame({
                'YearMonth': year_month,
                'Revenue': revenue_values,
                'Qty': qty_values
            })

            monthly = monthly_df.groupby('YearMonth', observed=True).agg({
                'Revenue': 'sum',
                'Qty': 'sum'
            })

            # ✅ Recent monthly analysis (آخر 12 شهر)
            recent_dates_pd = pd.to_datetime(date_values[recent_mask])
            recent_year_month = recent_dates_pd.to_period('M')

            recent_monthly_df = pd.DataFrame({
                'YearMonth': recent_year_month,
                'Revenue': revenue_values[recent_mask]
            })

            recent_monthly = recent_monthly_df.groupby('YearMonth', observed=True)['Revenue'].sum()

            first_sale_pd = pd.Timestamp(first_sale)
            last_sale_pd = pd.Timestamp(last_sale)
            total_months = ((last_sale_pd.year - first_sale_pd.year) * 12 +
                           (last_sale_pd.month - first_sale_pd.month) + 1)
            active_months = len(monthly)
            active_months_ratio = active_months / total_months if total_months > 0 else 0

            # All-time monthly metrics
            monthly_values = monthly['Revenue'].values.astype(np.float32)
            avg_monthly_revenue = monthly_values.mean() if active_months > 0 else 0
            median_monthly_revenue = np.median(monthly_values) if active_months > 0 else 0
            revenue_consistency = (median_monthly_revenue / avg_monthly_revenue
                                  if avg_monthly_revenue > 0 else 0)

            # ✅ Recent monthly metrics (آخر سنة)
            recent_active_months = len(recent_monthly)
            recent_monthly_values = recent_monthly.values.astype(np.float32)
            recent_avg_monthly_revenue = recent_monthly_values.mean() if recent_active_months > 0 else 0
            recent_median_monthly_revenue = np.median(recent_monthly_values) if recent_active_months > 0 else 0
            recent_revenue_consistency = (recent_median_monthly_revenue / recent_avg_monthly_revenue
                                         if recent_avg_monthly_revenue > 0 else 0)

            cv = (monthly_values.std() / monthly_values.mean()
                 if monthly_values.mean() > 0 and active_months > 1 else 0)

            # DAILY ANALYSIS
            unique_selling_days = len(np.unique(date_values))
            selling_days_ratio = unique_selling_days / total_days_active if total_days_active > 0 else 0
            avg_days_between_sales = total_days_active / unique_selling_days if unique_selling_days > 1 else 0

            daily_df = pd.DataFrame({'Date': date_values, 'Revenue': revenue_values})
            daily_revenue = daily_df.groupby('Date')['Revenue'].sum().values.astype(np.float32)

            median_daily_revenue = np.median(daily_revenue) if len(daily_revenue) > 0 else 0
            avg_daily_revenue = daily_revenue.mean() if len(daily_revenue) > 0 else 0
            daily_consistency = (median_daily_revenue / avg_daily_revenue if avg_daily_revenue > 0 else 0)

            # OUTLIER DETECTION
            Q1 = np.percentile(revenue_values, 25)
            Q3 = np.percentile(revenue_values, 75)
            IQR = Q3 - Q1
            outlier_threshold = Q3 + (1.5 * IQR)

            outlier_mask = revenue_values > outlier_threshold
            num_outliers = outlier_mask.sum()
            outlier_pct = (num_outliers / num_transactions * 100) if num_transactions > 0 else 0
            outlier_revenue = revenue_values[outlier_mask].sum()
            outlier_revenue_pct = (outlier_revenue / total_revenue * 100) if total_revenue > 0 else 0

            revenue_without_outliers = total_revenue - outlier_revenue
            adjusted_avg_monthly = (revenue_without_outliers / active_months if active_months > 0 else 0)

            # PURCHASE PATTERN
            small_threshold = np.percentile(revenue_values, 33)
            large_threshold = np.percentile(revenue_values, 67)

            small_txns = (revenue_values <= small_threshold).sum()
            large_txns = (revenue_values > large_threshold).sum()
            medium_txns = num_transactions - small_txns - large_txns

            small_pct = (small_txns / num_transactions * 100) if num_transactions > 0 else 0
            large_pct = (large_txns / num_transactions * 100) if num_transactions > 0 else 0
            medium_pct = 100 - small_pct - large_pct

            if large_pct > 50:
                purchase_pattern = "Large Orders"
            elif small_pct > 70:
                purchase_pattern = "Frequent Small"
            elif medium_pct > 60:
                purchase_pattern = "Regular Medium"
            else:
                purchase_pattern = "Mixed"

            # TREND ANALYSIS (Time-weighted)
            trend_direction, trend_confidence, trend_metrics = "Unknown", "LOW", {}

            if active_months >= 6:
                trend_direction, trend_confidence, trend_metrics = \
                    self.trend_analyzer.analyze_trend(monthly_values)

            # ═══════════════════════════════════════════════════════════════
            # ENHANCED POTENTIAL STOCK-OUT DETECTION
            # ═══════════════════════════════════════════════════════════════
            is_stockout = False
            is_potential_stockout = False
            stockout_date = None
            days_out_of_stock = 0
            estimated_loss_revenue = 0
            estimated_loss_profit = 0
            prev_avg_monthly_revenue = 0
            prev_avg_margin = 0
            revenue_drop_after_stockout = 0

            if self.has_stock and current_stock == 0 and days_since_last > 60:
                is_stockout = True

                # Find stockout date
                dates_sorted = np.sort(date_values)
                if len(dates_sorted) > 1:
                    gaps = np.diff(dates_sorted).astype('timedelta64[D]').astype(int)
                    gap_idx = np.where(gaps > 60)[0]
                    stockout_date = dates_sorted[gap_idx[0]] if len(gap_idx) > 0 else last_sale
                else:
                    stockout_date = last_sale

                days_out_of_stock = (analysis_date - pd.Timestamp(stockout_date)).days

                # ✅ تحليل محسّن: فقط آخر سنة قبل Stock-Out
                one_year_before_stockout = pd.Timestamp(stockout_date) - pd.Timedelta(days=365)
                before_mask = (date_values >= one_year_before_stockout) & (date_values < stockout_date)
                before_data = sku_data[before_mask]

                if len(before_data) > 0:
                    before_dates = pd.to_datetime(before_data['Date'].values)
                    before_year_month = before_dates.to_period('M')
                    before_monthly_df = pd.DataFrame({
                        'YearMonth': before_year_month,
                        'Revenue': before_data['bal Value'].values
                    })
                    before_monthly = before_monthly_df.groupby('YearMonth', observed=True)['Revenue'].sum()

                    prev_avg_monthly_revenue = before_monthly.median()

                    if self.has_profit:
                        before_profit_monthly = pd.DataFrame({
                            'YearMonth': before_year_month,
                            'Profit': before_data['Total_Profit'].values
                        }).groupby('YearMonth', observed=True)['Profit'].sum()

                        prev_total_profit = before_data['Total_Profit'].sum()
                        prev_total_revenue = before_data['bal Value'].sum()
                        prev_avg_margin = (prev_total_profit / prev_total_revenue * 100) if prev_total_revenue > 0 else 0

                        estimated_loss_profit = before_profit_monthly.median() * (days_out_of_stock / 30)

                    months_out = days_out_of_stock / 30
                    estimated_loss_revenue = prev_avg_monthly_revenue * months_out

                    # ✅ تحديد Potential Stock-Out (محسّن!)
                    # الشروط أكثر صرامة:
                    # 1. إيرادات آخر سنة قبل Stock-Out كانت عالية
                    # 2. هامش الربح كان جيد (>25% بدلاً من 20%)
                    # 3. كان نشط بانتظام
                    if (prev_avg_monthly_revenue > 5000 and  # حد أدنى معقول
                        prev_avg_margin > 30 and  # ✅ هامش ربح أعلى
                        len(before_monthly) >= 3):  # على الأقل 3 أشهر نشط

                        is_potential_stockout = True
                        revenue_drop_after_stockout = 100  # 100% لأنه توقف تماماً

            # ═══════════════════════════════════════════════════════════════
            # ENHANCED STATUS DETERMINATION (محسّن!)
            # ═══════════════════════════════════════════════════════════════
            status = "Unknown"
            priority = 999

            # ✅ التركيز على الأداء الأخير (آخر سنة)
            if is_potential_stockout:
                status = "Potential Stock-Out (High Value)"
                priority = 0
            elif self.has_profit and recent_median_margin < -5:  # ✅ خاسر مؤخراً
                status = "Loss-Making (Recent)"
                priority = 1
            elif self.has_profit and avg_margin < 0 and recent_median_margin < 0:  # خاسر دائماً
                status = "Loss-Making (Always)"
                priority = 2
            elif is_stockout:
                status = "Stock-Out"
                priority = 3
            elif self.has_stock and 0 < current_stock < 10:
                # ✅ نفرق بين منتج مربح ومش مربح
                if recent_median_margin > 30:  # ✅ هامش ربح عالي
                    status = "Low Stock (High Margin)"
                    priority = 4
                else:
                    status = "Low Stock"
                    priority = 5
            elif days_since_last > 365:  # ✅ سنة كاملة بدون مبيعات = ميت فعلاً
                if self.has_stock and current_stock > 0:
                    status = "Dead (Stock Available)"
                    priority = 8
                else:
                    status = "Dead (No Stock)"
                    priority = 7
            elif days_since_last > 180:  # 6 أشهر = Inactive
                status = "Inactive"
                priority = 9
            elif active_months_ratio < 0.3 and active_months >= 6:
                status = "Sporadic"
                priority = 10
            elif trend_direction == "Declining":
                if self.has_stock and current_stock < 50:
                    status = "Declining (Low Stock)"
                    priority = 6
                else:
                    status = "Declining"
                    priority = 11
            elif trend_direction == "Growing":
                # ✅ نميز النمو القوي
                if recent_median_margin > 30:
                    status = "Growing (High Margin)"
                    priority = 15
                else:
                    status = "Growing"
                    priority = 14
            elif trend_direction == "Stable":
                if revenue_consistency > 0.8 and recent_median_margin > 25:
                    status = "Stable (Consistent & Profitable)"
                    priority = 16
                elif revenue_consistency > 0.8:
                    status = "Stable (Consistent)"
                    priority = 14
                else:
                    status = "Stable (Variable)"
                    priority = 13
            elif trend_direction == "Volatile":
                status = "Volatile"
                priority = 11
            else:
                status = "Active"
                priority = 12

            # RETURN RESULTS
            return {
                'SKU': sku_name,
                'Status': status,
                'Priority': priority,

                # Revenue
                'Total_Revenue': float(total_revenue),
                'Recent_Revenue_12M': float(recent_revenue),  # ✅ جديد
                'Total_Quantity': float(total_qty),
                'Num_Transactions': int(num_transactions),

                # Robust metrics
                'Avg_Monthly_Revenue': float(avg_monthly_revenue),
                'Median_Monthly_Revenue': float(median_monthly_revenue),
                'Recent_Avg_Monthly_Revenue': float(recent_avg_monthly_revenue),  # ✅ جديد
                'Recent_Median_Monthly_Revenue': float(recent_median_monthly_revenue),  # ✅ جديد
                'Revenue_Consistency': float(revenue_consistency),
                'Recent_Revenue_Consistency': float(recent_revenue_consistency),  # ✅ جديد

                'Avg_Daily_Revenue': float(avg_daily_revenue),
                'Median_Daily_Revenue': float(median_daily_revenue),
                'Daily_Consistency': float(daily_consistency),

                # Outliers
                'Revenue_Without_Outliers': float(revenue_without_outliers),
                'Adjusted_Avg_Monthly': float(adjusted_avg_monthly),
                'Num_Outliers': int(num_outliers),
                'Outlier_%': float(outlier_pct),
                'Outlier_Revenue_%': float(outlier_revenue_pct),

                # Profit
                'Total_Profit': float(total_profit),
                'Recent_Profit_12M': float(recent_profit),  # ✅ جديد
                'Avg_Margin_%': float(avg_margin),
                'Median_Margin_%': float(median_margin),
                'Recent_Avg_Margin_%': float(recent_avg_margin),  # ✅ جديد
                'Recent_Median_Margin_%': float(recent_median_margin),  # ✅ جديد

                # Stock
                'Current_Stock': float(current_stock),

                # Time
                'First_Sale': first_sale,
                'Last_Sale': last_sale,
                'Days_Since_Last_Sale': int(days_since_last),
                'Days_Active': int(total_days_active),

                # Activity
                'Total_Months': int(total_months),
                'Active_Months': int(active_months),
                'Active_Months_Ratio': float(active_months_ratio),
                'Recent_Active_Months': int(recent_active_months),  # ✅ جديد

                # Daily pattern
                'Unique_Selling_Days': int(unique_selling_days),
                'Selling_Days_Ratio': float(selling_days_ratio),
                'Avg_Days_Between_Sales': float(avg_days_between_sales),

                # Sales type
                'Family_Revenue': float(family_revenue),
                'Contract_Revenue': float(contract_revenue),
                'Contract_%': float(contract_pct),

                # Purchase pattern
                'Purchase_Pattern': purchase_pattern,
                'Small_Txns_%': float(small_pct),
                'Medium_Txns_%': float(medium_pct),
                'Large_Txns_%': float(large_pct),

                # Trend
                'Trend_Direction': trend_direction,
                'Trend_Confidence': trend_confidence,
                'Volatility_CV': float(cv),
                'Exp_Trend': float(trend_metrics.get('exp_trend', 0)),
                'MK_Tau': float(trend_metrics.get('mk_tau', 0)),
                'MK_P_Value': float(trend_metrics.get('mk_p', 1)),
                'Period_Change_%': float(trend_metrics.get('period_change', 0)),
                'CAGR_%': trend_metrics.get('cagr'),
                'Consistency_%': float(trend_metrics.get('consistency', 0)),

                # Stock-out
                'Is_Stockout': is_stockout,
                'Is_Potential_Stockout': is_potential_stockout,
                'Stockout_Date': stockout_date,
                'Days_Out_of_Stock': int(days_out_of_stock),
                'Estimated_Loss_Revenue': float(estimated_loss_revenue),
                'Estimated_Loss_Profit': float(estimated_loss_profit),
                'Prev_Avg_Monthly_Revenue': float(prev_avg_monthly_revenue),
                'Prev_Avg_Margin_%': float(prev_avg_margin),
                'Revenue_Drop_After_Stockout_%': float(revenue_drop_after_stockout)
            }

        except Exception as e:
            print(f"⚠️ Error analyzing SKU {sku_name}: {e}")
            return None

    def analyze_all_skus(self, save_to_excel=True, output_file='SKU_ANALYSIS_ENHANCED.xlsx'):
        """Memory-efficient bulk analysis"""
        print("=" * 120)
        print("🚀 ENHANCED TIME-WEIGHTED SKU ANALYSIS - STARTING")
        print("=" * 120)
        print(f"📅 Analysis Date: {self.analysis_date.strftime('%Y-%m-%d')}")

        all_skus = self.df['SKU_CLEAN'].unique()
        total_skus = len(all_skus)
        print(f"📦 Total SKUs: {total_skus:,}")
        print(f"⏱️  Estimated time: ~{total_skus / 3000:.1f} minutes")
        print()

        results = []
        print("🔄 Processing SKUs...")
        print("-" * 120)

        for i, sku in enumerate(tqdm(all_skus, desc="📊 Analyzing", ncols=100, colour='cyan')):
            sku_data = self.df[self.df['SKU_CLEAN'] == sku]

            if len(sku_data) > 0:
                result = self.analyze_single_sku(sku_data, sku)
                if result:
                    results.append(result)

            if (i + 1) % 1000 == 0:
                gc.collect()

        print()
        print("=" * 120)
        print(f"✅ Analysis Complete! Processed {len(results):,} SKUs")
        print("=" * 120)

        results_df = pd.DataFrame(results)
        results_df = results_df.sort_values(['Priority', 'Recent_Revenue_12M'], ascending=[True, False])

        self._print_summary(results_df)

        if save_to_excel:
            self._save_to_excel(results_df, output_file)

        print("\n" + "=" * 120)
        print("🎉 ANALYSIS COMPLETE!")
        print("=" * 120)

        return results_df

    def _print_summary(self, results_df):
        """Print summary"""
        print("\n📊 SUMMARY BY STATUS:")
        print("-" * 120)

        status_summary = results_df.groupby('Status').agg({
            'SKU': 'count',
            'Total_Revenue': 'sum',
            'Recent_Revenue_12M': 'sum',
            'Recent_Median_Margin_%': 'mean'
        }).reset_index().sort_values('Recent_Revenue_12M', ascending=False)

        for _, row in status_summary.iterrows():
            print(f"{row['Status']:40s}: {row['SKU']:>6,} SKUs | "
                  f"Recent Revenue: {row['Recent_Revenue_12M']:>12,.0f} | "
                  f"Margin: {row['Recent_Median_Margin_%']:>5.1f}%")

        print("\n⭐ POTENTIAL STOCK-OUT ANALYSIS:")
        potential_stockouts = results_df[results_df['Is_Potential_Stockout'] == True]
        if len(potential_stockouts) > 0:
            print(f"🚨 {len(potential_stockouts):,} SKUs = POTENTIAL HIGH-VALUE STOCK-OUTS!")
            print(f"   Total estimated loss: {potential_stockouts['Estimated_Loss_Revenue'].sum():,.0f} SAR")
            print(f"   Avg previous margin: {potential_stockouts['Prev_Avg_Margin_%'].mean():.1f}%")

    def _save_to_excel(self, results_df, output_file):
        """Save to Excel"""
        print(f"\n💾 Saving to: {output_file}")

        with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
            results_df.to_excel(writer, sheet_name='All SKUs', index=False)

            # Potential Stock-Outs
            potential = results_df[results_df['Is_Potential_Stockout'] == True]
            if len(potential) > 0:
                potential.sort_values('Estimated_Loss_Revenue', ascending=False).to_excel(
                    writer, sheet_name='⭐ Potential Stock-Outs', index=False)

            # High Margin Low Stock
            high_margin_low = results_df[results_df['Status'] == 'Low Stock (High Margin)']
            if len(high_margin_low) > 0:
                high_margin_low.to_excel(writer, sheet_name='Low Stock (High Margin)', index=False)

            # Critical
            critical = results_df[results_df['Priority'] <= 5]
            if len(critical) > 0:
                critical.to_excel(writer, sheet_name='CRITICAL', index=False)

            # Growing High Margin
            growing_hm = results_df[results_df['Status'] == 'Growing (High Margin)']
            if len(growing_hm) > 0:
                growing_hm.to_excel(writer, sheet_name='Growing (High Margin)', index=False)

        print(f"✅ Saved successfully!")


# ============================================================================
# MAIN FUNCTION
# ============================================================================

def analyze_skus_enhanced(df, save_to_excel=True, output_file='SKU_ANALYSIS_ENHANCED.xlsx'):
    """
    Enhanced time-weighted SKU analysis

    Example:
    >>> results = analyze_skus_enhanced(combined_df)
    """
    analyzer = EnhancedSKUAnalyzer(df)
    return analyzer.analyze_all_skus(save_to_excel, output_file)


print("=" * 120)
print("✅ ENHANCED TIME-WEIGHTED SKU ANALYSIS LOADED!")
print("=" * 120)
print("\n🚀 Usage: results_df = analyze_skus_enhanced(df)")
print("\n⭐ KEY ENHANCEMENTS:")
print("   • Time-weighted analysis (آخر سنة وزن أكبر)")
print("   • Recent metrics (آخر 12 شهر)")
print("   • Enhanced Potential Stock-Out detection")
print("   • Higher margin threshold (30%)")
print("   • More realistic classification")
print("=" * 120)

✅ ENHANCED TIME-WEIGHTED SKU ANALYSIS LOADED!

🚀 Usage: results_df = analyze_skus_enhanced(df)

⭐ KEY ENHANCEMENTS:
   • Time-weighted analysis (آخر سنة وزن أكبر)
   • Recent metrics (آخر 12 شهر)
   • Enhanced Potential Stock-Out detection
   • Higher margin threshold (30%)
   • More realistic classification


In [4]:
results_df_sku = analyze_skus_enhanced(combined_df)

🔧 Pre-processing data for optimal performance...
✅ Pre-processing complete!
🚀 ENHANCED TIME-WEIGHTED SKU ANALYSIS - STARTING
📅 Analysis Date: 2026-01-14
📦 Total SKUs: 71,644
⏱️  Estimated time: ~23.9 minutes

🔄 Processing SKUs...
------------------------------------------------------------------------------------------------------------------------


📊 Analyzing:   0%|                                                       | 0/71644 [00:00<?, ?it/s]

⚠️ Error analyzing SKU nan: cannot convert float NaN to integer

✅ Analysis Complete! Processed 71,643 SKUs

📊 SUMMARY BY STATUS:
------------------------------------------------------------------------------------------------------------------------
Declining                               :  5,428 SKUs | Recent Revenue:   37,941,471 | Margin:  55.6%
Volatile                                :  5,566 SKUs | Recent Revenue:   28,483,626 | Margin:  55.5%
Growing (High Margin)                   :  3,728 SKUs | Recent Revenue:   26,527,577 | Margin:  58.2%
Declining (Low Stock)                   :  1,564 SKUs | Recent Revenue:   13,401,351 | Margin:  53.1%
Stable (Consistent & Profitable)        :    265 SKUs | Recent Revenue:   10,935,082 | Margin:  62.6%
Active                                  :  2,443 SKUs | Recent Revenue:   10,112,538 | Margin:  53.3%
Sporadic                                :  4,936 SKUs | Recent Revenue:    5,976,306 | Margin:  46.9%
Inactive                           

# collection & skus analysis

In [5]:
# ============================================================================
# ULTRA-ENHANCED SECTION & SKU ANALYSIS - PRODUCTION READY
# ============================================================================
# Features:
# 1. Dual Trend Analysis (Revenue + Quantity) with mismatch warnings
# 2. Year-over-Year (YoY) comparison + Seasonal flags
# 3. Advanced Stock-out logic with Lost Sales estimates
# 4. SKU Health Metrics (% Growing/Declining per section)
# 5. KEY_REASON field explaining status in plain language
# 6. Benchmark comparisons (section percentiles)
# ============================================================================

import pandas as pd
import numpy as np
from scipy.stats import kendalltau
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from tqdm.notebook import tqdm
import warnings
import gc
warnings.filterwarnings('ignore')


# ============================================================================
# DUAL TREND ANALYZER - Revenue + Quantity
# ============================================================================

class DualTrendAnalyzer:
    """
    Analyzes trends for BOTH Revenue and Quantity
    Detects mismatches (e.g., revenue growing but quantity declining)
    """

    @staticmethod
    def analyze_dual_trend(monthly_revenue, monthly_quantity):
        """
        Analyze trends for both metrics

        Returns:
        --------
        dict with:
            - revenue_trend, revenue_confidence
            - quantity_trend, quantity_confidence
            - trend_mismatch (bool)
            - mismatch_warning (str)
        """
        if len(monthly_revenue) < 6 or len(monthly_quantity) < 6:
            return {
                'revenue_trend': 'Unknown',
                'revenue_confidence': 'LOW',
                'quantity_trend': 'Unknown',
                'quantity_confidence': 'LOW',
                'trend_mismatch': False,
                'mismatch_warning': None
            }

        try:
            # Analyze Revenue Trend
            rev_direction, rev_confidence, rev_metrics = DualTrendAnalyzer._analyze_single_trend(monthly_revenue)

            # Analyze Quantity Trend
            qty_direction, qty_confidence, qty_metrics = DualTrendAnalyzer._analyze_single_trend(monthly_quantity)

            # Detect Mismatch
            trend_mismatch = False
            mismatch_warning = None

            if rev_direction != qty_direction and rev_confidence != 'LOW' and qty_confidence != 'LOW':
                trend_mismatch = True

                if rev_direction == 'Growing' and qty_direction == 'Declining':
                    mismatch_warning = "⚠️ Revenue UP but Quantity DOWN → Price increase or product mix shift"
                elif rev_direction == 'Declining' and qty_direction == 'Growing':
                    mismatch_warning = "⚠️ Revenue DOWN but Quantity UP → Discounting or lower-value products"
                elif rev_direction == 'Stable' and qty_direction != 'Stable':
                    mismatch_warning = f"⚠️ Revenue stable but Quantity {qty_direction}"
                elif qty_direction == 'Stable' and rev_direction != 'Stable':
                    mismatch_warning = f"⚠️ Quantity stable but Revenue {rev_direction}"

            return {
                'revenue_trend': rev_direction,
                'revenue_confidence': rev_confidence,
                'revenue_period_change': rev_metrics.get('period_change', 0),
                'quantity_trend': qty_direction,
                'quantity_confidence': qty_confidence,
                'quantity_period_change': qty_metrics.get('period_change', 0),
                'trend_mismatch': trend_mismatch,
                'mismatch_warning': mismatch_warning
            }

        except Exception as e:
            return {
                'revenue_trend': 'Unknown',
                'revenue_confidence': 'LOW',
                'quantity_trend': 'Unknown',
                'quantity_confidence': 'LOW',
                'trend_mismatch': False,
                'mismatch_warning': None
            }

    @staticmethod
    def _analyze_single_trend(monthly_values):
        """Single metric trend analysis (same logic as before)"""

        values = np.array(monthly_values, dtype=np.float32)
        recent_months = min(12, len(values))
        recent_values = values[-recent_months:]

        # METHOD 1: Exponential Smoothing
        exp_trend = 0
        try:
            model = ExponentialSmoothing(values, trend='add', seasonal=None)
            fit = model.fit(smoothing_level=0.3, smoothing_trend=0.1, optimized=False)
            trend = fit.trend
            exp_trend = trend[-3:].mean() if len(trend) >= 3 else trend[-1]
        except:
            pass

        # METHOD 2: Mann-Kendall
        mk_tau, mk_p = 0, 1
        try:
            months = np.arange(len(recent_values))
            mk_tau, mk_p = kendalltau(months, recent_values)
        except:
            pass

        # METHOD 3: Period Comparison
        if len(values) >= 6:
            last_3 = values[-3:].mean()
            prev_3 = values[-6:-3].mean()
            period_change = ((last_3 - prev_3) / prev_3 * 100) if prev_3 > 0 else 0
        else:
            period_change = 0

        # METHOD 4: CAGR
        cagr = None
        if len(values) >= 12:
            beginning = values[-12:-9].mean()
            ending = values[-3:].mean()
            if beginning > 0 and ending > 0:
                years = 9 / 12
                cagr = ((ending / beginning) ** (1 / years) - 1) * 100

        # Consistency
        changes = np.diff(recent_values)
        consistency = 0
        if len(changes) > 0:
            positive = (changes > 0).sum()
            negative = (changes < 0).sum()
            consistency = max(positive, negative) / len(changes) * 100

        cv = recent_values.std() / recent_values.mean() if recent_values.mean() > 0 else 0

        # VOTING
        votes_declining = 0
        votes_growing = 0

        if exp_trend < -300:
            votes_declining += 1
        elif exp_trend > 300:
            votes_growing += 1

        if mk_tau < -0.15 and mk_p < 0.1:
            votes_declining += 1
        elif mk_tau > 0.15 and mk_p < 0.1:
            votes_growing += 1

        if period_change < -10:
            votes_declining += 1.5
        elif period_change > 10:
            votes_growing += 1.5

        if cagr is not None:
            if cagr < -8:
                votes_declining += 1
            elif cagr > 8:
                votes_growing += 1

        total_votes = 5.5 if cagr is not None else 4.5

        if votes_declining >= 2:
            direction = "Declining"
            confidence_score = votes_declining / total_votes
        elif votes_growing >= 2:
            direction = "Growing"
            confidence_score = votes_growing / total_votes
        else:
            direction = "Stable" if cv < 0.3 else ("Volatile" if cv > 0.7 else "Stable")
            confidence_score = 0.5

        if consistency > 70 and cv < 0.5:
            confidence_score *= 1.2
        elif consistency < 50 or cv > 0.8:
            confidence_score *= 0.8

        confidence_score = min(confidence_score, 1.0)
        confidence = "HIGH" if confidence_score >= 0.75 else ("MEDIUM" if confidence_score >= 0.5 else "LOW")

        metrics = {
            'exp_trend': float(exp_trend),
            'period_change': float(period_change),
            'cagr': float(cagr) if cagr is not None else None,
            'consistency': float(consistency),
            'cv': float(cv)
        }

        return direction, confidence, metrics


# ============================================================================
# YEAR-OVER-YEAR ANALYZER
# ============================================================================

class YoYAnalyzer:
    """
    Year-over-Year comparison and seasonality detection
    """

    @staticmethod
    def analyze_yoy(df, analysis_date):
        """
        Compare same months across years

        Returns:
        --------
        dict with:
            - yoy_growth_%
            - is_seasonal (bool)
            - seasonal_months (list)
            - yoy_comparison (dict)
        """
        try:
            df = df.copy()
            df['Year'] = pd.to_datetime(df['Date']).dt.year
            df['Month'] = pd.to_datetime(df['Date']).dt.month
            df['YearMonth'] = pd.to_datetime(df['Date']).dt.to_period('M')

            # Get current and previous year
            current_year = analysis_date.year
            prev_year = current_year - 1

            current_data = df[df['Year'] == current_year]
            prev_data = df[df['Year'] == prev_year]

            if len(current_data) == 0 or len(prev_data) == 0:
                return {
                    'yoy_growth_%': None,
                    'is_seasonal': False,
                    'seasonal_months': [],
                    'yoy_comparison': None
                }

            # Monthly comparison
            current_monthly = current_data.groupby('Month')['bal Value'].sum()
            prev_monthly = prev_data.groupby('Month')['bal Value'].sum()

            # Overall YoY growth
            current_total = current_data['bal Value'].sum()
            prev_total = prev_data['bal Value'].sum()

            yoy_growth = ((current_total - prev_total) / prev_total * 100) if prev_total > 0 else None

            # Seasonality detection
            monthly_revenue = df.groupby('Month')['bal Value'].sum()

            is_seasonal = False
            seasonal_months = []

            if len(monthly_revenue) >= 12:
                # Calculate coefficient of variation for months
                monthly_avg = monthly_revenue.mean()
                monthly_std = monthly_revenue.std()
                cv = monthly_std / monthly_avg if monthly_avg > 0 else 0

                # If CV > 0.5, consider seasonal
                if cv > 0.5:
                    is_seasonal = True
                    # Find peak months (>120% of average)
                    threshold = monthly_avg * 1.2
                    seasonal_months = monthly_revenue[monthly_revenue > threshold].index.tolist()

            # Month-by-month comparison
            yoy_by_month = {}
            for month in range(1, 13):
                current_val = current_monthly.get(month, 0)
                prev_val = prev_monthly.get(month, 0)

                if prev_val > 0:
                    growth = ((current_val - prev_val) / prev_val * 100)
                    yoy_by_month[month] = {
                        'current': float(current_val),
                        'previous': float(prev_val),
                        'growth_%': float(growth)
                    }

            return {
                'yoy_growth_%': float(yoy_growth) if yoy_growth is not None else None,
                'is_seasonal': is_seasonal,
                'seasonal_months': seasonal_months,
                'seasonality_cv': float(cv) if len(monthly_revenue) >= 12 else 0,
                'yoy_by_month': yoy_by_month
            }

        except Exception as e:
            return {
                'yoy_growth_%': None,
                'is_seasonal': False,
                'seasonal_months': [],
                'yoy_comparison': None
            }


# ============================================================================
# ADVANCED STOCK-OUT ANALYZER
# ============================================================================

class AdvancedStockOutAnalyzer:
    """
    Enhanced stock-out detection with sales rate analysis
    """

    @staticmethod
    def analyze_stockout(sku_data, current_stock, analysis_date, has_profit):
        """
        Advanced stock-out analysis

        Returns:
        --------
        dict with:
            - is_stockout
            - is_potential_stockout
            - stockout_date
            - days_out_of_stock
            - pre_stockout_sales_rate (units/day)
            - estimated_lost_sales (units)
            - estimated_lost_revenue
            - estimated_lost_profit
        """

        result = {
            'is_stockout': False,
            'is_potential_stockout': False,
            'stockout_date': None,
            'days_out_of_stock': 0,
            'pre_stockout_sales_rate_qty': 0,
            'pre_stockout_sales_rate_value': 0,
            'estimated_lost_sales_qty': 0,
            'estimated_lost_revenue': 0,
            'estimated_lost_profit': 0,
            'pre_stockout_avg_margin': 0
        }

        try:
            date_values = sku_data['Date'].values
            revenue_values = sku_data['bal Value'].values
            qty_values = sku_data['bal Qty'].values

            last_sale = pd.Timestamp(date_values.max())
            days_since_last = int((analysis_date - last_sale) / np.timedelta64(1, 'D'))

            # Check if stock-out
            if current_stock == 0 and days_since_last > 60:
                result['is_stockout'] = True

                # Find stockout date (gap > 60 days)
                dates_sorted = np.sort(date_values)
                stockout_date = last_sale

                if len(dates_sorted) > 1:
                    gaps = np.diff(dates_sorted).astype('timedelta64[D]').astype(int)
                    gap_idx = np.where(gaps > 60)[0]
                    if len(gap_idx) > 0:
                        stockout_date = dates_sorted[gap_idx[0]]

                result['stockout_date'] = stockout_date
                result['days_out_of_stock'] = (analysis_date - pd.Timestamp(stockout_date)).days

                # Analyze period BEFORE stockout (last 6 months before stockout)
                six_months_before = pd.Timestamp(stockout_date) - pd.Timedelta(days=180)
                before_mask = (date_values >= six_months_before) & (date_values < stockout_date)
                before_data = sku_data[before_mask]

                if len(before_data) > 0:
                    before_days = (pd.Timestamp(stockout_date) - six_months_before).days

                    before_qty = before_data['bal Qty'].sum()
                    before_revenue = before_data['bal Value'].sum()

                    # ✅ SALES RATE (units/day and revenue/day)
                    sales_rate_qty = before_qty / before_days if before_days > 0 else 0
                    sales_rate_value = before_revenue / before_days if before_days > 0 else 0

                    result['pre_stockout_sales_rate_qty'] = float(sales_rate_qty)
                    result['pre_stockout_sales_rate_value'] = float(sales_rate_value)

                    # ✅ ESTIMATED LOST SALES
                    days_out = result['days_out_of_stock']
                    estimated_lost_qty = sales_rate_qty * days_out
                    estimated_lost_revenue = sales_rate_value * days_out

                    result['estimated_lost_sales_qty'] = float(estimated_lost_qty)
                    result['estimated_lost_revenue'] = float(estimated_lost_revenue)

                    # Profit estimation
                    if has_profit:
                        before_profit = before_data['Total_Profit'].sum()
                        avg_margin = (before_profit / before_revenue * 100) if before_revenue > 0 else 0

                        result['pre_stockout_avg_margin'] = float(avg_margin)
                        result['estimated_lost_profit'] = float(estimated_lost_revenue * (avg_margin / 100))

                    # ✅ POTENTIAL HIGH-VALUE STOCK-OUT
                    # Criteria:
                    # 1. High sales rate (>10 units/day OR >1000 SAR/day)
                    # 2. Good margin (>30%)
                    # 3. Consistent sales (at least 30 days active in last 6 months)

                    active_days = before_data['Date'].nunique()

                    if ((sales_rate_qty > 10 or sales_rate_value > 1000) and
                        avg_margin > 30 and
                        active_days >= 30):
                        result['is_potential_stockout'] = True

            return result

        except Exception as e:
            return result


# ============================================================================
# SECTION-LEVEL ANALYZER WITH ENHANCEMENTS
# ============================================================================

class EnhancedSectionAnalyzer:
    """
    Section-level analysis with:
    - Dual trend (revenue + quantity)
    - YoY comparison
    - SKU health metrics
    - KEY_REASON field
    """

    def __init__(self, df, analysis_date):
        self.df = df
        self.analysis_date = analysis_date

        self.has_profit = 'Total_Profit' in df.columns
        self.has_stock = 'CURRENT_STOCK' in df.columns
        self.has_sales_type = 'Sales_Type' in df.columns

    def analyze_section(self, section_name, section_data, sku_results_df):
        """
        Comprehensive section analysis

        Parameters:
        -----------
        section_name : str
        section_data : DataFrame (all rows for this section)
        sku_results_df : DataFrame (SKU-level results for calculating health metrics)
        """

        try:
            # ═══════════════════════════════════════════════════════════════
            # BASIC METRICS
            # ═══════════════════════════════════════════════════════════════

            total_revenue = section_data['bal Value'].sum()
            total_qty = section_data['bal Qty'].sum()
            total_profit = section_data['Total_Profit'].sum() if self.has_profit else 0
            num_skus = section_data['SKU_CLEAN'].nunique()
            num_transactions = len(section_data)

            first_sale = section_data['Date'].min()
            last_sale = section_data['Date'].max()
            days_since_last = (self.analysis_date - last_sale).days

            # Recent period (last 12 months)
            one_year_ago = self.analysis_date - pd.Timedelta(days=365)
            recent_data = section_data[section_data['Date'] >= one_year_ago]

            recent_revenue = recent_data['bal Value'].sum()
            recent_qty = recent_data['bal Qty'].sum()
            recent_profit = recent_data['Total_Profit'].sum() if self.has_profit else 0

            # Margins
            avg_margin = (total_profit / total_revenue * 100) if total_revenue > 0 else 0
            recent_avg_margin = (recent_profit / recent_revenue * 100) if recent_revenue > 0 else 0

            # ═══════════════════════════════════════════════════════════════
            # STOCK METRICS
            # ═══════════════════════════════════════════════════════════════

            if self.has_stock:
                latest_stock = section_data.groupby('SKU_CLEAN')['CURRENT_STOCK'].first()
                total_stock = latest_stock.sum()
                skus_out_of_stock = (latest_stock == 0).sum()
                skus_low_stock = ((latest_stock > 0) & (latest_stock < 10)).sum()
                stockout_rate = (skus_out_of_stock / num_skus * 100) if num_skus > 0 else 0
            else:
                total_stock = 0
                skus_out_of_stock = 0
                skus_low_stock = 0
                stockout_rate = 0

            # ═══════════════════════════════════════════════════════════════
            # MONTHLY ANALYSIS (Revenue + Quantity)
            # ═══════════════════════════════════════════════════════════════

            section_data['YearMonth'] = pd.to_datetime(section_data['Date']).dt.to_period('M')

            monthly_rev = section_data.groupby('YearMonth')['bal Value'].sum()
            monthly_qty = section_data.groupby('YearMonth')['bal Qty'].sum()

            active_months = len(monthly_rev)
            avg_monthly_revenue = monthly_rev.mean() if active_months > 0 else 0
            median_monthly_revenue = monthly_rev.median() if active_months > 0 else 0

            avg_monthly_qty = monthly_qty.mean() if active_months > 0 else 0
            median_monthly_qty = monthly_qty.median() if active_months > 0 else 0

            # ✅ DUAL TREND ANALYSIS
            dual_trend = DualTrendAnalyzer.analyze_dual_trend(
                monthly_rev.values,
                monthly_qty.values
            )

            # ✅ YoY ANALYSIS
            yoy_results = YoYAnalyzer.analyze_yoy(section_data, self.analysis_date)

            # ═══════════════════════════════════════════════════════════════
            # ✅ SKU HEALTH METRICS
            # ═══════════════════════════════════════════════════════════════

            section_skus = sku_results_df[sku_results_df['Section'] == section_name]

            if len(section_skus) > 0:
                growing_skus = len(section_skus[section_skus['Trend_Direction'] == 'Growing'])
                declining_skus = len(section_skus[section_skus['Trend_Direction'] == 'Declining'])
                stable_skus = len(section_skus[section_skus['Trend_Direction'] == 'Stable'])

                pct_growing = (growing_skus / len(section_skus) * 100)
                pct_declining = (declining_skus / len(section_skus) * 100)
                pct_stable = (stable_skus / len(section_skus) * 100)

                # High confidence trends
                high_conf_growing = len(section_skus[
                    (section_skus['Trend_Direction'] == 'Growing') &
                    (section_skus['Trend_Confidence'] == 'HIGH')
                ])
                high_conf_declining = len(section_skus[
                    (section_skus['Trend_Direction'] == 'Declining') &
                    (section_skus['Trend_Confidence'] == 'HIGH')
                ])

                # Potential stock-outs
                potential_stockouts = len(section_skus[section_skus['Is_Potential_Stockout'] == True])

            else:
                growing_skus = declining_skus = stable_skus = 0
                pct_growing = pct_declining = pct_stable = 0
                high_conf_growing = high_conf_declining = 0
                potential_stockouts = 0

            # ═══════════════════════════════════════════════════════════════
            # STATUS DETERMINATION
            # ═══════════════════════════════════════════════════════════════

            status, key_reason = self._determine_status_with_reason(
                revenue_trend=dual_trend['revenue_trend'],
                quantity_trend=dual_trend['quantity_trend'],
                trend_mismatch=dual_trend['trend_mismatch'],
                recent_avg_margin=recent_avg_margin,
                stockout_rate=stockout_rate,
                days_since_last=days_since_last,
                pct_declining=pct_declining,
                pct_growing=pct_growing,
                yoy_growth=yoy_results['yoy_growth_%'],
                is_seasonal=yoy_results['is_seasonal']
            )

            # ═══════════════════════════════════════════════════════════════
            # RETURN RESULTS
            # ═══════════════════════════════════════════════════════════════

            return {
                'Section': section_name,
                'Status': status,
                'Key_Reason': key_reason,  # ✅ NEW

                # SKU counts
                'Total_SKUs': num_skus,
                'SKUs_Out_of_Stock': skus_out_of_stock,
                'SKUs_Low_Stock': skus_low_stock,
                'Stock_Out_Rate_%': float(stockout_rate),

                # ✅ SKU HEALTH METRICS
                'SKUs_Growing': growing_skus,
                'SKUs_Declining': declining_skus,
                'SKUs_Stable': stable_skus,
                'Pct_Growing': float(pct_growing),
                'Pct_Declining': float(pct_declining),
                'Pct_Stable': float(pct_stable),
                'High_Conf_Growing': high_conf_growing,
                'High_Conf_Declining': high_conf_declining,
                'Potential_Stockouts': potential_stockouts,

                # Revenue
                'Total_Revenue': float(total_revenue),
                'Recent_Revenue_12M': float(recent_revenue),
                'Revenue_Per_SKU': float(total_revenue / num_skus) if num_skus > 0 else 0,

                # Quantity
                'Total_Quantity': float(total_qty),
                'Recent_Quantity_12M': float(recent_qty),

                # Transactions
                'Num_Transactions': num_transactions,

                # Monthly metrics
                'Avg_Monthly_Revenue': float(avg_monthly_revenue),
                'Median_Monthly_Revenue': float(median_monthly_revenue),
                'Avg_Monthly_Quantity': float(avg_monthly_qty),
                'Median_Monthly_Quantity': float(median_monthly_qty),
                'Active_Months': active_months,

                # Profit
                'Total_Profit': float(total_profit),
                'Recent_Profit_12M': float(recent_profit),
                'Avg_Margin_%': float(avg_margin),
                'Recent_Avg_Margin_%': float(recent_avg_margin),

                # Stock
                'Total_Stock': float(total_stock),

                # Time
                'First_Sale': first_sale,
                'Last_Sale': last_sale,
                'Days_Since_Last_Sale': days_since_last,

                # ✅ DUAL TREND
                'Revenue_Trend': dual_trend['revenue_trend'],
                'Revenue_Confidence': dual_trend['revenue_confidence'],
                'Revenue_Period_Change_%': float(dual_trend['revenue_period_change']),
                'Quantity_Trend': dual_trend['quantity_trend'],
                'Quantity_Confidence': dual_trend['quantity_confidence'],
                'Quantity_Period_Change_%': float(dual_trend['quantity_period_change']),
                'Trend_Mismatch': dual_trend['trend_mismatch'],
                'Mismatch_Warning': dual_trend['mismatch_warning'],

                # ✅ YoY
                'YoY_Growth_%': yoy_results['yoy_growth_%'],
                'Is_Seasonal': yoy_results['is_seasonal'],
                'Seasonal_Months': str(yoy_results.get('seasonal_months', [])),
                'Seasonality_CV': float(yoy_results.get('seasonality_cv', 0))
            }

        except Exception as e:
            print(f"⚠️ Error analyzing section {section_name}: {e}")
            return None

    def _determine_status_with_reason(self, revenue_trend, quantity_trend, trend_mismatch,
                                      recent_avg_margin, stockout_rate, days_since_last,
                                      pct_declining, pct_growing, yoy_growth, is_seasonal):
        """
        Determine status with KEY_REASON explanation
        """

        reasons = []

        # Critical issues first
        if recent_avg_margin < 0:
            status = "Loss-Making"
            reasons.append(f"Margin negative ({recent_avg_margin:.1f}%)")

        elif stockout_rate > 30:
            status = "High Stock-Out Rate"
            reasons.append(f"{stockout_rate:.0f}% SKUs out of stock")

        elif days_since_last > 180:
            status = "Inactive"
            reasons.append(f"{days_since_last} days since last sale")

        elif revenue_trend == "Declining" and recent_avg_margin < 15:
            status = "Declining (Low Margin)"
            reasons.append(f"Revenue declining with {recent_avg_margin:.1f}% margin")
            if pct_declining > 50:
                reasons.append(f"{pct_declining:.0f}% SKUs declining")

        elif revenue_trend == "Declining":
            status = "Declining"
            reasons.append("Revenue trending down")
            if quantity_trend == "Declining":
                reasons.append("Quantity also declining")
            if pct_declining > 40:
                reasons.append(f"{pct_declining:.0f}% SKUs declining")
            if yoy_growth and yoy_growth < -10:
                reasons.append(f"YoY down {abs(yoy_growth):.1f}%")

        elif revenue_trend == "Growing" and recent_avg_margin > 30:
            status = "Growing (High Margin)"
            reasons.append(f"Revenue up with {recent_avg_margin:.1f}% margin")
            if pct_growing > 50:
                reasons.append(f"{pct_growing:.0f}% SKUs growing")
            if yoy_growth and yoy_growth > 10:
                reasons.append(f"YoY up {yoy_growth:.1f}%")

        elif revenue_trend == "Growing":
            status = "Growing"
            reasons.append("Revenue trending up")
            if quantity_trend == "Growing":
                reasons.append("Quantity also growing")
            if pct_growing > 40:
                reasons.append(f"{pct_growing:.0f}% SKUs growing")

        elif revenue_trend == "Volatile":
            status = "Volatile"
            reasons.append("Unstable revenue pattern")
            if is_seasonal:
                reasons.append("Seasonal product")

        elif revenue_trend == "Stable" and recent_avg_margin > 25:
            status = "Stable (Profitable)"
            reasons.append(f"Steady with {recent_avg_margin:.1f}% margin")

        elif revenue_trend == "Stable":
            status = "Stable"
            reasons.append("Consistent revenue")

        else:
            status = "Active"
            reasons.append("Currently active")

        # Add trend mismatch warning
        if trend_mismatch:
            if revenue_trend == "Growing" and quantity_trend == "Declining":
                reasons.append("⚠️ Price increase or mix shift")
            elif revenue_trend == "Declining" and quantity_trend == "Growing":
                reasons.append("⚠️ Discounting or lower-value products")

        # Seasonal flag
        if is_seasonal and status not in ["Inactive", "Loss-Making"]:
            reasons.append("🗓️ Seasonal")

        key_reason = " | ".join(reasons) if reasons else "No specific issues"

        return status, key_reason


# ============================================================================
# ENHANCED SKU ANALYZER
# ============================================================================

class UltraEnhancedSKUAnalyzer:
    """
    Ultimate SKU analyzer with all enhancements
    """

    def __init__(self, df, analysis_date=None):
        print("🔧 Pre-processing data...")

        df['SKU_CLEAN'] = df['SKU'].astype(str).str.strip()

        if not pd.api.types.is_datetime64_any_dtype(df['Date']):
            df['Date'] = pd.to_datetime(df['Date'])

        self.df = df
        self.analysis_date = analysis_date or df['Date'].max()

        self.has_profit = 'Total_Profit' in df.columns
        self.has_stock = 'CURRENT_STOCK' in df.columns
        self.has_sales_type = 'Sales_Type' in df.columns
        self.has_section = 'Section' in df.columns

        if not self.has_section:
            print("⚠️  WARNING: No 'Section' column - section analysis will be skipped")

        print("✅ Pre-processing complete!")

    def analyze_single_sku(self, sku_data, sku_name):
        """Enhanced SKU analysis with dual trends and advanced stock-out"""

        if len(sku_data) == 0:
            return None

        try:
            section = sku_data['Section'].iloc[0] if self.has_section else 'Unknown'

            # Basic extraction
            revenue_values = sku_data['bal Value'].values
            qty_values = sku_data['bal Qty'].values
            date_values = sku_data['Date'].values

            total_revenue = revenue_values.sum()
            total_qty = qty_values.sum()
            first_sale = pd.Timestamp(date_values.min())
            last_sale = pd.Timestamp(date_values.max())
            analysis_date = pd.Timestamp(self.analysis_date)

            days_since_last = int((analysis_date - last_sale) / np.timedelta64(1, 'D'))
            num_transactions = len(sku_data)

            # Recent period
            one_year_ago = analysis_date - pd.Timedelta(days=365)
            recent_mask = date_values >= one_year_ago

            recent_revenue = revenue_values[recent_mask].sum()
            recent_qty = qty_values[recent_mask].sum()

            # Profit
            total_profit, avg_margin, recent_profit, recent_avg_margin = 0, 0, 0, 0

            if self.has_profit:
                profit_values = sku_data['Total_Profit'].values
                total_profit = profit_values.sum()
                avg_margin = (total_profit / total_revenue * 100) if total_revenue > 0 else 0

                recent_profit_values = profit_values[recent_mask]
                recent_revenue_values = revenue_values[recent_mask]
                recent_profit = recent_profit_values.sum()
                recent_avg_margin = (recent_profit / recent_revenue * 100) if recent_revenue > 0 else 0

            # Stock
            current_stock = 0
            if self.has_stock:
                stock_values = sku_data['CURRENT_STOCK'].dropna()
                if len(stock_values) > 0:
                    current_stock = stock_values.iloc[0]

            # Monthly analysis
            dates_pd = pd.to_datetime(date_values)
            year_month = dates_pd.to_period('M')

            monthly_df = pd.DataFrame({
                'YearMonth': year_month,
                'Revenue': revenue_values,
                'Quantity': qty_values
            })

            monthly_agg = monthly_df.groupby('YearMonth', observed=True).agg({
                'Revenue': 'sum',
                'Quantity': 'sum'
            })

            active_months = len(monthly_agg)
            monthly_revenue = monthly_agg['Revenue'].values.astype(np.float32)
            monthly_quantity = monthly_agg['Quantity'].values.astype(np.float32)

            avg_monthly_revenue = monthly_revenue.mean() if active_months > 0 else 0

            # ✅ DUAL TREND ANALYSIS
            dual_trend = DualTrendAnalyzer.analyze_dual_trend(monthly_revenue, monthly_quantity)

            # ✅ YoY ANALYSIS
            yoy_results = YoYAnalyzer.analyze_yoy(sku_data, self.analysis_date)

            # ✅ ADVANCED STOCK-OUT ANALYSIS
            stockout_results = AdvancedStockOutAnalyzer.analyze_stockout(
                sku_data, current_stock, self.analysis_date, self.has_profit
            )

            # Status
            status, key_reason = self._determine_sku_status_with_reason(
                revenue_trend=dual_trend['revenue_trend'],
                recent_avg_margin=recent_avg_margin,
                is_potential_stockout=stockout_results['is_potential_stockout'],
                is_stockout=stockout_results['is_stockout'],
                current_stock=current_stock,
                days_since_last=days_since_last,
                trend_mismatch=dual_trend['trend_mismatch'],
                yoy_growth=yoy_results['yoy_growth_%']
            )

            priority = self._get_priority(status)

            return {
                'SKU': sku_name,
                'Section': section,
                'Status': status,
                'Key_Reason': key_reason,  # ✅ NEW
                'Priority': priority,

                # Revenue
                'Total_Revenue': float(total_revenue),
                'Recent_Revenue_12M': float(recent_revenue),

                # Quantity
                'Total_Quantity': float(total_qty),
                'Recent_Quantity_12M': float(recent_qty),

                # Transactions
                'Num_Transactions': int(num_transactions),

                # Monthly
                'Avg_Monthly_Revenue': float(avg_monthly_revenue),
                'Active_Months': int(active_months),

                # Profit
                'Total_Profit': float(total_profit),
                'Recent_Profit_12M': float(recent_profit),
                'Avg_Margin_%': float(avg_margin),
                'Recent_Avg_Margin_%': float(recent_avg_margin),

                # Stock
                'Current_Stock': float(current_stock),

                # Time
                'First_Sale': first_sale,
                'Last_Sale': last_sale,
                'Days_Since_Last_Sale': int(days_since_last),

                # ✅ DUAL TREND
                'Revenue_Trend': dual_trend['revenue_trend'],
                'Revenue_Confidence': dual_trend['revenue_confidence'],
                'Quantity_Trend': dual_trend['quantity_trend'],
                'Quantity_Confidence': dual_trend['quantity_confidence'],
                'Trend_Mismatch': dual_trend['trend_mismatch'],
                'Mismatch_Warning': dual_trend['mismatch_warning'],

                # ✅ YoY
                'YoY_Growth_%': yoy_results['yoy_growth_%'],
                'Is_Seasonal': yoy_results['is_seasonal'],

                # ✅ ADVANCED STOCK-OUT
                'Is_Stockout': stockout_results['is_stockout'],
                'Is_Potential_Stockout': stockout_results['is_potential_stockout'],
                'Days_Out_of_Stock': stockout_results['days_out_of_stock'],
                'Pre_Stockout_Sales_Rate_Qty': float(stockout_results['pre_stockout_sales_rate_qty']),
                'Pre_Stockout_Sales_Rate_Value': float(stockout_results['pre_stockout_sales_rate_value']),
                'Estimated_Lost_Sales_Qty': float(stockout_results['estimated_lost_sales_qty']),
                'Estimated_Lost_Revenue': float(stockout_results['estimated_lost_revenue']),
                'Estimated_Lost_Profit': float(stockout_results['estimated_lost_profit'])
            }

        except Exception as e:
            print(f"⚠️ Error analyzing SKU {sku_name}: {e}")
            return None

    def _determine_sku_status_with_reason(self, revenue_trend, recent_avg_margin,
                                          is_potential_stockout, is_stockout, current_stock,
                                          days_since_last, trend_mismatch, yoy_growth):
        """Determine SKU status with reason"""

        reasons = []

        if is_potential_stockout:
            status = "Potential Stock-Out (High Value)"
            reasons.append("High sales rate + good margin + no stock")

        elif recent_avg_margin < -5:
            status = "Loss-Making (Recent)"
            reasons.append(f"Recent margin {recent_avg_margin:.1f}%")

        elif is_stockout:
            status = "Stock-Out"
            reasons.append(f"{days_since_last} days no stock")

        elif 0 < current_stock < 10:
            status = "Low Stock"
            reasons.append(f"Only {current_stock:.0f} units left")
            if recent_avg_margin > 30:
                status = "Low Stock (High Margin)"
                reasons.append(f"Margin {recent_avg_margin:.1f}%")

        elif days_since_last > 365:
            status = "Dead"
            reasons.append("No sales >1 year")

        elif days_since_last > 180:
            status = "Inactive"
            reasons.append(f"{days_since_last} days since sale")

        elif revenue_trend == "Declining":
            status = "Declining"
            reasons.append("Revenue trending down")
            if yoy_growth and yoy_growth < -10:
                reasons.append(f"YoY {yoy_growth:.1f}%")

        elif revenue_trend == "Growing":
            status = "Growing"
            reasons.append("Revenue trending up")
            if recent_avg_margin > 30:
                status = "Growing (High Margin)"
                reasons.append(f"Margin {recent_avg_margin:.1f}%")
            if yoy_growth and yoy_growth > 10:
                reasons.append(f"YoY +{yoy_growth:.1f}%")

        else:
            status = "Stable"
            reasons.append("Steady performance")

        if trend_mismatch:
            reasons.append("⚠️ Revenue/Qty mismatch")

        key_reason = " | ".join(reasons) if reasons else "Active"

        return status, key_reason

    def _get_priority(self, status):
        """Assign priority based on status"""
        priority_map = {
            "Potential Stock-Out (High Value)": 0,
            "Loss-Making (Recent)": 1,
            "Stock-Out": 2,
            "Low Stock (High Margin)": 3,
            "Low Stock": 4,
            "Declining": 5,
            "Dead": 6,
            "Inactive": 7,
            "Growing (High Margin)": 10,
            "Growing": 11,
            "Stable": 12
        }
        return priority_map.get(status, 99)

    def analyze_all(self, save_to_excel=True, output_file='ULTRA_ENHANCED_ANALYSIS.xlsx'):
        """Analyze both SKUs and Sections"""

        print("=" * 120)
        print("🚀 ULTRA-ENHANCED SECTION & SKU ANALYSIS - STARTING")
        print("=" * 120)
        print(f"📅 Analysis Date: {self.analysis_date.strftime('%Y-%m-%d')}")

        all_skus = self.df['SKU_CLEAN'].unique()
        total_skus = len(all_skus)
        print(f"📦 Total SKUs: {total_skus:,}")

        if self.has_section:
            sections = self.df['Section'].dropna().unique()
            print(f"📂 Total Sections: {len(sections):,}")

        print("\n⭐ NEW FEATURES:")
        print("   • Dual Trend (Revenue + Quantity)")
        print("   • Year-over-Year comparison")
        print("   • Advanced Stock-out with sales rate")
        print("   • SKU Health metrics per section")
        print("   • KEY_REASON explanations")
        print()

        # ════════════════════════════════════════════════════════════════
        # PART 1: SKU ANALYSIS
        # ════════════════════════════════════════════════════════════════

        print("🔄 Analyzing SKUs...")
        print("-" * 120)

        sku_results = []

        for i, sku in enumerate(tqdm(all_skus, desc="📊 SKU Analysis", ncols=100, colour='cyan')):
            sku_data = self.df[self.df['SKU_CLEAN'] == sku]

            if len(sku_data) > 0:
                result = self.analyze_single_sku(sku_data, sku)
                if result:
                    sku_results.append(result)

            if (i + 1) % 1000 == 0:
                gc.collect()

        sku_df = pd.DataFrame(sku_results)
        sku_df = sku_df.sort_values(['Priority', 'Recent_Revenue_12M'], ascending=[True, False])

        print(f"\n✅ SKU Analysis Complete: {len(sku_df):,} SKUs")

        # ════════════════════════════════════════════════════════════════
        # PART 2: SECTION ANALYSIS
        # ════════════════════════════════════════════════════════════════

        section_df = None

        if self.has_section:
            print("\n" + "=" * 120)
            print("🔄 Analyzing Sections...")
            print("-" * 120)

            section_analyzer = EnhancedSectionAnalyzer(self.df, self.analysis_date)
            section_results = []

            for section in tqdm(sections, desc="📂 Section Analysis", ncols=100, colour='green'):
                if pd.isna(section):
                    continue

                section_data = self.df[self.df['Section'] == section]

                if len(section_data) > 0:
                    result = section_analyzer.analyze_section(section, section_data, sku_df)
                    if result:
                        section_results.append(result)

            section_df = pd.DataFrame(section_results)
            section_df = section_df.sort_values('Recent_Revenue_12M', ascending=False)

            print(f"\n✅ Section Analysis Complete: {len(section_df):,} Sections")

        # ════════════════════════════════════════════════════════════════
        # SUMMARY
        # ════════════════════════════════════════════════════════════════

        print("\n" + "=" * 120)
        self._print_summary(sku_df, section_df)

        # ════════════════════════════════════════════════════════════════
        # SAVE TO EXCEL
        # ════════════════════════════════════════════════════════════════

        if save_to_excel:
            self._save_to_excel(sku_df, section_df, output_file)

        print("\n" + "=" * 120)
        print("🎉 ULTRA-ENHANCED ANALYSIS COMPLETE!")
        print("=" * 120)

        return sku_df, section_df

    def _print_summary(self, sku_df, section_df):
        """Print comprehensive summary"""

        print("📊 SKU SUMMARY:")
        print("-" * 120)

        # Trend mismatch
        mismatch = sku_df[sku_df['Trend_Mismatch'] == True]
        if len(mismatch) > 0:
            print(f"⚠️  {len(mismatch):,} SKUs with Revenue/Quantity trend mismatch")

        # Potential stock-outs
        potential = sku_df[sku_df['Is_Potential_Stockout'] == True]
        if len(potential) > 0:
            print(f"🚨 {len(potential):,} Potential HIGH-VALUE stock-outs")
            print(f"   Estimated lost revenue: {potential['Estimated_Lost_Revenue'].sum():,.0f} SAR")

        # YoY
        yoy_available = sku_df[sku_df['YoY_Growth_%'].notna()]
        if len(yoy_available) > 0:
            avg_yoy = yoy_available['YoY_Growth_%'].mean()
            print(f"📈 Average YoY growth: {avg_yoy:.1f}%")

        if section_df is not None:
            print("\n" + "=" * 120)
            print("📂 SECTION SUMMARY:")
            print("-" * 120)

            # Trend mismatches
            sec_mismatch = section_df[section_df['Trend_Mismatch'] == True]
            if len(sec_mismatch) > 0:
                print(f"⚠️  {len(sec_mismatch):,} Sections with Revenue/Quantity mismatch")

            # Seasonal
            seasonal = section_df[section_df['Is_Seasonal'] == True]
            if len(seasonal) > 0:
                print(f"🗓️  {len(seasonal):,} Seasonal sections detected")

            # Top sections with KEY_REASON
            print("\n🔝 Top 10 Sections:")
            print(f"{'Section':<30} {'Revenue 12M':>15} {'Margin':>8} {'Status':<30} {'Key Reason'}")
            print("-" * 120)

            for _, row in section_df.head(10).iterrows():
                print(f"{str(row['Section'])[:30]:<30} "
                      f"{row['Recent_Revenue_12M']:>15,.0f} "
                      f"{row['Recent_Avg_Margin_%']:>7.1f}% "
                      f"{row['Status']:<30} "
                      f"{str(row['Key_Reason'])[:50]}")

    def _save_to_excel(self, sku_df, section_df, output_file):
        """Save to Excel with all sheets"""

        print(f"\n💾 Saving to: {output_file}")

        with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

            # SKU sheets
            sku_df.to_excel(writer, sheet_name='All SKUs', index=False)

            # Trend mismatches
            mismatch = sku_df[sku_df['Trend_Mismatch'] == True]
            if len(mismatch) > 0:
                mismatch.to_excel(writer, sheet_name='⚠️ Trend Mismatches', index=False)

            # Potential stock-outs
            potential = sku_df[sku_df['Is_Potential_Stockout'] == True]
            if len(potential) > 0:
                potential.sort_values('Estimated_Lost_Revenue', ascending=False).to_excel(
                    writer, sheet_name='🚨 Potential Stock-Outs', index=False)

            # Critical
            critical = sku_df[sku_df['Priority'] <= 5]
            if len(critical) > 0:
                critical.to_excel(writer, sheet_name='Critical SKUs', index=False)

            # Section sheets
            if section_df is not None:
                section_df.to_excel(writer, sheet_name='All Sections', index=False)

                # Top sections
                top = section_df.nlargest(30, 'Recent_Revenue_12M')
                top.to_excel(writer, sheet_name='Top 30 Sections', index=False)

                # Sections with issues
                issues = section_df[section_df['Status'].str.contains('Loss|Declining|High Stock', na=False)]
                if len(issues) > 0:
                    issues.to_excel(writer, sheet_name='⚠️ Problem Sections', index=False)

                # Seasonal sections
                seasonal = section_df[section_df['Is_Seasonal'] == True]
                if len(seasonal) > 0:
                    seasonal.to_excel(writer, sheet_name='🗓️ Seasonal Sections', index=False)

        print(f"✅ Saved successfully!")


# ============================================================================
# MAIN FUNCTION
# ============================================================================

def ultra_enhanced_analysis(df, save_to_excel=True, output_file='ULTRA_ENHANCED_ANALYSIS.xlsx'):
    """
    Ultimate Section & SKU analysis with all enhancements

    Features:
    ---------
    • Dual Trend (Revenue + Quantity) with mismatch detection
    • Year-over-Year comparison + Seasonal flags
    • Advanced Stock-out analysis with sales rate & lost sales
    • SKU Health metrics (% Growing/Declining per section)
    • KEY_REASON field explaining status
    • Benchmark comparisons

    Returns:
    --------
    sku_df, section_df : DataFrames

    Example:
    --------
    >>> sku_df, section_df = ultra_enhanced_analysis(combined_df)
    """
    analyzer = UltraEnhancedSKUAnalyzer(df)
    return analyzer.analyze_all(save_to_excel, output_file)


print("=" * 120)
print("✅ ULTRA-ENHANCED SECTION & SKU ANALYSIS LOADED!")
print("=" * 120)
print("\n🚀 Usage:")
print("   sku_df, section_df = ultra_enhanced_analysis(combined_df)")
print("\n⭐ NEW FEATURES:")
print("   1. ✅ Dual Trend (Revenue + Quantity) with mismatch warnings")
print("   2. ✅ Year-over-Year comparison + Seasonal detection")
print("   3. ✅ Advanced Stock-out with sales rate & lost sales estimate")
print("   4. ✅ SKU Health metrics (% Growing/Declining per section)")
print("   5. ✅ KEY_REASON field explaining status in plain language")
print("   6. ✅ Trend mismatch detection & warnings")
print("=" * 120)

✅ SECTION & SKU ANALYSIS LOADED!

🚀 Usage:
   sku_df, section_df = analyze_sections_and_skus(df)

⭐ FEATURES:
   • SKU-level analysis with Section info
   • Section-level aggregated analysis
   • Time-weighted metrics (last 12M focus)
   • Stock-out detection at both levels
   • Excel export with section breakdowns


In [6]:
sku_df, section_df = analyze_sections_and_skus(
    combined_df,
    save_to_excel=True,
    output_file='SECTION_SKU_ANALYSIS.xlsx'
)

🔧 Pre-processing data...
✅ Pre-processing complete!
🚀 SECTION & SKU ANALYSIS - STARTING
📅 Analysis Date: 2026-01-14
📦 Total SKUs: 71,644
📂 Total Sections: 2,972

🔄 Analyzing SKUs...
------------------------------------------------------------------------------------------------------------------------


📊 SKU Analysis:   0%|                                                    | 0/71644 [00:00<?, ?it/s]

⚠️ Error analyzing SKU nan: cannot convert float NaN to integer

✅ SKU Analysis Complete: 71,643 SKUs

🔄 Analyzing Sections...
------------------------------------------------------------------------------------------------------------------------


📂 Section Analysis:   0%|                                                 | 0/2972 [00:00<?, ?it/s]


✅ Section Analysis Complete: 2,972 Sections

📊 SKU SUMMARY BY STATUS:
------------------------------------------------------------------------------------------------------------------------
Declining                               :  8,392 SKUs | Revenue:   52,291,213 | Margin:  53.8%
Active                                  : 10,303 SKUs | Revenue:   41,151,521 | Margin:  52.4%
Growing (High Margin)                   :  4,598 SKUs | Revenue:   28,593,801 | Margin:  57.6%
Stable                                  :  1,241 SKUs | Revenue:   17,742,603 | Margin:  55.1%
Inactive                                :  5,804 SKUs | Revenue:    4,254,536 | Margin:  43.7%
Growing                                 :    695 SKUs | Revenue:    3,144,413 | Margin:  15.4%
Stock-Out                               : 15,682 SKUs | Revenue:    2,941,604 | Margin:   4.0%
Low Stock (High Margin)                 :    547 SKUs | Revenue:    2,166,363 | Margin:  56.5%
Loss-Making (Recent)                    :  2,421

In [7]:
# ============================================================================
# CELL 1: SAVE VARIABLES (ضعها في نهاية Notebook)
# ============================================================================

import pickle
from pathlib import Path
from datetime import datetime

print("=" * 80)
print("💾 SAVING NOTEBOOK VARIABLES")
print("=" * 80)

# ═══════════════════════════════════════════════════════════════════════════
# 📋 STEP 1: List variables to save (Edit this list as needed)
# ═══════════════════════════════════════════════════════════════════════════

variables_to_save = [
    # Core DataFrames
    'combined_df', #the main dataframe
    'combined_df_updated', # updated dataframe for pattern dashboard
    'df',

    # Analysis Results
    'results_df', # for skus classification
    'classification_df',
    'forecasts_df',
    'inventory_df',
    'results_df_classi_collection'  ,
    'sku_df',
    'section_df',#dataframe for collection classification

    # Dashboard Objects
    'dashboard',

    # Pattern Analysis
    'pattern_stats',
    'sku_level',
    'color_analysis',

    # Configs & Dicts
    'config',
    'metrics',

    # Add any other variables here
]

# ═══════════════════════════════════════════════════════════════════════════
# 📁 STEP 2: Create save folder
# ═══════════════════════════════════════════════════════════════════════════

save_folder = Path.home() / "Desktop" / "notebook_variables"
save_folder.mkdir(exist_ok=True)

# Create timestamped subfolder
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
session_folder = save_folder / f"session_{timestamp}"
session_folder.mkdir(exist_ok=True)

print(f"\n📂 Save location: {session_folder}")
print()

# ═══════════════════════════════════════════════════════════════════════════
# 💾 STEP 3: Save each variable
# ═══════════════════════════════════════════════════════════════════════════

saved_vars = {}
success_count = 0
skip_count = 0
fail_count = 0

for var_name in variables_to_save:
    if var_name in globals():
        try:
            var_value = globals()[var_name]

            # Save to pickle
            file_path = session_folder / f"{var_name}.pkl"
            with open(file_path, 'wb') as f:
                pickle.dump(var_value, f)

            # Get file size
            size_kb = file_path.stat().st_size / 1024
            size_str = f"{size_kb:.1f} KB" if size_kb < 1024 else f"{size_kb/1024:.1f} MB"

            saved_vars[var_name] = {
                'saved': True,
                'type': type(var_value).__name__,
                'size': size_str
            }

            print(f"✅ {var_name:<25} | {saved_vars[var_name]['type']:<15} | {size_str}")
            success_count += 1

        except Exception as e:
            saved_vars[var_name] = {'saved': False, 'error': str(e)}
            print(f"❌ {var_name:<25} | FAILED: {str(e)[:40]}")
            fail_count += 1
    else:
        print(f"⚠️  {var_name:<25} | NOT FOUND (skipped)")
        skip_count += 1

# ═══════════════════════════════════════════════════════════════════════════
# 📝 STEP 4: Save metadata
# ═══════════════════════════════════════════════════════════════════════════

metadata = {
    'timestamp': timestamp,
    'datetime': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'saved_variables': saved_vars,
    'notebook_name': 'Current Notebook',  # You can customize this
    'success_count': success_count,
    'skip_count': skip_count,
    'fail_count': fail_count
}

metadata_path = session_folder / "metadata.pkl"
with open(metadata_path, 'wb') as f:
    pickle.dump(metadata, f)

# ═══════════════════════════════════════════════════════════════════════════
# ✅ STEP 5: Summary
# ═══════════════════════════════════════════════════════════════════════════

print()
print("=" * 80)
print("✅ SAVE COMPLETE!")
print("=" * 80)
print(f"📂 Location: {session_folder}")
print(f"✅ Saved:    {success_count} variables")
print(f"⚠️  Skipped:  {skip_count} variables (not found)")
print(f"❌ Failed:   {fail_count} variables")
print("=" * 80)
print()
print("💡 To load these variables in another notebook, use CELL 2")
print("=" * 80)

💾 SAVING NOTEBOOK VARIABLES

📂 Save location: C:\Users\User\Desktop\notebook_variables\session_20260118_092509

✅ combined_df               | DataFrame       | 1454.8 MB
⚠️  combined_df_updated       | NOT FOUND (skipped)
⚠️  df                        | NOT FOUND (skipped)
⚠️  results_df                | NOT FOUND (skipped)
⚠️  classification_df         | NOT FOUND (skipped)
⚠️  forecasts_df              | NOT FOUND (skipped)
✅ inventory_df              | DataFrame       | 17.6 MB
⚠️  results_df_classi_collection | NOT FOUND (skipped)
✅ sku_df                    | DataFrame       | 15.0 MB
✅ section_df                | DataFrame       | 855.7 KB
⚠️  dashboard                 | NOT FOUND (skipped)
⚠️  pattern_stats             | NOT FOUND (skipped)
⚠️  sku_level                 | NOT FOUND (skipped)
⚠️  color_analysis            | NOT FOUND (skipped)
⚠️  config                    | NOT FOUND (skipped)
⚠️  metrics                   | NOT FOUND (skipped)

✅ SAVE COMPLETE!
📂 Location: C:\U

In [3]:
# ============================================================================
# ULTRA-ENHANCED SECTION & SKU ANALYSIS - PRODUCTION READY (FIXED)
# ============================================================================
# Features:
# 1. Dual Trend Analysis (Revenue + Quantity) with mismatch warnings
# 2. Year-over-Year (YoY) comparison + Seasonal flags
# 3. Advanced Stock-out logic with Lost Sales estimates
# 4. SKU Health Metrics (% Growing/Declining per section)
# 5. KEY_REASON field explaining status in plain language
# 6. Benchmark comparisons (section percentiles)
# ============================================================================

import pandas as pd
import numpy as np
from scipy.stats import kendalltau
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from tqdm.notebook import tqdm
import warnings
import gc
warnings.filterwarnings('ignore')


# ============================================================================
# DUAL TREND ANALYZER - Revenue + Quantity
# ============================================================================

class DualTrendAnalyzer:
    """
    Analyzes trends for BOTH Revenue and Quantity
    Detects mismatches (e.g., revenue growing but quantity declining)
    """

    @staticmethod
    def analyze_dual_trend(monthly_revenue, monthly_quantity):
        """
        Analyze trends for both metrics

        Returns:
        --------
        dict with:
            - revenue_trend, revenue_confidence
            - quantity_trend, quantity_confidence
            - trend_mismatch (bool)
            - mismatch_warning (str)
        """
        if len(monthly_revenue) < 6 or len(monthly_quantity) < 6:
            return {
                'revenue_trend': 'Unknown',
                'revenue_confidence': 'LOW',
                'revenue_period_change': 0,
                'quantity_trend': 'Unknown',
                'quantity_confidence': 'LOW',
                'quantity_period_change': 0,
                'trend_mismatch': False,
                'mismatch_warning': None
            }

        try:
            # Analyze Revenue Trend
            rev_direction, rev_confidence, rev_metrics = DualTrendAnalyzer._analyze_single_trend(monthly_revenue)

            # Analyze Quantity Trend
            qty_direction, qty_confidence, qty_metrics = DualTrendAnalyzer._analyze_single_trend(monthly_quantity)

            # Detect Mismatch
            trend_mismatch = False
            mismatch_warning = None

            if rev_direction != qty_direction and rev_confidence != 'LOW' and qty_confidence != 'LOW':
                trend_mismatch = True

                if rev_direction == 'Growing' and qty_direction == 'Declining':
                    mismatch_warning = "⚠️ Revenue UP but Quantity DOWN → Price increase or product mix shift"
                elif rev_direction == 'Declining' and qty_direction == 'Growing':
                    mismatch_warning = "⚠️ Revenue DOWN but Quantity UP → Discounting or lower-value products"
                elif rev_direction == 'Stable' and qty_direction != 'Stable':
                    mismatch_warning = f"⚠️ Revenue stable but Quantity {qty_direction}"
                elif qty_direction == 'Stable' and rev_direction != 'Stable':
                    mismatch_warning = f"⚠️ Quantity stable but Revenue {rev_direction}"

            return {
                'revenue_trend': rev_direction,
                'revenue_confidence': rev_confidence,
                'revenue_period_change': rev_metrics.get('period_change', 0),
                'quantity_trend': qty_direction,
                'quantity_confidence': qty_confidence,
                'quantity_period_change': qty_metrics.get('period_change', 0),
                'trend_mismatch': trend_mismatch,
                'mismatch_warning': mismatch_warning
            }

        except Exception as e:
            return {
                'revenue_trend': 'Unknown',
                'revenue_confidence': 'LOW',
                'revenue_period_change': 0,
                'quantity_trend': 'Unknown',
                'quantity_confidence': 'LOW',
                'quantity_period_change': 0,
                'trend_mismatch': False,
                'mismatch_warning': None
            }

    @staticmethod
    def _analyze_single_trend(monthly_values):
        """Single metric trend analysis"""

        values = np.array(monthly_values, dtype=np.float32)
        recent_months = min(12, len(values))
        recent_values = values[-recent_months:]

        # METHOD 1: Exponential Smoothing
        exp_trend = 0
        try:
            model = ExponentialSmoothing(values, trend='add', seasonal=None)
            fit = model.fit(smoothing_level=0.3, smoothing_trend=0.1, optimized=False)
            trend = fit.trend
            exp_trend = trend[-3:].mean() if len(trend) >= 3 else trend[-1]
        except:
            pass

        # METHOD 2: Mann-Kendall
        mk_tau, mk_p = 0, 1
        try:
            months = np.arange(len(recent_values))
            mk_tau, mk_p = kendalltau(months, recent_values)
        except:
            pass

        # METHOD 3: Period Comparison
        if len(values) >= 6:
            last_3 = values[-3:].mean()
            prev_3 = values[-6:-3].mean()
            period_change = ((last_3 - prev_3) / prev_3 * 100) if prev_3 > 0 else 0
        else:
            period_change = 0

        # METHOD 4: CAGR
        cagr = None
        if len(values) >= 12:
            beginning = values[-12:-9].mean()
            ending = values[-3:].mean()
            if beginning > 0 and ending > 0:
                years = 9 / 12
                cagr = ((ending / beginning) ** (1 / years) - 1) * 100

        # Consistency
        changes = np.diff(recent_values)
        consistency = 0
        if len(changes) > 0:
            positive = (changes > 0).sum()
            negative = (changes < 0).sum()
            consistency = max(positive, negative) / len(changes) * 100

        cv = recent_values.std() / recent_values.mean() if recent_values.mean() > 0 else 0

        # VOTING
        votes_declining = 0
        votes_growing = 0

        if exp_trend < -300:
            votes_declining += 1
        elif exp_trend > 300:
            votes_growing += 1

        if mk_tau < -0.15 and mk_p < 0.1:
            votes_declining += 1
        elif mk_tau > 0.15 and mk_p < 0.1:
            votes_growing += 1

        if period_change < -10:
            votes_declining += 1.5
        elif period_change > 10:
            votes_growing += 1.5

        if cagr is not None:
            if cagr < -8:
                votes_declining += 1
            elif cagr > 8:
                votes_growing += 1

        total_votes = 5.5 if cagr is not None else 4.5

        if votes_declining >= 2:
            direction = "Declining"
            confidence_score = votes_declining / total_votes
        elif votes_growing >= 2:
            direction = "Growing"
            confidence_score = votes_growing / total_votes
        else:
            direction = "Stable" if cv < 0.3 else ("Volatile" if cv > 0.7 else "Stable")
            confidence_score = 0.5

        if consistency > 70 and cv < 0.5:
            confidence_score *= 1.2
        elif consistency < 50 or cv > 0.8:
            confidence_score *= 0.8

        confidence_score = min(confidence_score, 1.0)
        confidence = "HIGH" if confidence_score >= 0.75 else ("MEDIUM" if confidence_score >= 0.5 else "LOW")

        metrics = {
            'exp_trend': float(exp_trend),
            'period_change': float(period_change),
            'cagr': float(cagr) if cagr is not None else None,
            'consistency': float(consistency),
            'cv': float(cv)
        }

        return direction, confidence, metrics


# ============================================================================
# YEAR-OVER-YEAR ANALYZER
# ============================================================================

class YoYAnalyzer:
    """
    Year-over-Year comparison and seasonality detection
    """

    @staticmethod
    def analyze_yoy(df, analysis_date):
        """
        Compare same months across years

        Returns:
        --------
        dict with:
            - yoy_growth_%
            - is_seasonal (bool)
            - seasonal_months (list)
            - yoy_comparison (dict)
        """
        try:
            df = df.copy()
            df['Year'] = pd.to_datetime(df['Date']).dt.year
            df['Month'] = pd.to_datetime(df['Date']).dt.month
            df['YearMonth'] = pd.to_datetime(df['Date']).dt.to_period('M')

            # Get current and previous year
            current_year = analysis_date.year
            prev_year = current_year - 1

            current_data = df[df['Year'] == current_year]
            prev_data = df[df['Year'] == prev_year]

            if len(current_data) == 0 or len(prev_data) == 0:
                return {
                    'yoy_growth_%': None,
                    'is_seasonal': False,
                    'seasonal_months': [],
                    'seasonality_cv': 0,
                    'yoy_by_month': {}
                }

            # Monthly comparison
            current_monthly = current_data.groupby('Month')['bal Value'].sum()
            prev_monthly = prev_data.groupby('Month')['bal Value'].sum()

            # Overall YoY growth
            current_total = current_data['bal Value'].sum()
            prev_total = prev_data['bal Value'].sum()

            yoy_growth = ((current_total - prev_total) / prev_total * 100) if prev_total > 0 else None

            # Seasonality detection
            monthly_revenue = df.groupby('Month')['bal Value'].sum()

            is_seasonal = False
            seasonal_months = []
            cv = 0

            if len(monthly_revenue) >= 12:
                # Calculate coefficient of variation for months
                monthly_avg = monthly_revenue.mean()
                monthly_std = monthly_revenue.std()
                cv = monthly_std / monthly_avg if monthly_avg > 0 else 0

                # If CV > 0.5, consider seasonal
                if cv > 0.5:
                    is_seasonal = True
                    # Find peak months (>120% of average)
                    threshold = monthly_avg * 1.2
                    seasonal_months = monthly_revenue[monthly_revenue > threshold].index.tolist()

            # Month-by-month comparison
            yoy_by_month = {}
            for month in range(1, 13):
                current_val = current_monthly.get(month, 0)
                prev_val = prev_monthly.get(month, 0)

                if prev_val > 0:
                    growth = ((current_val - prev_val) / prev_val * 100)
                    yoy_by_month[month] = {
                        'current': float(current_val),
                        'previous': float(prev_val),
                        'growth_%': float(growth)
                    }

            return {
                'yoy_growth_%': float(yoy_growth) if yoy_growth is not None else None,
                'is_seasonal': is_seasonal,
                'seasonal_months': seasonal_months,
                'seasonality_cv': float(cv),
                'yoy_by_month': yoy_by_month
            }

        except Exception as e:
            return {
                'yoy_growth_%': None,
                'is_seasonal': False,
                'seasonal_months': [],
                'seasonality_cv': 0,
                'yoy_by_month': {}
            }


# ============================================================================
# ADVANCED STOCK-OUT ANALYZER
# ============================================================================

class AdvancedStockOutAnalyzer:
    """
    Enhanced stock-out detection with sales rate analysis
    """

    @staticmethod
    def analyze_stockout(sku_data, current_stock, analysis_date, has_profit):
        """
        Advanced stock-out analysis

        Returns:
        --------
        dict with:
            - is_stockout
            - is_potential_stockout
            - stockout_date
            - days_out_of_stock
            - pre_stockout_sales_rate (units/day)
            - estimated_lost_sales (units)
            - estimated_lost_revenue
            - estimated_lost_profit
        """

        result = {
            'is_stockout': False,
            'is_potential_stockout': False,
            'stockout_date': None,
            'days_out_of_stock': 0,
            'pre_stockout_sales_rate_qty': 0,
            'pre_stockout_sales_rate_value': 0,
            'estimated_lost_sales_qty': 0,
            'estimated_lost_revenue': 0,
            'estimated_lost_profit': 0,
            'pre_stockout_avg_margin': 0
        }

        try:
            date_values = sku_data['Date'].values
            revenue_values = sku_data['bal Value'].values
            qty_values = sku_data['bal Qty'].values

            last_sale = pd.Timestamp(date_values.max())
            days_since_last = int((analysis_date - last_sale) / np.timedelta64(1, 'D'))

            # Check if stock-out
            if current_stock == 0 and days_since_last > 90:
                result['is_stockout'] = True

                # Find stockout date (gap > 60 days)
                dates_sorted = np.sort(date_values)
                stockout_date = last_sale

                if len(dates_sorted) > 1:
                    gaps = np.diff(dates_sorted).astype('timedelta64[D]').astype(int)
                    gap_idx = np.where(gaps > 60)[0]
                    if len(gap_idx) > 0:
                        stockout_date = dates_sorted[gap_idx[0]]

                result['stockout_date'] = stockout_date
                result['days_out_of_stock'] = (analysis_date - pd.Timestamp(stockout_date)).days

                # Analyze period BEFORE stockout (last 6 months before stockout)
                six_months_before = pd.Timestamp(stockout_date) - pd.Timedelta(days=180)
                before_mask = (date_values >= six_months_before) & (date_values < stockout_date)
                before_data = sku_data[before_mask]

                if len(before_data) > 0:
                    before_days = (pd.Timestamp(stockout_date) - six_months_before).days

                    before_qty = before_data['bal Qty'].sum()
                    before_revenue = before_data['bal Value'].sum()

                    # Sales rate (units/day and revenue/day)
                    sales_rate_qty = before_qty / before_days if before_days > 0 else 0
                    sales_rate_value = before_revenue / before_days if before_days > 0 else 0

                    result['pre_stockout_sales_rate_qty'] = float(sales_rate_qty)
                    result['pre_stockout_sales_rate_value'] = float(sales_rate_value)

                    # Estimated lost sales
                    days_out = result['days_out_of_stock']
                    estimated_lost_qty = sales_rate_qty * days_out
                    estimated_lost_revenue = sales_rate_value * days_out

                    result['estimated_lost_sales_qty'] = float(estimated_lost_qty)
                    result['estimated_lost_revenue'] = float(estimated_lost_revenue)

                    # Profit estimation
                    if has_profit and 'Total_Profit' in before_data.columns:
                        before_profit = before_data['Total_Profit'].sum()
                        avg_margin = (before_profit / before_revenue * 100) if before_revenue > 0 else 0

                        result['pre_stockout_avg_margin'] = float(avg_margin)
                        result['estimated_lost_profit'] = float(estimated_lost_revenue * (avg_margin / 100))

                    # Potential high-value stock-out
                    active_days = before_data['Date'].nunique()
                    avg_margin = result['pre_stockout_avg_margin']

                    if ((sales_rate_qty > 10 or sales_rate_value > 1000) and
                        avg_margin > 30 and
                        active_days >= 30):
                        result['is_potential_stockout'] = True

            return result

        except Exception as e:
            return result


# ============================================================================
# SECTION-LEVEL ANALYZER WITH ENHANCEMENTS
# ============================================================================

class EnhancedSectionAnalyzer:
    """
    Section-level analysis with:
    - Dual trend (revenue + quantity)
    - YoY comparison
    - SKU health metrics
    - KEY_REASON field
    """

    def __init__(self, df, analysis_date):
        self.df = df
        self.analysis_date = analysis_date

        self.has_profit = 'Total_Profit' in df.columns
        self.has_stock = 'CURRENT_STOCK' in df.columns
        self.has_sales_type = 'Sales_Type' in df.columns

    def analyze_section(self, section_name, section_data, sku_results_df):
        """
        Comprehensive section analysis
        """

        try:
            # Basic metrics
            total_revenue = section_data['bal Value'].sum()
            total_qty = section_data['bal Qty'].sum()
            total_profit = section_data['Total_Profit'].sum() if self.has_profit else 0
            num_skus = section_data['SKU_CLEAN'].nunique()
            num_transactions = len(section_data)

            first_sale = section_data['Date'].min()
            last_sale = section_data['Date'].max()
            days_since_last = (self.analysis_date - last_sale).days

            # Recent period (last 12 months)
            one_year_ago = self.analysis_date - pd.Timedelta(days=365)
            recent_data = section_data[section_data['Date'] >= one_year_ago]

            recent_revenue = recent_data['bal Value'].sum()
            recent_qty = recent_data['bal Qty'].sum()
            recent_profit = recent_data['Total_Profit'].sum() if self.has_profit else 0

            # Margins
            avg_margin = (total_profit / total_revenue * 100) if total_revenue > 0 else 0
            recent_avg_margin = (recent_profit / recent_revenue * 100) if recent_revenue > 0 else 0

            # Stock metrics
            if self.has_stock:
                latest_stock = section_data.groupby('SKU_CLEAN')['CURRENT_STOCK'].first()
                total_stock = latest_stock.sum()
                skus_out_of_stock = (latest_stock == 0).sum()
                skus_low_stock = ((latest_stock > 0) & (latest_stock < 10)).sum()
                stockout_rate = (skus_out_of_stock / num_skus * 100) if num_skus > 0 else 0
            else:
                total_stock = 0
                skus_out_of_stock = 0
                skus_low_stock = 0
                stockout_rate = 0

            # Monthly analysis
            section_data_copy = section_data.copy()
            section_data_copy['YearMonth'] = pd.to_datetime(section_data_copy['Date']).dt.to_period('M')

            monthly_rev = section_data_copy.groupby('YearMonth')['bal Value'].sum()
            monthly_qty = section_data_copy.groupby('YearMonth')['bal Qty'].sum()

            active_months = len(monthly_rev)
            avg_monthly_revenue = monthly_rev.mean() if active_months > 0 else 0
            median_monthly_revenue = monthly_rev.median() if active_months > 0 else 0

            avg_monthly_qty = monthly_qty.mean() if active_months > 0 else 0
            median_monthly_qty = monthly_qty.median() if active_months > 0 else 0

            # Dual trend analysis
            dual_trend = DualTrendAnalyzer.analyze_dual_trend(
                monthly_rev.values,
                monthly_qty.values
            )

            # YoY analysis
            yoy_results = YoYAnalyzer.analyze_yoy(section_data, self.analysis_date)

            # SKU health metrics
            section_skus = sku_results_df[sku_results_df['Section'] == section_name]

            if len(section_skus) > 0:
                growing_skus = len(section_skus[section_skus['Revenue_Trend'] == 'Growing'])
                declining_skus = len(section_skus[section_skus['Revenue_Trend'] == 'Declining'])
                stable_skus = len(section_skus[section_skus['Revenue_Trend'] == 'Stable'])

                pct_growing = (growing_skus / len(section_skus) * 100)
                pct_declining = (declining_skus / len(section_skus) * 100)
                pct_stable = (stable_skus / len(section_skus) * 100)

                high_conf_growing = len(section_skus[
                    (section_skus['Revenue_Trend'] == 'Growing') &
                    (section_skus['Revenue_Confidence'] == 'HIGH')
                ])
                high_conf_declining = len(section_skus[
                    (section_skus['Revenue_Trend'] == 'Declining') &
                    (section_skus['Revenue_Confidence'] == 'HIGH')
                ])

                potential_stockouts = len(section_skus[section_skus['Is_Potential_Stockout'] == True])

            else:
                growing_skus = declining_skus = stable_skus = 0
                pct_growing = pct_declining = pct_stable = 0
                high_conf_growing = high_conf_declining = 0
                potential_stockouts = 0

            # Status determination
            status, key_reason = self._determine_status_with_reason(
                revenue_trend=dual_trend['revenue_trend'],
                quantity_trend=dual_trend['quantity_trend'],
                trend_mismatch=dual_trend['trend_mismatch'],
                recent_avg_margin=recent_avg_margin,
                stockout_rate=stockout_rate,
                days_since_last=days_since_last,
                pct_declining=pct_declining,
                pct_growing=pct_growing,
                yoy_growth=yoy_results['yoy_growth_%'],
                is_seasonal=yoy_results['is_seasonal']
            )

            return {
                'Section': section_name,
                'Status': status,
                'Key_Reason': key_reason,

                # SKU counts
                'Total_SKUs': num_skus,
                'SKUs_Out_of_Stock': skus_out_of_stock,
                'SKUs_Low_Stock': skus_low_stock,
                'Stock_Out_Rate_%': float(stockout_rate),

                # SKU health metrics
                'SKUs_Growing': growing_skus,
                'SKUs_Declining': declining_skus,
                'SKUs_Stable': stable_skus,
                'Pct_Growing': float(pct_growing),
                'Pct_Declining': float(pct_declining),
                'Pct_Stable': float(pct_stable),
                'High_Conf_Growing': high_conf_growing,
                'High_Conf_Declining': high_conf_declining,
                'Potential_Stockouts': potential_stockouts,

                # Revenue
                'Total_Revenue': float(total_revenue),
                'Recent_Revenue_12M': float(recent_revenue),
                'Revenue_Per_SKU': float(total_revenue / num_skus) if num_skus > 0 else 0,

                # Quantity
                'Total_Quantity': float(total_qty),
                'Recent_Quantity_12M': float(recent_qty),

                # Transactions
                'Num_Transactions': num_transactions,

                # Monthly metrics
                'Avg_Monthly_Revenue': float(avg_monthly_revenue),
                'Median_Monthly_Revenue': float(median_monthly_revenue),
                'Avg_Monthly_Quantity': float(avg_monthly_qty),
                'Median_Monthly_Quantity': float(median_monthly_qty),
                'Active_Months': active_months,

                # Profit
                'Total_Profit': float(total_profit),
                'Recent_Profit_12M': float(recent_profit),
                'Avg_Margin_%': float(avg_margin),
                'Recent_Avg_Margin_%': float(recent_avg_margin),

                # Stock
                'Total_Stock': float(total_stock),

                # Time
                'First_Sale': first_sale,
                'Last_Sale': last_sale,
                'Days_Since_Last_Sale': days_since_last,

                # Dual trend
                'Revenue_Trend': dual_trend['revenue_trend'],
                'Revenue_Confidence': dual_trend['revenue_confidence'],
                'Revenue_Period_Change_%': float(dual_trend['revenue_period_change']),
                'Quantity_Trend': dual_trend['quantity_trend'],
                'Quantity_Confidence': dual_trend['quantity_confidence'],
                'Quantity_Period_Change_%': float(dual_trend['quantity_period_change']),
                'Trend_Mismatch': dual_trend['trend_mismatch'],
                'Mismatch_Warning': dual_trend['mismatch_warning'],

                # YoY
                'YoY_Growth_%': yoy_results['yoy_growth_%'],
                'Is_Seasonal': yoy_results['is_seasonal'],
                'Seasonal_Months': str(yoy_results.get('seasonal_months', [])),
                'Seasonality_CV': float(yoy_results.get('seasonality_cv', 0))
            }

        except Exception as e:
            print(f"⚠️ Error analyzing section {section_name}: {e}")
            import traceback
            traceback.print_exc()
            return None

    def _determine_status_with_reason(self, revenue_trend, quantity_trend, trend_mismatch,
                                      recent_avg_margin, stockout_rate, days_since_last,
                                      pct_declining, pct_growing, yoy_growth, is_seasonal):
        """
        Determine status with KEY_REASON explanation
        """

        reasons = []

        # Critical issues first
        if recent_avg_margin < 0:
            status = "Loss-Making"
            reasons.append(f"Margin negative ({recent_avg_margin:.1f}%)")

        elif stockout_rate > 30:
            status = "High Stock-Out Rate"
            reasons.append(f"{stockout_rate:.0f}% SKUs out of stock")

        elif days_since_last > 180:
            status = "Inactive"
            reasons.append(f"{days_since_last} days since last sale")

        elif revenue_trend == "Declining" and recent_avg_margin < 15:
            status = "Declining (Low Margin)"
            reasons.append(f"Revenue declining with {recent_avg_margin:.1f}% margin")
            if pct_declining > 50:
                reasons.append(f"{pct_declining:.0f}% SKUs declining")

        elif revenue_trend == "Declining":
            status = "Declining"
            reasons.append("Revenue trending down")
            if quantity_trend == "Declining":
                reasons.append("Quantity also declining")
            if pct_declining > 40:
                reasons.append(f"{pct_declining:.0f}% SKUs declining")
            if yoy_growth and yoy_growth < -10:
                reasons.append(f"YoY down {abs(yoy_growth):.1f}%")

        elif revenue_trend == "Growing" and recent_avg_margin > 30:
            status = "Growing (High Margin)"
            reasons.append(f"Revenue up with {recent_avg_margin:.1f}% margin")
            if pct_growing > 50:
                reasons.append(f"{pct_growing:.0f}% SKUs growing")
            if yoy_growth and yoy_growth > 10:
                reasons.append(f"YoY up {yoy_growth:.1f}%")

        elif revenue_trend == "Growing":
            status = "Growing"
            reasons.append("Revenue trending up")
            if quantity_trend == "Growing":
                reasons.append("Quantity also growing")
            if pct_growing > 40:
                reasons.append(f"{pct_growing:.0f}% SKUs growing")

        elif revenue_trend == "Volatile":
            status = "Volatile"
            reasons.append("Unstable revenue pattern")
            if is_seasonal:
                reasons.append("Seasonal product")

        elif revenue_trend == "Stable" and recent_avg_margin > 25:
            status = "Stable (Profitable)"
            reasons.append(f"Steady with {recent_avg_margin:.1f}% margin")

        elif revenue_trend == "Stable":
            status = "Stable"
            reasons.append("Consistent revenue")

        else:
            status = "Active"
            reasons.append("Currently active")

        # Add trend mismatch warning
        if trend_mismatch:
            if revenue_trend == "Growing" and quantity_trend == "Declining":
                reasons.append("⚠️ Price increase or mix shift")
            elif revenue_trend == "Declining" and quantity_trend == "Growing":
                reasons.append("⚠️ Discounting or lower-value products")

        # Seasonal flag
        if is_seasonal and status not in ["Inactive", "Loss-Making"]:
            reasons.append("🗓️ Seasonal")

        key_reason = " | ".join(reasons) if reasons else "No specific issues"

        return status, key_reason


# ============================================================================
# ENHANCED SKU ANALYZER
# ============================================================================

class UltraEnhancedSKUAnalyzer:
    """
    Ultimate SKU analyzer with all enhancements
    """

    def __init__(self, df, analysis_date=None):
        print("🔧 Pre-processing data...")

        df['SKU_CLEAN'] = df['SKU'].astype(str).str.strip()

        if not pd.api.types.is_datetime64_any_dtype(df['Date']):
            df['Date'] = pd.to_datetime(df['Date'])

        self.df = df
        self.analysis_date = analysis_date or df['Date'].max()

        self.has_profit = 'Total_Profit' in df.columns
        self.has_stock = 'CURRENT_STOCK' in df.columns
        self.has_sales_type = 'Sales_Type' in df.columns
        self.has_section = 'Section' in df.columns

        if not self.has_section:
            print("⚠️  WARNING: No 'Section' column - section analysis will be skipped")

        print("✅ Pre-processing complete!")

    def analyze_single_sku(self, sku_data, sku_name):
        """Enhanced SKU analysis with dual trends and advanced stock-out"""

        if len(sku_data) == 0:
            return None

        try:
            section = sku_data['Section'].iloc[0] if self.has_section else 'Unknown'

            # Basic extraction
            revenue_values = sku_data['bal Value'].values
            qty_values = sku_data['bal Qty'].values
            date_values = sku_data['Date'].values

            total_revenue = revenue_values.sum()
            total_qty = qty_values.sum()
            first_sale = pd.Timestamp(date_values.min())
            last_sale = pd.Timestamp(date_values.max())
            analysis_date = pd.Timestamp(self.analysis_date)

            days_since_last = int((analysis_date - last_sale) / np.timedelta64(1, 'D'))
            num_transactions = len(sku_data)

            # Recent period
            one_year_ago = analysis_date - pd.Timedelta(days=365)
            recent_mask = date_values >= one_year_ago

            recent_revenue = revenue_values[recent_mask].sum()
            recent_qty = qty_values[recent_mask].sum()

            # Profit
            total_profit, avg_margin, recent_profit, recent_avg_margin = 0, 0, 0, 0

            if self.has_profit:
                profit_values = sku_data['Total_Profit'].values
                total_profit = profit_values.sum()
                avg_margin = (total_profit / total_revenue * 100) if total_revenue > 0 else 0

                recent_profit_values = profit_values[recent_mask]
                recent_revenue_values = revenue_values[recent_mask]
                recent_profit = recent_profit_values.sum()
                recent_avg_margin = (recent_profit / recent_revenue * 100) if recent_revenue > 0 else 0

            # Stock
            current_stock = 0
            if self.has_stock:
                stock_values = sku_data['CURRENT_STOCK'].dropna()
                if len(stock_values) > 0:
                    current_stock = stock_values.iloc[0]

            # Monthly analysis
            dates_pd = pd.to_datetime(date_values)
            year_month = dates_pd.to_period('M')

            monthly_df = pd.DataFrame({
                'YearMonth': year_month,
                'Revenue': revenue_values,
                'Quantity': qty_values
            })

            monthly_agg = monthly_df.groupby('YearMonth', observed=True).agg({
                'Revenue': 'sum',
                'Quantity': 'sum'
            })

            active_months = len(monthly_agg)
            monthly_revenue = monthly_agg['Revenue'].values.astype(np.float32)
            monthly_quantity = monthly_agg['Quantity'].values.astype(np.float32)

            avg_monthly_revenue = monthly_revenue.mean() if active_months > 0 else 0

            # Dual trend analysis
            dual_trend = DualTrendAnalyzer.analyze_dual_trend(monthly_revenue, monthly_quantity)

            # YoY analysis
            yoy_results = YoYAnalyzer.analyze_yoy(sku_data, self.analysis_date)

            # Advanced stock-out analysis
            stockout_results = AdvancedStockOutAnalyzer.analyze_stockout(
                sku_data, current_stock, self.analysis_date, self.has_profit
            )

            # Status
            status, key_reason = self._determine_sku_status_with_reason(
                revenue_trend=dual_trend['revenue_trend'],
                recent_avg_margin=recent_avg_margin,
                is_potential_stockout=stockout_results['is_potential_stockout'],
                is_stockout=stockout_results['is_stockout'],
                current_stock=current_stock,
                days_since_last=days_since_last,
                trend_mismatch=dual_trend['trend_mismatch'],
                yoy_growth=yoy_results['yoy_growth_%']
            )

            priority = self._get_priority(status)

            return {
                'SKU': sku_name,
                'Section': section,
                'Status': status,
                'Key_Reason': key_reason,
                'Priority': priority,

                # Revenue
                'Total_Revenue': float(total_revenue),
                'Recent_Revenue_12M': float(recent_revenue),

                # Quantity
                'Total_Quantity': float(total_qty),
                'Recent_Quantity_12M': float(recent_qty),

                # Transactions
                'Num_Transactions': int(num_transactions),

                # Monthly
                'Avg_Monthly_Revenue': float(avg_monthly_revenue),
                'Active_Months': int(active_months),

                # Profit
                'Total_Profit': float(total_profit),
                'Recent_Profit_12M': float(recent_profit),
                'Avg_Margin_%': float(avg_margin),
                'Recent_Avg_Margin_%': float(recent_avg_margin),

                # Stock
                'Current_Stock': float(current_stock),

                # Time
                'First_Sale': first_sale,
                'Last_Sale': last_sale,
                'Days_Since_Last_Sale': int(days_since_last),

                # Dual trend
                'Revenue_Trend': dual_trend['revenue_trend'],
                'Revenue_Confidence': dual_trend['revenue_confidence'],
                'Quantity_Trend': dual_trend['quantity_trend'],
                'Quantity_Confidence': dual_trend['quantity_confidence'],
                'Trend_Mismatch': dual_trend['trend_mismatch'],
                'Mismatch_Warning': dual_trend['mismatch_warning'],

                # YoY
                'YoY_Growth_%': yoy_results['yoy_growth_%'],
                'Is_Seasonal': yoy_results['is_seasonal'],

                # Advanced stock-out
                'Is_Stockout': stockout_results['is_stockout'],
                'Is_Potential_Stockout': stockout_results['is_potential_stockout'],
                'Days_Out_of_Stock': stockout_results['days_out_of_stock'],
                'Pre_Stockout_Sales_Rate_Qty': float(stockout_results['pre_stockout_sales_rate_qty']),
                'Pre_Stockout_Sales_Rate_Value': float(stockout_results['pre_stockout_sales_rate_value']),
                'Estimated_Lost_Sales_Qty': float(stockout_results['estimated_lost_sales_qty']),
                'Estimated_Lost_Revenue': float(stockout_results['estimated_lost_revenue']),
                'Estimated_Lost_Profit': float(stockout_results['estimated_lost_profit'])
            }

        except Exception as e:
            print(f"⚠️ Error analyzing SKU {sku_name}: {e}")
            import traceback
            traceback.print_exc()
            return None

    def _determine_sku_status_with_reason(self, revenue_trend, recent_avg_margin,
                                          is_potential_stockout, is_stockout, current_stock,
                                          days_since_last, trend_mismatch, yoy_growth):
        """Determine SKU status with reason"""

        reasons = []

        if is_potential_stockout:
            status = "Potential Stock-Out (High Value)"
            reasons.append("High sales rate + good margin + no stock")

        elif recent_avg_margin < -5:
            status = "Loss-Making (Recent)"
            reasons.append(f"Recent margin {recent_avg_margin:.1f}%")

        elif is_stockout:
            status = "Stock-Out"
            reasons.append(f"{days_since_last} days no stock")

        elif 0 < current_stock < 10:
            status = "Low Stock"
            reasons.append(f"Only {current_stock:.0f} units left")
            if recent_avg_margin > 30:
                status = "Low Stock (High Margin)"
                reasons.append(f"Margin {recent_avg_margin:.1f}%")

        elif days_since_last > 365:
            status = "Dead"
            reasons.append("No sales >1 year")

        elif days_since_last > 180:
            status = "Inactive"
            reasons.append(f"{days_since_last} days since sale")

        elif revenue_trend == "Declining":
            status = "Declining"
            reasons.append("Revenue trending down")
            if yoy_growth and yoy_growth < -10:
                reasons.append(f"YoY {yoy_growth:.1f}%")

        elif revenue_trend == "Growing":
            status = "Growing"
            reasons.append("Revenue trending up")
            if recent_avg_margin > 30:
                status = "Growing (High Margin)"
                reasons.append(f"Margin {recent_avg_margin:.1f}%")
            if yoy_growth and yoy_growth > 10:
                reasons.append(f"YoY +{yoy_growth:.1f}%")

        else:
            status = "Stable"
            reasons.append("Steady performance")

        if trend_mismatch:
            reasons.append("⚠️ Revenue/Qty mismatch")

        key_reason = " | ".join(reasons) if reasons else "Active"

        return status, key_reason

    def _get_priority(self, status):
        """Assign priority based on status"""
        priority_map = {
            "Potential Stock-Out (High Value)": 0,
            "Loss-Making (Recent)": 1,
            "Stock-Out": 2,
            "Low Stock (High Margin)": 3,
            "Low Stock": 4,
            "Declining": 5,
            "Dead": 6,
            "Inactive": 7,
            "Growing (High Margin)": 10,
            "Growing": 11,
            "Stable": 12
        }
        return priority_map.get(status, 99)

    def analyze_all(self, save_to_excel=True, output_file='ULTRA_ENHANCED_ANALYSIS.xlsx'):
        """Analyze both SKUs and Sections - FIXED VERSION"""

        print("=" * 120)
        print("🚀 ULTRA-ENHANCED SECTION & SKU ANALYSIS - STARTING")
        print("=" * 120)
        print(f"📅 Analysis Date: {self.analysis_date.strftime('%Y-%m-%d')}")

        all_skus = self.df['SKU_CLEAN'].unique()
        total_skus = len(all_skus)
        print(f"📦 Total SKUs: {total_skus:,}")

        # ✅ CHECK SECTION COLUMN
        if self.has_section:
            sections = self.df['Section'].dropna().unique()
            print(f"📂 Total Sections: {len(sections):,}")
            if len(sections) == 0:
                print("⚠️ WARNING: 'Section' column exists but has NO valid values!")
                self.has_section = False
        else:
            print("⚠️ WARNING: No 'Section' column found - Section analysis will be SKIPPED")

        print("\n⭐ NEW FEATURES:")
        print("   • Dual Trend (Revenue + Quantity)")
        print("   • Year-over-Year comparison")
        print("   • Advanced Stock-out with sales rate")
        print("   • SKU Health metrics per section")
        print("   • KEY_REASON explanations")
        print()

        # ════════════════════════════════════════════════════════════════
        # PART 1: SKU ANALYSIS
        # ════════════════════════════════════════════════════════════════

        print("🔄 Analyzing SKUs...")
        print("-" * 120)

        sku_results = []

        for i, sku in enumerate(tqdm(all_skus, desc="📊 SKU Analysis", ncols=100, colour='cyan')):
            sku_data = self.df[self.df['SKU_CLEAN'] == sku]

            if len(sku_data) > 0:
                result = self.analyze_single_sku(sku_data, sku)
                if result:
                    sku_results.append(result)

            if (i + 1) % 1000 == 0:
                gc.collect()

        sku_df = pd.DataFrame(sku_results)

        # ✅ CHECK IF SKU ANALYSIS SUCCEEDED
        if len(sku_df) == 0:
            print("❌ ERROR: No SKUs were analyzed successfully!")
            return None, None

        sku_df = sku_df.sort_values(['Priority', 'Recent_Revenue_12M'], ascending=[True, False])
        print(f"\n✅ SKU Analysis Complete: {len(sku_df):,} SKUs")

        # ════════════════════════════════════════════════════════════════
        # PART 2: SECTION ANALYSIS
        # ════════════════════════════════════════════════════════════════

        section_df = None

        if self.has_section:
            print("\n" + "=" * 120)
            print("🔄 Analyzing Sections...")
            print("-" * 120)

            section_analyzer = EnhancedSectionAnalyzer(self.df, self.analysis_date)
            section_results = []

            sections = self.df['Section'].dropna().unique()

            for section in tqdm(sections, desc="📂 Section Analysis", ncols=100, colour='green'):
                if pd.isna(section):
                    continue

                section_data = self.df[self.df['Section'] == section]

                if len(section_data) > 0:
                    try:
                        result = section_analyzer.analyze_section(section, section_data, sku_df)
                        if result:
                            section_results.append(result)
                    except Exception as e:
                        print(f"\n⚠️ Failed to analyze section '{section}': {e}")
                        import traceback
                        traceback.print_exc()
                        continue

            # ✅ CHECK IF SECTION ANALYSIS SUCCEEDED
            if len(section_results) == 0:
                print("\n⚠️ WARNING: No sections were analyzed successfully!")
                print("   All sections may have failed during analysis.")
                section_df = None
            else:
                section_df = pd.DataFrame(section_results)

                # ✅ CHECK IF REQUIRED COLUMNS EXIST
                if 'Recent_Revenue_12M' in section_df.columns:
                    section_df = section_df.sort_values('Recent_Revenue_12M', ascending=False)
                    print(f"\n✅ Section Analysis Complete: {len(section_df):,} Sections")
                else:
                    print("\n⚠️ WARNING: Section DataFrame is missing expected columns!")
                    print(f"   Available columns: {section_df.columns.tolist()}")
                    section_df = None

        # ════════════════════════════════════════════════════════════════
        # SUMMARY
        # ════════════════════════════════════════════════════════════════

        print("\n" + "=" * 120)
        self._print_summary(sku_df, section_df)

        # ════════════════════════════════════════════════════════════════
        # SAVE TO EXCEL
        # ════════════════════════════════════════════════════════════════

        if save_to_excel:
            self._save_to_excel(sku_df, section_df, output_file)

        print("\n" + "=" * 120)
        print("🎉 ULTRA-ENHANCED ANALYSIS COMPLETE!")
        print("=" * 120)

        return sku_df, section_df

    def _print_summary(self, sku_df, section_df):
        """Print comprehensive summary"""

        print("📊 SKU SUMMARY:")
        print("-" * 120)

        # Status distribution
        print("\nStatus Distribution:")
        status_counts = sku_df['Status'].value_counts()
        for status, count in status_counts.head(10).items():
            print(f"  {status}: {count:,}")

        # Trend mismatch
        mismatch = sku_df[sku_df['Trend_Mismatch'] == True]
        if len(mismatch) > 0:
            print(f"\n⚠️  {len(mismatch):,} SKUs with Revenue/Quantity trend mismatch")

        # Potential stock-outs
        potential = sku_df[sku_df['Is_Potential_Stockout'] == True]
        if len(potential) > 0:
            print(f"🚨 {len(potential):,} Potential HIGH-VALUE stock-outs")
            print(f"   Estimated lost revenue: {potential['Estimated_Lost_Revenue'].sum():,.0f} SAR")

        # YoY
        yoy_available = sku_df[sku_df['YoY_Growth_%'].notna()]
        if len(yoy_available) > 0:
            avg_yoy = yoy_available['YoY_Growth_%'].mean()
            print(f"📈 Average YoY growth: {avg_yoy:.1f}%")

        if section_df is not None and len(section_df) > 0:
            print("\n" + "=" * 120)
            print("📂 SECTION SUMMARY:")
            print("-" * 120)

            # Status distribution
            print("\nStatus Distribution:")
            sec_status_counts = section_df['Status'].value_counts()
            for status, count in sec_status_counts.head(10).items():
                print(f"  {status}: {count:,}")

            # Trend mismatches
            sec_mismatch = section_df[section_df['Trend_Mismatch'] == True]
            if len(sec_mismatch) > 0:
                print(f"\n⚠️  {len(sec_mismatch):,} Sections with Revenue/Quantity mismatch")

            # Seasonal
            seasonal = section_df[section_df['Is_Seasonal'] == True]
            if len(seasonal) > 0:
                print(f"🗓️  {len(seasonal):,} Seasonal sections detected")

            # Top sections with KEY_REASON
            print("\n🔝 Top 10 Sections:")
            print(f"{'Section':<30} {'Revenue 12M':>15} {'Margin':>8} {'Status':<30} {'Key Reason'}")
            print("-" * 120)

            for _, row in section_df.head(10).iterrows():
                section_str = str(row['Section'])[:30]
                revenue = row['Recent_Revenue_12M']
                margin = row['Recent_Avg_Margin_%']
                status = str(row['Status'])[:30]
                reason = str(row['Key_Reason'])[:50]

                print(f"{section_str:<30} {revenue:>15,.0f} {margin:>7.1f}% {status:<30} {reason}")

    def _save_to_excel(self, sku_df, section_df, output_file):
        """Save to Excel with all sheets"""

        print(f"\n💾 Saving to: {output_file}")

        try:
            with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

                # SKU sheets
                sku_df.to_excel(writer, sheet_name='All SKUs', index=False)

                # Trend mismatches
                mismatch = sku_df[sku_df['Trend_Mismatch'] == True]
                if len(mismatch) > 0:
                    mismatch.to_excel(writer, sheet_name='⚠️ Trend Mismatches', index=False)

                # Potential stock-outs
                potential = sku_df[sku_df['Is_Potential_Stockout'] == True]
                if len(potential) > 0:
                    potential.sort_values('Estimated_Lost_Revenue', ascending=False).to_excel(
                        writer, sheet_name='🚨 Potential Stock-Outs', index=False)

                # Critical
                critical = sku_df[sku_df['Priority'] <= 5]
                if len(critical) > 0:
                    critical.to_excel(writer, sheet_name='Critical SKUs', index=False)

                # Section sheets
                if section_df is not None and len(section_df) > 0:
                    section_df.to_excel(writer, sheet_name='All Sections', index=False)

                    # Top sections
                    if len(section_df) >= 30:
                        top = section_df.nlargest(30, 'Recent_Revenue_12M')
                        top.to_excel(writer, sheet_name='Top 30 Sections', index=False)

                    # Sections with issues
                    issues = section_df[section_df['Status'].str.contains('Loss|Declining|High Stock', na=False)]
                    if len(issues) > 0:
                        issues.to_excel(writer, sheet_name='⚠️ Problem Sections', index=False)

                    # Seasonal sections
                    seasonal = section_df[section_df['Is_Seasonal'] == True]
                    if len(seasonal) > 0:
                        seasonal.to_excel(writer, sheet_name='🗓️ Seasonal Sections', index=False)

            print(f"✅ Saved successfully!")

        except Exception as e:
            print(f"❌ Error saving Excel file: {e}")
            import traceback
            traceback.print_exc()


# ============================================================================
# MAIN FUNCTION
# ============================================================================

def ultra_enhanced_analysis(df, save_to_excel=True, output_file='ULTRA_ENHANCED_ANALYSIS.xlsx'):
    """
    Ultimate Section & SKU analysis with all enhancements

    Features:
    ---------
    • Dual Trend (Revenue + Quantity) with mismatch detection
    • Year-over-Year comparison + Seasonal flags
    • Advanced Stock-out analysis with sales rate & lost sales
    • SKU Health metrics (% Growing/Declining per section)
    • KEY_REASON field explaining status
    • Benchmark comparisons

    Returns:
    --------
    sku_df, section_df : DataFrames

    Example:
    --------
    >>> sku_df, section_df = ultra_enhanced_analysis(combined_df)
    """
    analyzer = UltraEnhancedSKUAnalyzer(df)
    return analyzer.analyze_all(save_to_excel, output_file)


print("=" * 120)
print("✅ ULTRA-ENHANCED SECTION & SKU ANALYSIS LOADED! (FIXED VERSION)")
print("=" * 120)
print("\n🚀 Usage:")
print("   sku_df, section_df = ultra_enhanced_analysis(combined_df)")
print("\n⭐ NEW FEATURES:")
print("   1. ✅ Dual Trend (Revenue + Quantity) with mismatch warnings")
print("   2. ✅ Year-over-Year comparison + Seasonal detection")
print("   3. ✅ Advanced Stock-out with sales rate & lost sales estimate")
print("   4. ✅ SKU Health metrics (% Growing/Declining per section)")
print("   5. ✅ KEY_REASON field explaining status in plain language")
print("   6. ✅ Trend mismatch detection & warnings")
print("   7. ✅ FIXED: Robust error handling for sections")
print("=" * 120)

✅ ULTRA-ENHANCED SECTION & SKU ANALYSIS LOADED! (FIXED VERSION)

🚀 Usage:
   sku_df, section_df = ultra_enhanced_analysis(combined_df)

⭐ NEW FEATURES:
   1. ✅ Dual Trend (Revenue + Quantity) with mismatch warnings
   2. ✅ Year-over-Year comparison + Seasonal detection
   3. ✅ Advanced Stock-out with sales rate & lost sales estimate
   4. ✅ SKU Health metrics (% Growing/Declining per section)
   5. ✅ KEY_REASON field explaining status in plain language
   6. ✅ Trend mismatch detection & warnings
   7. ✅ FIXED: Robust error handling for sections


In [1]:
import  pandas as pd
combined_df = pd.read_parquet(r'C:\Users\User\PycharmProjects\JupyterProject\MD\Output\combined_df.parquet')

In [4]:
sku_df, section_df = ultra_enhanced_analysis(combined_df)

🔧 Pre-processing data...
✅ Pre-processing complete!
🚀 ULTRA-ENHANCED SECTION & SKU ANALYSIS - STARTING
📅 Analysis Date: 2026-02-02
📦 Total SKUs: 63,437
📂 Total Sections: 2,565

⭐ NEW FEATURES:
   • Dual Trend (Revenue + Quantity)
   • Year-over-Year comparison
   • Advanced Stock-out with sales rate
   • SKU Health metrics per section
   • KEY_REASON explanations

🔄 Analyzing SKUs...
------------------------------------------------------------------------------------------------------------------------


📊 SKU Analysis:   0%|                                                    | 0/63437 [00:00<?, ?it/s]

⚠️ Error analyzing SKU <NA>: cannot convert float NaN to integer


Traceback (most recent call last):
  File "C:\Users\User\AppData\Local\Temp\ipykernel_5476\1727007036.py", line 778, in analyze_single_sku
    days_since_last = int((analysis_date - last_sale) / np.timedelta64(1, 'D'))
ValueError: cannot convert float NaN to integer



✅ SKU Analysis Complete: 63,436 SKUs

🔄 Analyzing Sections...
------------------------------------------------------------------------------------------------------------------------


📂 Section Analysis:   0%|                                                 | 0/2565 [00:00<?, ?it/s]


✅ Section Analysis Complete: 2,565 Sections

📊 SKU SUMMARY:
------------------------------------------------------------------------------------------------------------------------

Status Distribution:
  Dead: 19,845
  Stable: 11,009
  Stock-Out: 9,720
  Declining: 7,481
  Inactive: 5,384
  Growing (High Margin): 4,772
  Low Stock: 2,172
  Loss-Making (Recent): 2,027
  Low Stock (High Margin): 608
  Growing: 412

⚠️  936 SKUs with Revenue/Quantity trend mismatch
🚨 6 Potential HIGH-VALUE stock-outs
   Estimated lost revenue: 3,263,705 SAR
📈 Average YoY growth: 78227.1%

📂 SECTION SUMMARY:
------------------------------------------------------------------------------------------------------------------------

Status Distribution:
  Declining: 537
  High Stock-Out Rate: 534
  Inactive: 395
  Volatile: 303
  Growing (High Margin): 284
  Loss-Making: 243
  Stable (Profitable): 137
  Active: 63
  Growing: 47
  Declining (Low Margin): 16

⚠️  40 Sections with Revenue/Quantity mismatch
🗓️  3

In [2]:
# ✅ تأكد من وجود العمود Section
print("Columns in combined_df:", combined_df.columns.tolist())
print("\n'Section' column exists?", 'Section' in combined_df.columns)

if 'Section' in combined_df.columns:
    print("\nUnique Sections:", combined_df['Section'].nunique())
    print("Sample Sections:", combined_df['Section'].dropna().unique()[:10])
    print("NaN Sections:", combined_df['Section'].isna().sum())
else:
    print("⚠️ 'Section' column is MISSING!")

Columns in combined_df: ['Section', 'Section Name', 'bal Value', 'bal Qty', 'U Price', 'Client', 'Client Name', 'Outlet', 'Outlet Name', 'Date', 'درجة أهمية العميل ', 'SKU', 'Catalog No.', 'Year', 'Cost_2019.0', 'Profit_2019.0', 'Margin_%_2019.0', 'Cost_2020.0', 'Profit_2020.0', 'Margin_%_2020.0', 'Cost_2021.0', 'Profit_2021.0', 'Margin_%_2021.0', 'Cost_2022.0', 'Profit_2022.0', 'Margin_%_2022.0', 'Cost_2023.0', 'Profit_2023.0', 'Margin_%_2023.0', 'Cost_2024.0', 'Profit_2024.0', 'Margin_%_2024.0', 'Cost_2025.0', 'Profit_2025.0', 'Margin_%_2025.0', 'Cost_2026.0', 'Profit_2026.0', 'Margin_%_2026.0', 'Cost_nan', 'Profit_nan', 'Margin_%_nan', 'Avg_Profit', 'Avg_Margin_%', 'Total_Cost', 'Total_Profit', 'Profit_Margin_%', 'CURRENT_STOCK', 'OUTSTANDING', 'PR', 'EFFECTIVE_STOCK', 'CATEGORY', 'Supplier', 'LAST_ENTRY_DATE', 'Nb. Days (Avail. Balance)', 'SKU-STATUES-Mahmoud', 'SKU STATUES -Python', 'OLD_QTY', 'ARTPATERN', 'TEXTURE', 'SUB_CATEGORY', 'First_Inv_Year', 'Outlet_Type', 'Area_Master_Ca

In [ ]:
# ============================================================================
# ULTIMATE FABRIC & APPAREL ANALYZER - PRODUCTION READY (COMPLETE)
# ============================================================================
#
# ✅ ROBUST STATISTICS - Log-normal & High Volatility Optimized
# ✅ SEASONAL DECOMPOSITION - True YoY comparison for fashion
# ✅ SLOW MOVER DETECTION - Differentiate from stock-outs
# ✅ INVENTORY TURNOVER - Speed metrics for fabric industry
# ✅ ABC ANALYSIS - Top-heavy risk detection
# ✅ DUAL TREND - Revenue + Quantity with mismatch warnings
# ✅ ADVANCED STOCK-OUT - Sales rate & lost estimates
# ✅ BENCHMARKING - Section percentiles
# ✅ KEY_REASON - Plain language explanations
#
# ============================================================================

import pandas as pd
import numpy as np
from scipy.stats import kendalltau, percentileofscore
from scipy.ndimage import uniform_filter1d
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL
from tqdm import tqdm
import warnings
import gc
from datetime import timedelta
import traceback
warnings.filterwarnings('ignore')

# ============================================================================
# [1] ROBUST STATISTICS FOR LOG-NORMAL DISTRIBUTIONS
# ============================================================================

class RobustStats:
    """
    Robust statistical handler for log-normal and highly volatile data
    - Uses Median/MAD instead of Mean/Std
    - Tolerates outliers (bulk orders, seasonal spikes)
    - Perfect for fabric and fashion industry
    """

    @staticmethod
    def robust_cv(series):
        """
        Robust Coefficient of Variation using Median and MAD

        Ranges:
        0.0 - 0.3: Low volatility
        0.3 - 0.6: Medium volatility
        0.6 - 1.0: High volatility
        > 1.0: Extreme volatility (normal for seasonal fashion)
        """
        if len(series) == 0:
            return 0.0

        series = np.array(series)
        median = np.median(series)

        if median == 0 or np.isnan(median):
            return 1.0 if series.max() > 0 else 0.0

        mad = np.median(np.abs(series - median))
        robust_cv = (mad / median) * 1.4826
        return float(min(robust_cv, 10.0))

    @staticmethod
    def detect_outliers_iqr(series, threshold=1.5):
        """
        Detect outliers using IQR rule
        Suitable for non-normal distributions
        """
        if len(series) < 4:
            return []

        q75, q25 = np.percentile(series, [75, 25])
        iqr = q75 - q25

        if iqr == 0:
            return []

        lower = q25 - (threshold * iqr)
        upper = q75 + (threshold * iqr)

        return [i for i, val in enumerate(series) if val < lower or val > upper]

    @staticmethod
    def winsorize_series(series, limits=0.05):
        """
        Winsorize data: replace outliers with nearest non-outlier values
        Prevents bulk orders from skewing analysis
        """
        if len(series) < 10:
            return series

        series = np.array(series)
        lower = np.percentile(series, limits * 100)
        upper = np.percentile(series, (1 - limits) * 100)

        return np.clip(series, lower, upper)

    @staticmethod
    def seasonal_strength(monthly_series):
        """
        Measure seasonal strength using STL decomposition
        Returns value between 0 (no seasonality) and 1 (strong seasonality)
        """
        if len(monthly_series) < 24:
            return 0.0

        try:
            series = pd.Series(monthly_series).values.astype(float)

            stl = STL(series, period=12, robust=True)
            res = stl.fit()

            seasonal_var = np.var(res.seasonal)
            resid_var = np.var(res.resid)

            if seasonal_var + resid_var == 0:
                return 0.0

            strength = 1 - (resid_var / (seasonal_var + resid_var))
            return float(np.clip(strength, 0, 1))

        except Exception:
            return 0.0


# ============================================================================
# [2] ADVANCED DUAL TREND ANALYZER - SEASONAL AWARE
# ============================================================================

class AdvancedTrendAnalyzer:
    """
    Advanced trend analyzer - seasonality and volatility aware
    - True YoY comparison for seasonal products
    - Data smoothing using moving averages
    - Weighted multi-criteria voting system
    """

    @classmethod
    def analyze_dual_trend(cls, monthly_revenue, monthly_quantity, is_seasonal=False):
        """
        Integrated revenue and quantity trend analysis with seasonality awareness
        """
        if len(monthly_revenue) < 6 or len(monthly_quantity) < 6:
            return {
                'revenue_trend': 'Unknown',
                'revenue_confidence': 'LOW',
                'revenue_period_change': 0,
                'revenue_robust_cv': 0,
                'revenue_yoy_change': 0,
                'quantity_trend': 'Unknown',
                'quantity_confidence': 'LOW',
                'quantity_period_change': 0,
                'quantity_robust_cv': 0,
                'quantity_yoy_change': 0,
                'trend_mismatch': False,
                'mismatch_warning': None
            }

        try:
            rev_direction, rev_confidence, rev_metrics = cls._analyze_single_trend(
                monthly_revenue,
                is_seasonal
            )

            qty_direction, qty_confidence, qty_metrics = cls._analyze_single_trend(
                monthly_quantity,
                is_seasonal
            )

            trend_mismatch = False
            mismatch_warning = None

            if rev_direction != qty_direction and rev_confidence != 'LOW' and qty_confidence != 'LOW':
                trend_mismatch = True

                if rev_direction == 'Growing' and qty_direction == 'Declining':
                    mismatch_warning = "⚠️ Revenue UP but Quantity DOWN → Price increase or premium mix shift"
                elif rev_direction == 'Declining' and qty_direction == 'Growing':
                    mismatch_warning = "⚠️ Revenue DOWN but Quantity UP → Discounting or lower-value products"
                elif rev_direction == 'Stable' and qty_direction != 'Stable':
                    mismatch_warning = f"⚠️ Revenue stable but Quantity {qty_direction}"
                elif qty_direction == 'Stable' and rev_direction != 'Stable':
                    mismatch_warning = f"⚠️ Quantity stable but Revenue {rev_direction}"

            return {
                'revenue_trend': rev_direction,
                'revenue_confidence': rev_confidence,
                'revenue_period_change': float(rev_metrics.get('period_change', 0)),
                'revenue_robust_cv': float(rev_metrics.get('robust_cv', 0)),
                'revenue_yoy_change': float(rev_metrics.get('yoy_change', 0)) if rev_metrics.get('yoy_change') else 0,
                'quantity_trend': qty_direction,
                'quantity_confidence': qty_confidence,
                'quantity_period_change': float(qty_metrics.get('period_change', 0)),
                'quantity_robust_cv': float(qty_metrics.get('robust_cv', 0)),
                'quantity_yoy_change': float(qty_metrics.get('yoy_change', 0)) if qty_metrics.get('yoy_change') else 0,
                'trend_mismatch': trend_mismatch,
                'mismatch_warning': mismatch_warning
            }

        except Exception as e:
            return {
                'revenue_trend': 'Error',
                'revenue_confidence': 'LOW',
                'revenue_period_change': 0,
                'revenue_robust_cv': 0,
                'revenue_yoy_change': 0,
                'quantity_trend': 'Error',
                'quantity_confidence': 'LOW',
                'quantity_period_change': 0,
                'quantity_robust_cv': 0,
                'quantity_yoy_change': 0,
                'trend_mismatch': False,
                'mismatch_warning': None
            }

    @staticmethod
    def _analyze_single_trend(monthly_values, is_seasonal=False):
        """Advanced trend analysis for single metric"""

        values = np.array(monthly_values, dtype=np.float32)

        # ========== 1. Smooth data ==========
        if len(values) >= 6:
            smoothed = uniform_filter1d(values, size=3, mode='nearest')
            recent_smoothed = smoothed[-min(12, len(smoothed)):]
        else:
            smoothed = values
            recent_smoothed = values

        # ========== 2. Calculate changes ==========
        period_change = 0
        yoy_change = None

        if is_seasonal and len(values) >= 24:
            last_3 = values[-3:].mean()
            prev_year_same = values[-15:-12].mean()
            if prev_year_same > 0:
                yoy_change = ((last_3 - prev_year_same) / prev_year_same * 100)
                period_change = yoy_change
        else:
            if len(values) >= 6:
                last_3 = values[-3:].mean()
                prev_3 = values[-6:-3].mean()
                if prev_3 > 0:
                    period_change = ((last_3 - prev_3) / prev_3 * 100)

        # ========== 3. Mann-Kendall on smoothed data ==========
        mk_tau, mk_p = 0, 1
        if len(recent_smoothed) >= 4:
            try:
                months = np.arange(len(recent_smoothed))
                mk_tau, mk_p = kendalltau(months, recent_smoothed)
            except:
                pass

        # ========== 4. Exponential Smoothing ==========
        exp_trend = 0
        try:
            model = ExponentialSmoothing(values, trend='add', seasonal=None)
            fit = model.fit(smoothing_level=0.2, smoothing_trend=0.1, optimized=False)
            if fit.trend is not None and len(fit.trend) >= 3:
                exp_trend = fit.trend[-3:].mean()
            elif fit.trend is not None:
                exp_trend = fit.trend[-1]
        except:
            pass

        # ========== 5. Robust CV ==========
        robust_cv = RobustStats.robust_cv(recent_smoothed)

        # ========== 6. Consistency ==========
        changes = np.diff(recent_smoothed)
        consistency = 0
        if len(changes) > 0:
            positive = (changes > 0).sum()
            negative = (changes < 0).sum()
            consistency = max(positive, negative) / len(changes) * 100

        # ========== 7. Weighted voting ==========
        votes_declining = 0
        votes_growing = 0
        total_weight = 0

        if exp_trend < -200:
            votes_declining += 1.5
            total_weight += 1.5
        elif exp_trend > 200:
            votes_growing += 1.5
            total_weight += 1.5

        if mk_tau < -0.1 and mk_p < 0.15:
            votes_declining += 2
            total_weight += 2
        elif mk_tau > 0.1 and mk_p < 0.15:
            votes_growing += 2
            total_weight += 2

        if period_change < -8:
            votes_declining += 1.5
            total_weight += 1.5
        elif period_change > 8:
            votes_growing += 1.5
            total_weight += 1.5

        if yoy_change is not None:
            if yoy_change < -10:
                votes_declining += 2
                total_weight += 2
            elif yoy_change > 10:
                votes_growing += 2
                total_weight += 2

        if robust_cv < 0.4:
            if votes_growing == 0 and votes_declining == 0:
                votes_growing += 0.5
                votes_declining += 0.5
                total_weight += 1

        # ========== 8. Determine direction ==========
        if total_weight == 0:
            direction = "Stable"
            confidence_score = 0.5
        else:
            growing_score = votes_growing / total_weight
            declining_score = votes_declining / total_weight

            if growing_score > 0.6:
                direction = "Growing"
                confidence_score = growing_score
            elif declining_score > 0.6:
                direction = "Declining"
                confidence_score = declining_score
            else:
                direction = "Stable"
                confidence_score = 1 - abs(growing_score - declining_score)

        # ========== 9. Adjust confidence ==========
        if consistency > 70:
            confidence_score = min(confidence_score * 1.2, 1.0)
        elif consistency < 40:
            confidence_score *= 0.8

        if robust_cv > 0.8:
            confidence_score *= 0.9

        if confidence_score >= 0.7:
            confidence = "HIGH"
        elif confidence_score >= 0.5:
            confidence = "MEDIUM"
        else:
            confidence = "LOW"

        metrics = {
            'period_change': float(period_change),
            'yoy_change': float(yoy_change) if yoy_change is not None else None,
            'robust_cv': float(robust_cv),
            'consistency': float(consistency),
            'mk_tau': float(mk_tau),
            'exp_trend': float(exp_trend)
        }

        return direction, confidence, metrics


# ============================================================================
# [3] FABRIC-SPECIFIC STOCK-OUT ANALYZER
# ============================================================================

class FabricStockOutAnalyzer:
    """
    Stock-out analyzer specifically designed for fabric industry
    - Differentiates between Slow Mover and True Stockout
    - Calculates daily sales rate accurately
    - Estimates losses with seasonality awareness
    """

    @classmethod
    def analyze(cls, sku_data, current_stock, analysis_date, has_profit=False):
        """
        Advanced inventory status analysis
        """

        result = {
            'is_stockout': False,
            'is_potential_stockout': False,
            'is_slow_mover': False,
            'is_true_stockout': False,
            'stockout_date': None,
            'days_out_of_stock': 0,
            'pre_stockout_sales_rate_qty': 0,
            'pre_stockout_sales_rate_value': 0,
            'estimated_lost_sales_qty': 0,
            'estimated_lost_revenue': 0,
            'estimated_lost_profit': 0,
            'pre_stockout_avg_margin': 0,
            'activity_ratio': 0,
            'avg_daily_sales': 0,
            'inventory_turnover_days': 0
        }

        try:
            dates = pd.to_datetime(sku_data['Date'].values)
            revenue = sku_data['bal Value'].values
            qty = sku_data['bal Qty'].values

            last_sale = dates.max()
            first_sale = dates.min()
            days_since_last = (analysis_date - last_sale).days

            # ========== Slow Mover Detection ==========
            total_days_span = (analysis_date - first_sale).days
            if total_days_span < 1:
                total_days_span = 1

            active_days = len(np.unique(dates))
            activity_ratio = active_days / total_days_span

            total_qty_sold = qty.sum()
            avg_daily_sales = total_qty_sold / total_days_span

            is_slow_mover = activity_ratio < 0.1 and avg_daily_sales < 0.5

            result['is_slow_mover'] = is_slow_mover
            result['activity_ratio'] = float(activity_ratio)
            result['avg_daily_sales'] = float(avg_daily_sales)

            if is_slow_mover:
                result['is_stockout'] = False
                result['is_potential_stockout'] = False
                return result

            # ========== True Stock-out Detection ==========
            if current_stock == 0 and days_since_last > 30:
                result['is_stockout'] = True
                result['stockout_date'] = last_sale
                result['days_out_of_stock'] = days_since_last

                if avg_daily_sales > 1 and active_days > 30:
                    result['is_true_stockout'] = True

                # ========== Pre-stockout analysis ==========
                cutoff_date = last_sale - timedelta(days=90)
                mask = dates < last_sale

                if mask.any():
                    pre_dates = dates[mask]
                    pre_revenue = revenue[mask]
                    pre_qty = qty[mask]

                    pre_active_days = len(np.unique(pre_dates))
                    pre_period_days = (last_sale - pre_dates.min()).days

                    if pre_period_days > 0:
                        sales_rate_qty = pre_qty.sum() / pre_period_days
                        sales_rate_value = pre_revenue.sum() / pre_period_days

                        result['pre_stockout_sales_rate_qty'] = float(sales_rate_qty)
                        result['pre_stockout_sales_rate_value'] = float(sales_rate_value)

                        estimated_lost_qty = sales_rate_qty * days_since_last
                        estimated_lost_revenue = sales_rate_value * days_since_last

                        result['estimated_lost_sales_qty'] = float(estimated_lost_qty)
                        result['estimated_lost_revenue'] = float(estimated_lost_revenue)

                        if has_profit and 'Total_Profit' in sku_data.columns:
                            pre_profit = sku_data['Total_Profit'].values[mask].sum()
                            pre_revenue_total = pre_revenue.sum()
                            if pre_revenue_total > 0:
                                avg_margin = (pre_profit / pre_revenue_total * 100)
                                result['pre_stockout_avg_margin'] = float(avg_margin)
                                result['estimated_lost_profit'] = float(estimated_lost_revenue * (avg_margin / 100))

                        if (sales_rate_qty > 2 or sales_rate_value > 2000) and pre_active_days >= 30:
                            result['is_potential_stockout'] = True

            # ========== Inventory Turnover ==========
            if avg_daily_sales > 0 and current_stock > 0:
                inventory_turnover_days = current_stock / avg_daily_sales
                result['inventory_turnover_days'] = float(inventory_turnover_days)

            return result

        except Exception as e:
            return result


# ============================================================================
# [4] ENHANCED SECTION ANALYZER WITH ABC ANALYSIS
# ============================================================================

class EnhancedSectionAnalyzer:
    """
    Section-level analysis with:
    - ABC analysis for risk detection
    - SKU health metrics
    - Inventory turnover
    - YoY comparison
    """

    def __init__(self, df, analysis_date):
        self.df = df
        self.analysis_date = analysis_date
        self.has_profit = 'Total_Profit' in df.columns
        self.has_stock = 'CURRENT_STOCK' in df.columns

    def analyze_section(self, section_name, section_data, sku_results_df):
        """
        Comprehensive section analysis
        """

        try:
            # ========== Basic Metrics ==========
            total_revenue = section_data['bal Value'].sum()
            total_qty = section_data['bal Qty'].sum()
            total_profit = section_data['Total_Profit'].sum() if self.has_profit else 0
            num_skus = section_data['SKU_CLEAN'].nunique()
            num_transactions = len(section_data)

            first_sale = section_data['Date'].min()
            last_sale = section_data['Date'].max()
            days_since_last = (self.analysis_date - last_sale).days

            # ========== Recent Period (Last 12 Months) ==========
            one_year_ago = self.analysis_date - timedelta(days=365)
            recent_data = section_data[section_data['Date'] >= one_year_ago]

            recent_revenue = recent_data['bal Value'].sum() if len(recent_data) > 0 else 0
            recent_qty = recent_data['bal Qty'].sum() if len(recent_data) > 0 else 0
            recent_profit = recent_data['Total_Profit'].sum() if self.has_profit and len(recent_data) > 0 else 0

            # ========== Margins ==========
            avg_margin = (total_profit / total_revenue * 100) if total_revenue > 0 else 0
            recent_avg_margin = (recent_profit / recent_revenue * 100) if recent_revenue > 0 else 0

            # ========== Stock Metrics ==========
            if self.has_stock:
                latest_stock = section_data.groupby('SKU_CLEAN')['CURRENT_STOCK'].first()
                total_stock = latest_stock.sum()
                skus_out_of_stock = (latest_stock == 0).sum()
                skus_low_stock = ((latest_stock > 0) & (latest_stock < 10)).sum()
                stockout_rate = (skus_out_of_stock / num_skus * 100) if num_skus > 0 else 0
            else:
                total_stock = 0
                skus_out_of_stock = 0
                skus_low_stock = 0
                stockout_rate = 0

            # ========== Monthly Analysis ==========
            section_data_copy = section_data.copy()
            section_data_copy['YearMonth'] = pd.to_datetime(section_data_copy['Date']).dt.to_period('M')

            monthly_rev = section_data_copy.groupby('YearMonth')['bal Value'].sum()
            monthly_qty = section_data_copy.groupby('YearMonth')['bal Qty'].sum()

            active_months = len(monthly_rev)

            # ========== Robust Seasonality Detection ==========
            seasonal_strength = RobustStats.seasonal_strength(monthly_rev.values) if len(monthly_rev) >= 24 else 0
            is_seasonal = seasonal_strength > 0.3
            robust_cv_revenue = RobustStats.robust_cv(monthly_rev.values) if len(monthly_rev) > 0 else 0

            # ========== Dual Trend Analysis ==========
            dual_trend = AdvancedTrendAnalyzer.analyze_dual_trend(
                monthly_rev.values if len(monthly_rev) > 0 else [],
                monthly_qty.values if len(monthly_qty) > 0 else [],
                is_seasonal=is_seasonal
            )

            # ========== SKU Health Metrics ==========
            section_skus = sku_results_df[sku_results_df['Section'] == section_name] if len(sku_results_df) > 0 else pd.DataFrame()

            if len(section_skus) > 0:
                growing_skus = len(section_skus[section_skus['Revenue_Trend'] == 'Growing'])
                declining_skus = len(section_skus[section_skus['Revenue_Trend'] == 'Declining'])
                stable_skus = len(section_skus[section_skus['Revenue_Trend'] == 'Stable'])

                pct_growing = (growing_skus / len(section_skus) * 100) if len(section_skus) > 0 else 0
                pct_declining = (declining_skus / len(section_skus) * 100) if len(section_skus) > 0 else 0

                high_conf_growing = len(section_skus[
                    (section_skus['Revenue_Trend'] == 'Growing') &
                    (section_skus['Revenue_Confidence'] == 'HIGH')
                ])
                high_conf_declining = len(section_skus[
                    (section_skus['Revenue_Trend'] == 'Declining') &
                    (section_skus['Revenue_Confidence'] == 'HIGH')
                ])

                slow_movers = len(section_skus[section_skus['Is_Slow_Mover'] == True]) if 'Is_Slow_Mover' in section_skus.columns else 0
                potential_stockouts = len(section_skus[section_skus['Is_Potential_Stockout'] == True]) if 'Is_Potential_Stockout' in section_skus.columns else 0
                true_stockouts = len(section_skus[section_skus['Is_True_Stockout'] == True]) if 'Is_True_Stockout' in section_skus.columns else 0
            else:
                growing_skus = declining_skus = stable_skus = 0
                pct_growing = pct_declining = 0
                high_conf_growing = high_conf_declining = 0
                slow_movers = potential_stockouts = true_stockouts = 0

            # ========== ABC Analysis (Top-heavy risk) ==========
            sku_revenue = section_data.groupby('SKU_CLEAN')['bal Value'].sum().sort_values(ascending=False)

            if len(sku_revenue) > 0:
                top_20_pct_count = max(1, int(len(sku_revenue) * 0.2))
                top_20_revenue = sku_revenue.head(top_20_pct_count).sum()
                top_20_contribution = (top_20_revenue / total_revenue * 100) if total_revenue > 0 else 0
                is_top_heavy = top_20_contribution > 80
            else:
                top_20_contribution = 0
                is_top_heavy = False

            # ========== Inventory Turnover ==========
            if self.has_stock and total_stock > 0 and recent_revenue > 0:
                inventory_turnover_ratio = recent_revenue / total_stock
            else:
                inventory_turnover_ratio = 0

            # ========== Status Determination ==========
            status, key_reason = self._determine_status_with_reason(
                revenue_trend=dual_trend['revenue_trend'],
                quantity_trend=dual_trend['quantity_trend'],
                trend_mismatch=dual_trend['trend_mismatch'],
                recent_avg_margin=recent_avg_margin,
                stockout_rate=stockout_rate,
                days_since_last=days_since_last,
                pct_declining=pct_declining,
                pct_growing=pct_growing,
                is_seasonal=is_seasonal,
                robust_cv=robust_cv_revenue,
                is_top_heavy=is_top_heavy,
                top_20_contribution=top_20_contribution,
                inventory_turnover_ratio=inventory_turnover_ratio,
                true_stockouts=true_stockouts,
                slow_movers=slow_movers
            )

            return {
                'Section': section_name,
                'Status': status,
                'Key_Reason': key_reason,

                # SKU counts
                'Total_SKUs': num_skus,
                'SKUs_Out_of_Stock': skus_out_of_stock,
                'SKUs_Low_Stock': skus_low_stock,
                'Stock_Out_Rate_%': float(stockout_rate),
                'True_Stockouts': true_stockouts,
                'Slow_Movers': slow_movers,

                # SKU health
                'SKUs_Growing': growing_skus,
                'SKUs_Declining': declining_skus,
                'SKUs_Stable': stable_skus,
                'Pct_Growing': float(pct_growing),
                'Pct_Declining': float(pct_declining),
                'High_Conf_Growing': high_conf_growing,
                'High_Conf_Declining': high_conf_declining,
                'Potential_Stockouts': potential_stockouts,

                # Revenue
                'Total_Revenue': float(total_revenue),
                'Recent_Revenue_12M': float(recent_revenue),
                'Revenue_Per_SKU': float(total_revenue / num_skus) if num_skus > 0 else 0,

                # Quantity
                'Total_Quantity': float(total_qty),
                'Recent_Quantity_12M': float(recent_qty),

                # Transactions
                'Num_Transactions': num_transactions,

                # Monthly metrics
                'Active_Months': active_months,
                'Revenue_Robust_CV': float(robust_cv_revenue),

                # Profit
                'Total_Profit': float(total_profit),
                'Recent_Profit_12M': float(recent_profit),
                'Avg_Margin_%': float(avg_margin),
                'Recent_Avg_Margin_%': float(recent_avg_margin),

                # Stock
                'Total_Stock': float(total_stock),
                'Inventory_Turnover_Ratio': float(inventory_turnover_ratio),

                # Time
                'First_Sale': first_sale,
                'Last_Sale': last_sale,
                'Days_Since_Last_Sale': days_since_last,

                # Dual trend
                'Revenue_Trend': dual_trend['revenue_trend'],
                'Revenue_Confidence': dual_trend['revenue_confidence'],
                'Revenue_Period_Change_%': float(dual_trend['revenue_period_change']),
                'Revenue_YoY_Change_%': float(dual_trend['revenue_yoy_change']),
                'Quantity_Trend': dual_trend['quantity_trend'],
                'Quantity_Confidence': dual_trend['quantity_confidence'],
                'Quantity_Period_Change_%': float(dual_trend['quantity_period_change']),
                'Quantity_YoY_Change_%': float(dual_trend['quantity_yoy_change']),
                'Trend_Mismatch': dual_trend['trend_mismatch'],
                'Mismatch_Warning': dual_trend['mismatch_warning'],

                # Seasonality
                'Is_Seasonal': is_seasonal,
                'Seasonal_Strength': float(seasonal_strength),

                # ABC Analysis
                'Top20_Contribution_%': float(top_20_contribution),
                'Is_Top_Heavy': is_top_heavy
            }

        except Exception as e:
            print(f"⚠️ Error analyzing section {section_name}: {e}")
            traceback.print_exc()
            return None

    def _determine_status_with_reason(self, revenue_trend, quantity_trend, trend_mismatch,
                                    recent_avg_margin, stockout_rate, days_since_last,
                                    pct_declining, pct_growing, is_seasonal, robust_cv,
                                    is_top_heavy, top_20_contribution, inventory_turnover_ratio,
                                    true_stockouts, slow_movers):
        """Determine section status with clear reason"""

        reasons = []

        # ========== Critical Issues ==========
        if recent_avg_margin < 0:
            status = "🔴 Loss-Making"
            reasons.append(f"Negative margin {recent_avg_margin:.1f}%")

        elif stockout_rate > 30:
            status = "🔴 High Stock-Out Rate"
            reasons.append(f"{stockout_rate:.0f}% SKUs out of stock")

        elif true_stockouts > 5:
            status = "🔴 Multiple True Stockouts"
            reasons.append(f"{true_stockouts} SKUs with sudden stops")

        elif days_since_last > 180:
            status = "⚪ Inactive"
            reasons.append(f"{days_since_last} days since last sale")

        # ========== Major Issues ==========
        elif revenue_trend == "Declining" and recent_avg_margin < 15:
            status = "🟠 Declining (Low Margin)"
            reasons.append(f"Revenue declining with {recent_avg_margin:.1f}% margin")
            if pct_declining > 50:
                reasons.append(f"{pct_declining:.0f}% SKUs declining")

        elif revenue_trend == "Declining":
            status = "🟠 Declining"
            reasons.append("Revenue trending down")
            if quantity_trend == "Declining":
                reasons.append("Quantity also declining")

        elif is_top_heavy:
            status = "🟠 High Concentration Risk"
            reasons.append(f"Top 20% SKUs = {top_20_contribution:.0f}% of revenue")

        elif slow_movers > num_skus * 0.3:  # >30% slow movers
            status = "🟠 High Slow Mover Ratio"
            reasons.append(f"{slow_movers} slow moving SKUs")

        # ========== Opportunities ==========
        elif revenue_trend == "Growing" and recent_avg_margin > 30:
            status = "🟢 Growing (High Margin)"
            reasons.append(f"Revenue up with {recent_avg_margin:.1f}% margin")
            if pct_growing > 50:
                reasons.append(f"{pct_growing:.0f}% SKUs growing")

        elif revenue_trend == "Growing":
            status = "🟢 Growing"
            reasons.append("Revenue trending up")

        elif revenue_trend == "Volatile":
            if is_seasonal:
                status = "🎯 Seasonal"
                reasons.append("Clear seasonal pattern")
            else:
                status = "🌊 Volatile"
                reasons.append(f"High volatility (CV: {robust_cv:.1f})")

        elif revenue_trend == "Stable" and recent_avg_margin > 25:
            status = "✅ Stable (Profitable)"
            reasons.append(f"Steady with {recent_avg_margin:.1f}% margin")

        elif revenue_trend == "Stable":
            status = "✅ Stable"
            reasons.append("Consistent performance")

        else:
            status = "Active"
            reasons.append("Currently active")

        # ========== Additional Indicators ==========
        if trend_mismatch:
            if revenue_trend == "Growing" and quantity_trend == "Declining":
                reasons.append("⚠️ Price increase or premium mix shift")
            elif revenue_trend == "Declining" and quantity_trend == "Growing":
                reasons.append("⚠️ Discounting or lower-value products")

        if inventory_turnover_ratio < 2 and recent_revenue > 10000:
            reasons.append(f"🐢 Slow turnover ({inventory_turnover_ratio:.1f}x)")
        elif inventory_turnover_ratio > 12:
            reasons.append(f"⚡ Fast turnover ({inventory_turnover_ratio:.1f}x)")

        key_reason = " | ".join(reasons) if reasons else "No specific issues"

        return status, key_reason


# ============================================================================
# [5] ULTIMATE SKU ANALYZER - FABRIC OPTIMIZED
# ============================================================================

class UltimateSKUAnalyzer:
    """
    Ultimate SKU analyzer - specifically designed for fabric industry
    - All robust statistical improvements
    - True seasonality detection
    - Smart inventory analysis
    - Comprehensive performance reports
    """

    def __init__(self, df, analysis_date=None):
        print("=" * 120)
        print("🧵 ULTIMATE FABRIC & APPAREL ANALYZER - INITIALIZING")
        print("=" * 120)

        # Data preparation
        print("🔧 Pre-processing data...")

        df = df.copy()
        df['SKU_CLEAN'] = df['SKU'].astype(str).str.strip()

        if not pd.api.types.is_datetime64_any_dtype(df['Date']):
            df['Date'] = pd.to_datetime(df['Date'])

        self.df = df
        self.analysis_date = analysis_date or df['Date'].max()
        self.analysis_date = pd.Timestamp(self.analysis_date)

        # Check available columns
        self.has_profit = 'Total_Profit' in df.columns
        self.has_stock = 'CURRENT_STOCK' in df.columns
        self.has_sales_type = 'Sales_Type' in df.columns
        self.has_section = 'Section' in df.columns
        self.has_cost = 'Cost' in df.columns or 'Unit_Cost' in df.columns

        # Print summary
        print(f"📅 Analysis Date: {self.analysis_date.strftime('%Y-%m-%d')}")
        print(f"📦 Total SKUs: {df['SKU_CLEAN'].nunique():,}")
        print(f"📊 Total Transactions: {len(df):,}")
        print(f"💰 Profit Data: {'✅' if self.has_profit else '❌'}")
        print(f"📦 Stock Data: {'✅' if self.has_stock else '❌'}")
        print(f"📂 Section Data: {'✅' if self.has_section else '❌'}")
        print("=" * 120)

    def analyze_single_sku(self, sku_data, sku_name):
        """Single SKU analysis with all enhancements"""

        if len(sku_data) == 0:
            return None

        try:
            # Basic info
            section = sku_data['Section'].iloc[0] if self.has_section else 'Unknown'

            # Totals
            total_revenue = sku_data['bal Value'].sum()
            total_qty = sku_data['bal Qty'].sum()

            # Dates
            first_sale = pd.Timestamp(sku_data['Date'].min())
            last_sale = pd.Timestamp(sku_data['Date'].max())
            days_since_last = (self.analysis_date - last_sale).days
            product_age_days = (self.analysis_date - first_sale).days

            # Recent period (last 12 months)
            one_year_ago = self.analysis_date - timedelta(days=365)
            recent_mask = pd.to_datetime(sku_data['Date']) >= one_year_ago
            recent_data = sku_data[recent_mask]

            recent_revenue = recent_data['bal Value'].sum() if len(recent_data) > 0 else 0
            recent_qty = recent_data['bal Qty'].sum() if len(recent_data) > 0 else 0
            recent_transactions = len(recent_data)

            # Profit
            total_profit = 0
            avg_margin = 0
            recent_profit = 0
            recent_avg_margin = 0

            if self.has_profit:
                total_profit = sku_data['Total_Profit'].sum()
                avg_margin = (total_profit / total_revenue * 100) if total_revenue > 0 else 0

                if len(recent_data) > 0:
                    recent_profit = recent_data['Total_Profit'].sum()
                    recent_avg_margin = (recent_profit / recent_revenue * 100) if recent_revenue > 0 else 0

            # Current stock
            current_stock = 0
            if self.has_stock:
                stock_values = sku_data['CURRENT_STOCK'].dropna()
                if len(stock_values) > 0:
                    current_stock = stock_values.iloc[0]

            # Monthly analysis
            sku_data_copy = sku_data.copy()
            sku_data_copy['YearMonth'] = pd.to_datetime(sku_data_copy['Date']).dt.to_period('M')

            monthly = sku_data_copy.groupby('YearMonth', observed=True).agg({
                'bal Value': 'sum',
                'bal Qty': 'sum'
            })

            monthly_revenue = monthly['bal Value'].values if len(monthly) > 0 else []
            monthly_quantity = monthly['bal Qty'].values if len(monthly) > 0 else []

            # Robust metrics
            robust_cv_revenue = RobustStats.robust_cv(monthly_revenue) if len(monthly_revenue) > 0 else 0
            robust_cv_qty = RobustStats.robust_cv(monthly_quantity) if len(monthly_quantity) > 0 else 0

            # Seasonality detection
            seasonal_strength = RobustStats.seasonal_strength(monthly_revenue) if len(monthly_revenue) >= 24 else 0
            is_seasonal = seasonal_strength > 0.3

            # Advanced trend analysis
            trend_analysis = AdvancedTrendAnalyzer.analyze_dual_trend(
                monthly_revenue,
                monthly_quantity,
                is_seasonal=is_seasonal
            )

            # Advanced stock-out analysis
            stockout_analysis = FabricStockOutAnalyzer.analyze(
                sku_data,
                current_stock,
                self.analysis_date,
                self.has_profit
            )

            # ========== Status determination with KEY_REASON ==========
            status, key_reason = self._determine_sku_status(
                revenue_trend=trend_analysis['revenue_trend'],
                recent_avg_margin=recent_avg_margin,
                is_stockout=stockout_analysis['is_stockout'],
                is_true_stockout=stockout_analysis['is_true_stockout'],
                is_potential_stockout=stockout_analysis['is_potential_stockout'],
                is_slow_mover=stockout_analysis['is_slow_mover'],
                current_stock=current_stock,
                days_since_last=days_since_last,
                recent_transactions=recent_transactions,
                robust_cv=robust_cv_revenue,
                is_seasonal=is_seasonal,
                seasonal_strength=seasonal_strength,
                trend_mismatch=trend_analysis['trend_mismatch']
            )

            # Priority
            priority = self._get_priority(status)

            # ========== Compile results ==========
            return {
                'SKU': sku_name,
                'Section': section,
                'Status': status,
                'Key_Reason': key_reason,
                'Priority': priority,

                # Revenue
                'Total_Revenue': float(total_revenue),
                'Recent_Revenue_12M': float(recent_revenue),
                'Revenue_Robust_CV': float(robust_cv_revenue),

                # Quantity
                'Total_Quantity': float(total_qty),
                'Recent_Quantity_12M': float(recent_qty),
                'Quantity_Robust_CV': float(robust_cv_qty),

                # Transactions
                'Total_Transactions': len(sku_data),
                'Recent_Transactions_12M': recent_transactions,

                # Monthly averages
                'Avg_Monthly_Revenue': float(monthly_revenue.mean()) if len(monthly_revenue) > 0 else 0,
                'Active_Months': len(monthly_revenue),

                # Profit
                'Total_Profit': float(total_profit),
                'Recent_Profit_12M': float(recent_profit),
                'Avg_Margin_%': float(avg_margin),
                'Recent_Avg_Margin_%': float(recent_avg_margin),

                # Stock
                'Current_Stock': float(current_stock),
                'Is_Slow_Mover': stockout_analysis['is_slow_mover'],
                'Inventory_Turnover_Days': float(stockout_analysis['inventory_turnover_days']),
                'Activity_Ratio': float(stockout_analysis['activity_ratio']),
                'Avg_Daily_Sales': float(stockout_analysis['avg_daily_sales']),

                # Dates
                'First_Sale': first_sale,
                'Last_Sale': last_sale,
                'Days_Since_Last_Sale': days_since_last,
                'Product_Age_Days': product_age_days,

                # Trends
                'Revenue_Trend': trend_analysis['revenue_trend'],
                'Revenue_Confidence': trend_analysis['revenue_confidence'],
                'Revenue_Period_Change_%': float(trend_analysis['revenue_period_change']),
                'Revenue_YoY_Change_%': float(trend_analysis['revenue_yoy_change']),

                'Quantity_Trend': trend_analysis['quantity_trend'],
                'Quantity_Confidence': trend_analysis['quantity_confidence'],
                'Quantity_Period_Change_%': float(trend_analysis['quantity_period_change']),
                'Quantity_YoY_Change_%': float(trend_analysis['quantity_yoy_change']),

                'Trend_Mismatch': trend_analysis['trend_mismatch'],
                'Mismatch_Warning': trend_analysis['mismatch_warning'],

                # Seasonality
                'Is_Seasonal': is_seasonal,
                'Seasonal_Strength': float(seasonal_strength),

                # Stock-out analysis
                'Is_Stockout': stockout_analysis['is_stockout'],
                'Is_True_Stockout': stockout_analysis['is_true_stockout'],
                'Is_Potential_Stockout': stockout_analysis['is_potential_stockout'],
                'Days_Out_of_Stock': stockout_analysis['days_out_of_stock'],
                'Estimated_Lost_Sales_Qty': float(stockout_analysis['estimated_lost_sales_qty']),
                'Estimated_Lost_Revenue': float(stockout_analysis['estimated_lost_revenue']),
                'Estimated_Lost_Profit': float(stockout_analysis['estimated_lost_profit'])
            }

        except Exception as e:
            print(f"⚠️ Error analyzing SKU {sku_name}: {e}")
            traceback.print_exc()
            return None

    def _determine_sku_status(self, revenue_trend, recent_avg_margin,
                            is_stockout, is_true_stockout, is_potential_stockout,
                            is_slow_mover, current_stock, days_since_last,
                            recent_transactions, robust_cv, is_seasonal,
                            seasonal_strength, trend_mismatch):
        """Determine SKU status with clear reason"""

        reasons = []

        # ========== Critical (Priority 0-4) ==========
        if is_true_stockout:
            status = "🔴 True Stockout"
            reasons.append("Sudden stop after good sales")

        elif is_potential_stockout:
            status = "🟠 Potential Stockout (High Value)"
            reasons.append("High sales rate + zero stock")

        elif is_stockout:
            status = "🟡 Stockout"
            reasons.append(f"{days_since_last} days no stock")

        elif recent_avg_margin < 0:
            status = "🔴 Loss-Making"
            reasons.append(f"Recent margin {recent_avg_margin:.1f}%")

        elif 0 < current_stock < 10 and recent_avg_margin > 30:
            status = "🟠 Low Stock (High Margin)"
            reasons.append(f"Only {current_stock:.0f} units left + {recent_avg_margin:.1f}% margin")

        elif 0 < current_stock < 10:
            status = "🟡 Low Stock"
            reasons.append(f"Only {current_stock:.0f} units left")

        elif is_slow_mover:
            status = "🟤 Slow Mover"
            reasons.append("Activity ratio < 10%")

        elif days_since_last > 365:
            status = "⚫ Dead"
            reasons.append("No sales >1 year")

        elif days_since_last > 180:
            status = "⚪ Inactive"
            reasons.append(f"{days_since_last} days since sale")

        elif recent_transactions < 3 and days_since_last < 90:
            status = "🆕 New"
            reasons.append("New SKU - under evaluation")

        # ========== Trend-based ==========
        elif revenue_trend == "Declining":
            if recent_avg_margin < 15:
                status = "📉 Declining (Low Margin)"
                reasons.append(f"Revenue down + {recent_avg_margin:.1f}% margin")
            else:
                status = "📉 Declining"
                reasons.append("Revenue trending down")

        elif revenue_trend == "Growing":
            if recent_avg_margin > 30:
                status = "📈 Growing (High Margin)"
                reasons.append(f"Revenue up + {recent_avg_margin:.1f}% margin")
            else:
                status = "📈 Growing"
                reasons.append("Revenue trending up")

        elif revenue_trend == "Volatile":
            if is_seasonal:
                status = "🎯 Seasonal"
                reasons.append(f"Seasonal strength: {seasonal_strength:.1%}")
            else:
                status = "🌊 Volatile"
                reasons.append(f"High volatility (CV: {robust_cv:.1f})")

        else:  # Stable
            if recent_avg_margin > 25:
                status = "✅ Stable (Profitable)"
                reasons.append(f"Steady + {recent_avg_margin:.1f}% margin")
            else:
                status = "✅ Stable"
                reasons.append("Steady performance")

        # ========== Additional warnings ==========
        if trend_mismatch:
            reasons.append("⚠️ Revenue/Qty mismatch")

        key_reason = " | ".join(reasons) if reasons else "Active - no issues"

        return status, key_reason

    def _get_priority(self, status):
        """Get priority (0 = highest)"""

        priority_map = {
            "🔴 True Stockout": 0,
            "🟠 Potential Stockout (High Value)": 1,
            "🔴 Loss-Making": 2,
            "🟠 Low Stock (High Margin)": 3,
            "🟡 Stockout": 4,
            "🟡 Low Stock": 5,
            "📉 Declining (Low Margin)": 6,
            "📉 Declining": 7,
            "🟤 Slow Mover": 8,
            "⚫ Dead": 9,
            "⚪ Inactive": 10,
            "📈 Growing (High Margin)": 20,
            "📈 Growing": 21,
            "✅ Stable (Profitable)": 22,
            "✅ Stable": 23,
            "🎯 Seasonal": 24,
            "🆕 New": 25,
            "🌊 Volatile": 26
        }

        return priority_map.get(status, 99)

    def analyze_all(self, save_to_excel=True, output_file='FABRIC_ANALYSIS_COMPLETE.xlsx'):
        """Analyze all SKUs and sections"""

        print("\n" + "=" * 120)
        print("🚀 STARTING COMPREHENSIVE ANALYSIS...")
        print("=" * 120)

        # ========== 1. SKU Analysis ==========
        all_skus = self.df['SKU_CLEAN'].unique()
        total_skus = len(all_skus)

        print(f"\n📦 Analyzing {total_skus:,} SKUs...")
        print("-" * 120)

        sku_results = []

        for i, sku in enumerate(tqdm(all_skus, desc="📊 SKU Analysis", ncols=100, colour='cyan')):
            sku_data = self.df[self.df['SKU_CLEAN'] == sku]

            if len(sku_data) > 0:
                result = self.analyze_single_sku(sku_data, sku)
                if result:
                    sku_results.append(result)

            if (i + 1) % 1000 == 0:
                gc.collect()

        sku_df = pd.DataFrame(sku_results)

        if len(sku_df) == 0:
            print("❌ ERROR: No SKUs were analyzed successfully!")
            return None, None

        # Sort by priority and revenue
        if 'Priority' in sku_df.columns and 'Recent_Revenue_12M' in sku_df.columns:
            sku_df = sku_df.sort_values(['Priority', 'Recent_Revenue_12M'], ascending=[True, False])

        print(f"\n✅ SKU Analysis Complete: {len(sku_df):,} SKUs")

        # ========== 2. Section Analysis ==========
        section_df = None

        if self.has_section:
            print("\n" + "=" * 120)
            print("📂 Analyzing Sections...")
            print("-" * 120)

            section_analyzer = EnhancedSectionAnalyzer(self.df, self.analysis_date)
            section_results = []

            sections = self.df['Section'].dropna().unique()

            for section in tqdm(sections, desc="📂 Section Analysis", ncols=100, colour='green'):
                if pd.isna(section):
                    continue

                section_data = self.df[self.df['Section'] == section]

                if len(section_data) > 0:
                    result = section_analyzer.analyze_section(section, section_data, sku_df)
                    if result:
                        section_results.append(result)

            if len(section_results) > 0:
                section_df = pd.DataFrame(section_results)

                if 'Recent_Revenue_12M' in section_df.columns:
                    section_df = section_df.sort_values('Recent_Revenue_12M', ascending=False)
                    print(f"\n✅ Section Analysis Complete: {len(section_df):,} Sections")
                else:
                    print("\n⚠️ WARNING: Section DataFrame missing expected columns")
                    section_df = None
            else:
                print("\n⚠️ WARNING: No sections were analyzed successfully")
                section_df = None

        # ========== 3. Summary ==========
        self._print_summary(sku_df, section_df)

        # ========== 4. Save to Excel ==========
        if save_to_excel:
            self._save_to_excel(sku_df, section_df, output_file)

        print("\n" + "=" * 120)
        print("🎉 ULTIMATE FABRIC ANALYSIS COMPLETE!")
        print("=" * 120)

        return sku_df, section_df

    def _print_summary(self, sku_df, section_df):
        """Print comprehensive summary"""

        print("\n" + "=" * 120)
        print("📊 ANALYSIS SUMMARY")
        print("=" * 120)

        # ========== SKU Summary ==========
        print("\n📦 SKU STATUS DISTRIBUTION:")
        print("-" * 80)

        if 'Status' in sku_df.columns:
            status_counts = sku_df['Status'].value_counts().head(15)
            for status, count in status_counts.items():
                pct = (count / len(sku_df) * 100)
                print(f"  {status}: {count:,} ({pct:.1f}%)")

        # Critical metrics
        print("\n⚠️  CRITICAL ISSUES:")
        print("-" * 80)

        true_stockouts = len(sku_df[sku_df['Is_True_Stockout'] == True]) if 'Is_True_Stockout' in sku_df.columns else 0
        if true_stockouts > 0:
            lost_revenue = sku_df[sku_df['Is_True_Stockout'] == True]['Estimated_Lost_Revenue'].sum()
            print(f"  🔴 True Stockouts: {true_stockouts:,} SKUs")
            print(f"     Estimated lost revenue: {lost_revenue:,.0f} SAR")

        potential_stockouts = len(sku_df[sku_df['Is_Potential_Stockout'] == True]) if 'Is_Potential_Stockout' in sku_df.columns else 0
        if potential_stockouts > 0:
            print(f"  🟠 Potential High-Value Stockouts: {potential_stockouts:,} SKUs")

        loss_making = len(sku_df[sku_df['Recent_Avg_Margin_%'] < 0]) if 'Recent_Avg_Margin_%' in sku_df.columns else 0
        if loss_making > 0:
            print(f"  🔴 Loss-Making SKUs: {loss_making:,}")

        low_stock_high_margin = len(sku_df[
            (sku_df['Current_Stock'] < 10) &
            (sku_df['Recent_Avg_Margin_%'] > 30)
        ]) if 'Current_Stock' in sku_df.columns and 'Recent_Avg_Margin_%' in sku_df.columns else 0
        if low_stock_high_margin > 0:
            print(f"  🟠 Low Stock + High Margin: {low_stock_high_margin:,} SKUs")

        # Trend mismatches
        mismatches = len(sku_df[sku_df['Trend_Mismatch'] == True]) if 'Trend_Mismatch' in sku_df.columns else 0
        if mismatches > 0:
            print(f"\n  ⚠️  Revenue/Quantity Trend Mismatches: {mismatches:,} SKUs")

        # Seasonality
        seasonal = len(sku_df[sku_df['Is_Seasonal'] == True]) if 'Is_Seasonal' in sku_df.columns else 0
        if seasonal > 0:
            print(f"\n  🎯 Seasonal SKUs: {seasonal:,} ({seasonal/len(sku_df)*100:.1f}%)")

        # ========== Section Summary ==========
        if section_df is not None and len(section_df) > 0:
            print("\n" + "=" * 120)
            print("📂 SECTION STATUS DISTRIBUTION:")
            print("-" * 80)

            if 'Status' in section_df.columns:
                sec_status_counts = section_df['Status'].value_counts().head(10)
                for status, count in sec_status_counts.items():
                    pct = (count / len(section_df) * 100)
                    print(f"  {status}: {count:,} ({pct:.1f}%)")

            # Top 10 sections by revenue
            print("\n🔝 TOP 10 SECTIONS BY REVENUE:")
            print("-" * 120)
            print(f"{'Section':<30} {'Revenue 12M':>15} {'Margin':>10} {'Turnover':>10} {'Status':<25}")
            print("-" * 120)

            for _, row in section_df.head(10).iterrows():
                section = str(row['Section'])[:28]
                revenue = row['Recent_Revenue_12M']
                margin = row['Recent_Avg_Margin_%']
                turnover = row['Inventory_Turnover_Ratio'] if 'Inventory_Turnover_Ratio' in row else 0
                status = str(row['Status'])[:25]

                print(f"{section:<30} {revenue:>15,.0f} {margin:>9.1f}% {turnover:>10.1f}x {status:<25}")

    def _save_to_excel(self, sku_df, section_df, output_file):
        """Save to Excel with multiple sheets"""

        print(f"\n💾 Saving to: {output_file}")

        try:
            with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

                # ========== SKU Sheets ==========
                sku_df.to_excel(writer, sheet_name='All SKUs', index=False)

                # Critical SKUs (Priority 0-5)
                critical = sku_df[sku_df['Priority'] <= 5] if 'Priority' in sku_df.columns else pd.DataFrame()
                if len(critical) > 0:
                    critical.to_excel(writer, sheet_name='🔴 Critical SKUs', index=False)

                # True Stockouts
                true_stockouts = sku_df[sku_df['Is_True_Stockout'] == True] if 'Is_True_Stockout' in sku_df.columns else pd.DataFrame()
                if len(true_stockouts) > 0:
                    true_stockouts.sort_values('Estimated_Lost_Revenue', ascending=False).to_excel(
                        writer, sheet_name='🔴 True Stockouts', index=False)

                # Potential Stockouts
                potential = sku_df[sku_df['Is_Potential_Stockout'] == True] if 'Is_Potential_Stockout' in sku_df.columns else pd.DataFrame()
                if len(potential) > 0:
                    potential.sort_values('Estimated_Lost_Revenue', ascending=False).to_excel(
                        writer, sheet_name='🟠 Potential Stockouts', index=False)

                # Low Stock + High Margin
                low_stock_high_margin = sku_df[
                    (sku_df['Current_Stock'] < 10) &
                    (sku_df['Recent_Avg_Margin_%'] > 30)
                ] if 'Current_Stock' in sku_df.columns and 'Recent_Avg_Margin_%' in sku_df.columns else pd.DataFrame()
                if len(low_stock_high_margin) > 0:
                    low_stock_high_margin.to_excel(writer, sheet_name='🟠 Low Stock-High Margin', index=False)

                # Trend Mismatches
                mismatches = sku_df[sku_df['Trend_Mismatch'] == True] if 'Trend_Mismatch' in sku_df.columns else pd.DataFrame()
                if len(mismatches) > 0:
                    mismatches.to_excel(writer, sheet_name='⚠️ Trend Mismatches', index=False)

                # Growing SKUs
                growing = sku_df[sku_df['Revenue_Trend'] == 'Growing'] if 'Revenue_Trend' in sku_df.columns else pd.DataFrame()
                if len(growing) > 0:
                    growing.to_excel(writer, sheet_name='📈 Growing SKUs', index=False)

                # Declining SKUs
                declining = sku_df[sku_df['Revenue_Trend'] == 'Declining'] if 'Revenue_Trend' in sku_df.columns else pd.DataFrame()
                if len(declining) > 0:
                    declining.to_excel(writer, sheet_name='📉 Declining SKUs', index=False)

                # Seasonal SKUs
                seasonal = sku_df[sku_df['Is_Seasonal'] == True] if 'Is_Seasonal' in sku_df.columns else pd.DataFrame()
                if len(seasonal) > 0:
                    seasonal.to_excel(writer, sheet_name='🎯 Seasonal SKUs', index=False)

                # ========== Section Sheets ==========
                if section_df is not None and len(section_df) > 0:
                    section_df.to_excel(writer, sheet_name='All Sections', index=False)

                    # Problem Sections
                    problem_sections = section_df[
                        section_df['Status'].str.contains('Loss|Stockout|Declining|Risk', na=False)
                    ] if 'Status' in section_df.columns else pd.DataFrame()
                    if len(problem_sections) > 0:
                        problem_sections.to_excel(writer, sheet_name='⚠️ Problem Sections', index=False)

                    # Top Heavy Sections
                    top_heavy = section_df[section_df['Is_Top_Heavy'] == True] if 'Is_Top_Heavy' in section_df.columns else pd.DataFrame()
                    if len(top_heavy) > 0:
                        top_heavy.to_excel(writer, sheet_name='🎯 Top Heavy Sections', index=False)

                    # Top 20 Sections
                    if len(section_df) >= 20:
                        top_20 = section_df.nlargest(20, 'Recent_Revenue_12M')
                        top_20.to_excel(writer, sheet_name='Top 20 Sections', index=False)

                print(f"✅ Excel file saved successfully!")

        except Exception as e:
            print(f"❌ Error saving Excel file: {e}")
            traceback.print_exc()


# ============================================================================
# [6] MAIN FUNCTION - ULTIMATE FABRIC ANALYSIS
# ============================================================================

def ultimate_fabric_analysis(df, analysis_date=None, save_to_excel=True,
                           output_file='FABRIC_ANALYSIS_COMPLETE.xlsx'):
    """
    ULTIMATE FABRIC & APPAREL ANALYSIS - PRODUCTION READY

    Features:
    ---------
    ✅ Robust Statistics for Log-Normal Data (Median/MAD instead of Mean/Std)
    ✅ Advanced Trend Analysis with True YoY Comparison
    ✅ Smart Stock-out Detection (Slow Mover vs True Stockout)
    ✅ Fabric-Specific Inventory Metrics (Turnover Days)
    ✅ ABC Analysis with Top-Heavy Risk Detection
    ✅ Seasonal Strength Measurement (0-1 scale)
    ✅ Clear Status with Emojis and Key_Reason
    ✅ Comprehensive Excel Export with Multiple Sheets

    Parameters:
    -----------
    df : pandas.DataFrame
        Must contain columns: 'SKU', 'Date', 'bal Value', 'bal Qty'
        Optional: 'Section', 'CURRENT_STOCK', 'Total_Profit'

    analysis_date : datetime, optional
        Date to use as "today" for analysis. Defaults to max date in data.

    save_to_excel : bool
        Save results to Excel file

    output_file : str
        Output Excel filename

    Returns:
    --------
    sku_df, section_df : tuple of pandas.DataFrame
        Complete analysis results for SKUs and Sections

    Example:
    --------
    >>> # Basic usage
    >>> sku_df, section_df = ultimate_fabric_analysis(sales_data)
    >>>
    >>> # With custom date and filename
    >>> sku_df, section_df = ultimate_fabric_analysis(
    ...     sales_data,
    ...     analysis_date='2024-12-31',
    ...     output_file='FABRIC_Q4_2024.xlsx'
    ... )
    """

    analyzer = UltimateSKUAnalyzer(df, analysis_date)
    return analyzer.analyze_all(save_to_excel, output_file)


# ============================================================================
# MODULE LOADING MESSAGE
# ============================================================================

print("=" * 120)
print("🧵 ULTIMATE FABRIC & APPAREL ANALYZER - LOADED SUCCESSFULLY!")
print("=" * 120)
print("\n📋 READY FOR PRODUCTION:")
print("   • Log-normal & High Volatility Optimized")
print("   • True YoY Seasonal Comparison")
print("   • Smart Stock-out Detection (Slow Mover vs True Stockout)")
print("   • Inventory Turnover Metrics")
print("   • ABC Analysis & Top-Heavy Risk")
print("   • 20+ Excel Export Sheets")
print("\n🚀 QUICK START:")
print("   >>> sku_df, section_df = ultimate_fabric_analysis(your_dataframe)")
print("\n📊 EXAMPLE:")
print("   >>> # Load your data")
print("   >>> df = pd.read_excel('fabric_sales.xlsx')")
print("   >>> # Run analysis")
print("   >>> sku_df, section_df = ultimate_fabric_analysis(")
print("   ...     df,")
print("   ...     analysis_date='2024-12-31',")
print("   ...     output_file='FABRIC_ANALYSIS_Q4.xlsx'")
print("   ... )")
print("=" * 120)

In [7]:
import pandas as pd
combined_df = pd.read_parquet(r'C:\Users\User\PycharmProjects\JupyterProject\MD\Output\combined_df.parquet')


In [10]:
# = pd.read_excel('your_fabric_data.xlsx')

# 2. Run the ultimate analysis
sku_results, section_results = ultimate_fabric_analysis(
    combined_df,
    analysis_date='2026-02-12',  # Optional: defaults to last date in data
    save_to_excel=True,
    output_file='FABRIC_ANALYSIS_RESULTS.xlsx'
)

🧵 ULTIMATE FABRIC & APPAREL ANALYZER - INITIALIZING
🔧 Pre-processing data...
📅 Analysis Date: 2026-02-12
📦 Total SKUs: 134,003
📊 Total Transactions: 4,510,519
💰 Profit Data: ✅
📦 Stock Data: ✅
📂 Section Data: ✅

🚀 STARTING COMPREHENSIVE ANALYSIS...

📦 Analyzing 134,003 SKUs...
------------------------------------------------------------------------------------------------------------------------


📊 SKU Analysis: 100%|███████████████████████████████████| 134003/134003 [10:53:45<00:00,  3.42it/s]



✅ SKU Analysis Complete: 134,003 SKUs

📂 Analyzing Sections...
------------------------------------------------------------------------------------------------------------------------


📂 Section Analysis:   0%|                                                 | 0/2565 [00:00<?, ?it/s]Traceback (most recent call last):
  File "C:\Users\User\AppData\Local\Temp\ipykernel_13020\942741698.py", line 621, in analyze_section
    status, key_reason = self._determine_status_with_reason(
                         ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        revenue_trend=dual_trend['revenue_trend'],
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<13 lines>...
        slow_movers=slow_movers
        ^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\User\AppData\Local\Temp\ipykernel_13020\942741698.py", line 762, in _determine_status_with_reason
    elif slow_movers > num_skus * 0.3:  # >30% slow movers
                       ^^^^^^^^
NameError: name 'num_skus' is not defined
📂 Section Analysis:   0%|                                         | 1/2565 [00:00<22:11,  1.93it/s]

⚠️ Error analyzing section nan: name 'num_skus' is not defined


...

⚠️ Error analyzing section 962: name 'recent_revenue' is not defined


Traceback (most recent call last):
  File "C:\Users\User\AppData\Local\Temp\ipykernel_13020\942741698.py", line 621, in analyze_section
    status, key_reason = self._determine_status_with_reason(
                         ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        revenue_trend=dual_trend['revenue_trend'],
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<13 lines>...
        slow_movers=slow_movers
        ^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\User\AppData\Local\Temp\ipykernel_13020\942741698.py", line 804, in _determine_status_with_reason
    if inventory_turnover_ratio < 2 and recent_revenue > 10000:
                                        ^^^^^^^^^^^^^^
NameError: name 'recent_revenue' is not defined
📂 Section Analysis: 100%|█████████████████████████████████████▉| 2564/2565 [17:15<00:00,  2.47it/s]

⚠️ Error analyzing section 965: name 'recent_revenue' is not defined


📂 Section Analysis: 100%|██████████████████████████████████████| 2565/2565 [17:15<00:00,  2.48it/s]



✅ Section Analysis Complete: 689 Sections

📊 ANALYSIS SUMMARY

📦 SKU STATUS DISTRIBUTION:
--------------------------------------------------------------------------------
  ✅ Stable: 70,577 (52.7%)
  🟤 Slow Mover: 47,141 (35.2%)
  📉 Declining: 4,370 (3.3%)
  📈 Growing (High Margin): 2,600 (1.9%)
  🔴 Loss-Making: 2,244 (1.7%)
  🟡 Low Stock: 2,175 (1.6%)
  ✅ Stable (Profitable): 1,571 (1.2%)
  🟡 Stockout: 741 (0.6%)
  ⚪ Inactive: 633 (0.5%)
  🟠 Low Stock (High Margin): 599 (0.4%)
  ⚫ Dead: 437 (0.3%)
  🔴 True Stockout: 416 (0.3%)
  🆕 New: 362 (0.3%)
  📈 Growing: 57 (0.0%)
  🟠 Potential Stockout (High Value): 41 (0.0%)

⚠️  CRITICAL ISSUES:
--------------------------------------------------------------------------------
  🔴 True Stockouts: 416 SKUs
     Estimated lost revenue: 15,118,025 SAR
  🟠 Potential High-Value Stockouts: 278 SKUs
  🔴 Loss-Making SKUs: 2,270
  🟠 Low Stock + High Margin: 2,277 SKUs

  ⚠️  Revenue/Quantity Trend Mismatches: 5,560 SKUs

  🎯 Seasonal SKUs: 18,640 (13.9%